# WeltmeisterKI4 SOTA

V4 baut auf dem Clean-Hybrid auf: v2 bleibt die starke XGBoost-Baseline, Deep v1 wird regularisiert, danach lernt ein OOF-Stacking-Meta-Modell dynamisch, wann v2, Deep oder ein Mix besser ist. Zusätzlich wird die Scoreline-Matrix mit Low-Score-/Dixon-Coles-artigen Korrekturen kalibriert.


## 02. Setup und Rohdaten

Legt Projektordner auf D: an, setzt TEMP/PIP-Cache weg von C: und lädt die Basisdaten von martj42. Ergebnis: results, goalscorers und shootouts ab 1980 sind im RAM.


In [33]:

# Was diese Zelle macht:
# Erkennt den Projektordner automatisch, legt data/ und models/ direkt neben dem Notebook/Repo an und lädt die Basisdaten.
# Wichtig fürs Repo: alle trainierten Modelle landen in PROJECT/models statt in einem hardcodierten D:-Pfad.

import os, sys, subprocess, shutil, urllib.request, json, time, re, math, unicodedata
from pathlib import Path
from datetime import date, datetime, timedelta
from collections import defaultdict, deque

import numpy as np
import polars as pl

PROJECT = Path.cwd().resolve()
# Falls Jupyter aus einem anderen Ordner startet, hier manuell setzen:
# PROJECT = Path(r"D:/ml/projects/WeltmeisterKI").resolve()

DATA = PROJECT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
MODELS = PROJECT / "models"
SB_RAW = RAW / "statsbomb"
FIFA_RAW = RAW / "fifa"
TMP_DIR = PROJECT / ".tmp"
PIP_CACHE = TMP_DIR / "pip_cache"

for p in [PROJECT, DATA, RAW, PROCESSED, MODELS, SB_RAW, FIFA_RAW, TMP_DIR, PIP_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["TMP"] = str(TMP_DIR)
os.environ["TEMP"] = str(TMP_DIR)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)

print("PROJECT:", PROJECT)
print("MODELS:", MODELS)
print("RAW:", RAW)
print("PROCESSED:", PROCESSED)
print("TEMP:", TMP_DIR)

URLS = {
    "results": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "goalscorers": "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv",
    "shootouts": "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv",
    "sb_competitions": "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json",
}

def download(url, out):
    out = Path(out)
    if not out.exists() or out.stat().st_size == 0:
        print("download", out.name)
        urllib.request.urlretrieve(url, out)
    return out

download(URLS["results"], RAW / "results.csv")
download(URLS["goalscorers"], RAW / "goalscorers.csv")
download(URLS["shootouts"], RAW / "shootouts.csv")

NULLS = ["", "NA", "N/A", "NULL", "null", "None"]

results_raw = pl.read_csv(
    RAW / "results.csv",
    null_values=NULLS,
    infer_schema_length=20000,
    schema_overrides={"home_score": pl.Int64, "away_score": pl.Int64, "neutral": pl.Boolean},
    try_parse_dates=True,
)

results = (
    results_raw
    .filter(pl.col("date") >= pl.date(1980, 1, 1))
    .filter(pl.col("home_score").is_not_null() & pl.col("away_score").is_not_null())
    .sort("date")
)

goalscorers = pl.read_csv(RAW / "goalscorers.csv", null_values=NULLS, infer_schema_length=20000, try_parse_dates=True)
shootouts = pl.read_csv(RAW / "shootouts.csv", null_values=NULLS, infer_schema_length=20000, try_parse_dates=True)

print("results:", results.shape)
print("goalscorers:", goalscorers.shape)
print("shootouts:", shootouts.shape)


PROJECT: C:\ml\projects\WeltmeisterKI
MODELS: C:\ml\projects\WeltmeisterKI\models
RAW: C:\ml\projects\WeltmeisterKI\data\raw
PROCESSED: C:\ml\projects\WeltmeisterKI\data\processed
TEMP: C:\ml\projects\WeltmeisterKI\.tmp
results: (37332, 9)
goalscorers: (47663, 8)
shootouts: (678, 5)


## 04. FIFA-Ranking

Normalisiert Ländernamen und baut die historische FIFA-Ranking-Tabelle. Ergebnis: fifa_rankings und fifa_before(...) für zeitpunktkorrekte Rankings.


In [34]:
# Was diese Zelle macht:
# Normalisiert Ländernamen und baut die historische FIFA-Ranking-Tabelle. Ergebnis: fifa_rankings und fifa_before(...) für zeitpunktkorrekte Rankings.

# Fifa Ranking
def norm_team(s):
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", "", s.lower().strip())
    aliases = {
        "usa": "unitedstates", "unitedstatesofamerica": "unitedstates",
        "deutschland": "germany", "westgermany": "germany", "germanyfr": "germany",
        "cotedivoire": "ivorycoast", "coteivoire": "ivorycoast",
        "ivorycoast": "ivorycoast", "elfenbeinkuste": "ivorycoast",
        "curacao": "curacao", "curaao": "curacao",
        "korearepublic": "southkorea", "republicofkorea": "southkorea",
        "iriran": "iran", "czechia": "czechrepublic",
        "congodr": "drcongo", "drcongo": "drcongo",
        "capeverdeislands": "capeverde", "chinapr": "china",
    }
    return aliases.get(s, s)

def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

def parse_any_date(v):
    if v is None:
        return None
    s = str(v).strip().replace("Sept", "Sep")
    if not s or s.lower() in {"nan", "none", "null"}:
        return None
    for fmt in ["%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y", "%d/%m/%Y", "%d %b %Y", "%d %B %Y"]:
        try:
            return datetime.strptime(s, fmt).date()
        except Exception:
            pass
    if re.fullmatch(r"\d+", s):
        n = int(s)
        if 19920101 <= n <= 20301231:
            return datetime.strptime(s, "%Y%m%d").date()
        if 10000 <= n <= 30000:
            return date(1970, 1, 1) + timedelta(days=n)
        if 1992 <= n <= 2030:
            return date(n, 12, 31)
    return None

def to_float(v):
    if v is None:
        return math.nan
    s = str(v).replace(",", "").strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return math.nan
    try:
        return float(s)
    except Exception:
        return math.nan

def pick(cols, names):
    for n in names:
        if n in cols:
            return cols[n]
    return None

# Falls die CSVs noch nicht da sind, KaggleHub probieren.
if not list(FIFA_RAW.glob("*.csv")):
    try:
        import kagglehub
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "kagglehub"], check=True)
        import kagglehub
    for ds in ["lucasyukioimafuko/fifa-mens-world-ranking", "cashncarry/fifaworldranking"]:
        try:
            p = Path(kagglehub.dataset_download(ds))
            for src in p.rglob("*.csv"):
                shutil.copy2(src, FIFA_RAW / src.name)
        except Exception as e:
            print("Kaggle failed:", ds, repr(e))

def read_fifa_csv(path):
    df = pl.read_csv(path, infer_schema=False, null_values=NULLS, ignore_errors=True)
    df = df.rename({c: norm_col(c) for c in df.columns})
    cols = {c: c for c in df.columns}

    date_col = pick(cols, ["rank_date", "ranking_date", "date"])
    country_col = pick(cols, ["country_full", "country", "team", "name"])
    rank_col = pick(cols, ["rank", "world_rank", "ranking"])
    points_col = pick(cols, ["total_points", "points", "rating", "score"])
    confed_col = pick(cols, ["confederation", "confed", "conf"])

    if not date_col or not country_col:
        return None

    rows = []
    for r in df.iter_rows(named=True):
        d = parse_any_date(r.get(date_col))
        if d is None or d < date(1992, 12, 1) or d > date(2026, 12, 31):
            continue
        rank = to_float(r.get(rank_col)) if rank_col else math.nan
        points = to_float(r.get(points_col)) if points_col else math.nan
        rows.append({
            "rank_date": d,
            "country_full": r.get(country_col),
            "rank": rank,
            "total_points": points,
            "confederation": str(r.get(confed_col) or "") if confed_col else "",
        })

    if not rows:
        return None
    out = pl.DataFrame(rows).drop_nulls(["country_full"])
    return out.filter(pl.col("rank").is_not_null())

parts = []
for f in FIFA_RAW.glob("*.csv"):
    part = read_fifa_csv(f)
    if part is not None and part.height:
        print(f.name, part.shape, part["rank_date"].min(), part["rank_date"].max())
        parts.append(part)

if not parts:
    raise RuntimeError("Keine historische FIFA-Ranking-CSV gefunden.")

fifa_rankings_hist = (
    pl.concat(parts, how="vertical_relaxed")
    .unique(subset=["rank_date", "country_full"], keep="last")
    .sort(["rank_date", "rank"])
)

# Aktuelles Ranking aus TXT neben Notebook.
current_text_path = Path.cwd() / "fifa_current_page_text_en.txt"
if not current_text_path.exists():
    raise FileNotFoundError("fifa_current_page_text_en.txt muss neben dem Notebook liegen.")

GENERIC = {"home team logo", "away team logo", "ft", "more", "news", "rankings", "match centre"}

def is_float_text(s):
    return re.fullmatch(r"[+-]?\d+\.\d+", s.replace(",", "")) is not None

def is_team_line(s):
    low = s.lower().strip()
    if not s or low in GENERIC or "logo" in low:
        return False
    if re.fullmatch(r"\d{1,3}", s) or re.fullmatch(r"\d{1,2}:\d{2}", s):
        return False
    if re.fullmatch(r"\d{1,2}\s+[A-Z][a-z]{2}", s):
        return False
    return not is_float_text(s)

def parse_current_fifa_text(path, current_date=date(2026, 6, 8)):
    lines = [x.strip() for x in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines() if x.strip()]
    headers = []
    pos = 0

    for rank in range(1, 260):
        found = None
        for i in range(pos, len(lines) - 1):
            if lines[i] != str(rank):
                continue
            for k in range(i + 1, min(i + 16, len(lines) - 1)):
                if lines[k] == lines[k + 1] and is_team_line(lines[k]):
                    found = {"rank": rank, "idx": i, "team_idx": k, "team": lines[k]}
                    break
            if found:
                break
        if found:
            headers.append(found)
            pos = found["team_idx"] + 2

    rows = []
    for n, h in enumerate(headers):
        start = h["team_idx"] + 2
        end = headers[n + 1]["idx"] if n + 1 < len(headers) else len(lines)
        floats = []
        for w in lines[start:end]:
            s = w.replace(",", "")
            if is_float_text(s):
                val = float(s)
                if 600 <= val <= 2200:
                    floats.append(val)
        if floats:
            rows.append({
                "rank_date": current_date,
                "country_full": h["team"],
                "rank": float(h["rank"]),
                "total_points": float(floats[-1]),
                "confederation": "",
            })

    return pl.DataFrame(rows).unique(subset=["rank_date", "rank"], keep="first").sort("rank")

fifa_current = parse_current_fifa_text(current_text_path)
print("current rows:", fifa_current.shape)
if fifa_current.height < 180:
    raise RuntimeError("Aktuelle FIFA-TXT wurde nicht komplett geparst. Datei prüfen.")

fifa_rankings = (
    pl.concat([fifa_rankings_hist, fifa_current], how="vertical_relaxed")
    .unique(subset=["rank_date", "country_full"], keep="last")
    .sort(["rank_date", "rank"])
)

fifa_rankings.write_parquet(PROCESSED / "fifa_rankings.parquet")

rank_hist = defaultdict(list)
for r in fifa_rankings.iter_rows(named=True):
    rank_hist[norm_team(r["country_full"])].append((
        r["rank_date"],
        float(r["rank"]),
        float(r["total_points"]) if r["total_points"] is not None else math.nan,
        str(r["confederation"] or ""),
    ))

for t in rank_hist:
    rank_hist[t].sort(key=lambda x: x[0])

def fifa_before(team_norm, d):
    arr = rank_hist.get(team_norm, [])
    if not arr:
        return math.nan, math.nan, ""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid][0] <= d:
            lo = mid + 1
        else:
            hi = mid
    idx = lo - 1
    if idx < 0:
        return math.nan, math.nan, ""
    _, rank, points, confed = arr[idx]
    return rank, points, confed

for team in ["Germany", "Curacao", "Curaçao", "Ivory Coast", "Ecuador", "Argentina"]:
    print(team, norm_team(team), fifa_before(norm_team(team), date(2026, 6, 14)))

print("FIFA range:", fifa_rankings["rank_date"].min(), fifa_rankings["rank_date"].max())
print("Teams with FIFA history:", len(rank_hist))


fifa_mens_rank.csv (13130, 5) 1992-12-31 2024-12-31
fifa_ranking-2023-07-20.csv (64757, 5) 1992-12-31 2023-07-20
fifa_ranking-2024-04-04.csv (67261, 5) 1992-12-31 2024-04-04
fifa_ranking-2024-06-20.csv (67472, 5) 1992-12-31 2024-06-20
current rows: (210, 5)
Germany germany (10.0, 1735.77, '')
Curacao curacao (82.0, 1294.77, '')
Curaçao curacao (82.0, 1294.77, '')
Ivory Coast ivorycoast (33.0, 1540.87, '')
Ecuador ecuador (23.0, 1598.52, '')
Argentina argentina (1.0, 1901.48, '')
FIFA range: 1992-12-31 2026-06-08
Teams with FIFA history: 225


## 06. StatsBomb-xG

Lädt StatsBomb Open Data für verfügbare WM-Spiele und baut xG/Shot/Pressure-Lookups. Ergebnis: sb_map für erweiterte Match-Features, wo Daten existieren.


In [35]:
# Was diese Zelle macht:
# Lädt StatsBomb Open Data für verfügbare WM-Spiele und baut xG/Shot/Pressure-Lookups. Ergebnis: sb_map für erweiterte Match-Features, wo Daten existieren.

# StatsBomb xG
def fetch_json(url, out=None):
    out = Path(out) if out else None
    if out and out.exists():
        return json.loads(out.read_text(encoding="utf-8"))
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read().decode("utf-8"))
    if out:
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(data), encoding="utf-8")
    return data

sb_path = PROCESSED / "statsbomb_team_match.parquet"

if sb_path.exists():
    sb = pl.read_parquet(sb_path)
else:
    competitions = fetch_json(URLS["sb_competitions"], SB_RAW / "competitions.json")
    wc = [c for c in competitions if c.get("competition_name") == "FIFA World Cup" and c.get("competition_gender", "male") == "male"]

    sb_rows = []
    for c in wc:
        comp_id, season_id = c["competition_id"], c["season_id"]
        matches_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/{comp_id}/{season_id}.json"
        matches = fetch_json(matches_url, SB_RAW / "matches" / f"{comp_id}_{season_id}.json")

        for m in matches:
            mid = m["match_id"]
            events_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{mid}.json"
            events = fetch_json(events_url, SB_RAW / "events" / f"{mid}.json")

            by_team = defaultdict(lambda: {"xg": 0.0, "shots": 0, "sot": 0, "passes": 0, "pressures": 0})
            for e in events:
                team = e.get("team", {}).get("name")
                typ = e.get("type", {}).get("name")
                if not team:
                    continue
                if typ == "Shot":
                    shot = e.get("shot", {})
                    by_team[team]["shots"] += 1
                    by_team[team]["xg"] += float(shot.get("statsbomb_xg") or 0.0)
                    if shot.get("outcome", {}).get("name") in {"Goal", "Saved", "Saved To Post"}:
                        by_team[team]["sot"] += 1
                elif typ == "Pass":
                    by_team[team]["passes"] += 1
                elif typ == "Pressure":
                    by_team[team]["pressures"] += 1

            d = datetime.fromisoformat(m["match_date"]).date()
            home = m["home_team"]["home_team_name"]
            away = m["away_team"]["away_team_name"]

            for team, opp in [(home, away), (away, home)]:
                v = by_team[team]
                sb_rows.append({
                    "date": d,
                    "team_norm": norm_team(team),
                    "opp_norm": norm_team(opp),
                    "sb_xg_for": v["xg"],
                    "sb_shots_for": v["shots"],
                    "sb_sot_for": v["sot"],
                    "sb_passes_for": v["passes"],
                    "sb_pressures_for": v["pressures"],
                })

    sb = pl.DataFrame(sb_rows)
    sb.write_parquet(sb_path)

print("StatsBomb:", sb.shape)


StatsBomb: (294, 8)


## 08. Feature Builder v2

Baut die zentrale Feature-Tabelle ohne Data Leakage: Elo, FIFA, Form, H2H, xG, Scorer, Shootouts sowie Attack/Defense-Ratings. Ergebnis: features und state für spätere Fixtures.


In [36]:
# Was diese Zelle macht:
# Baut die zentrale Feature-Tabelle ohne Data Leakage: Elo, FIFA, Form, H2H, xG, Scorer, Shootouts sowie Attack/Defense-Ratings. Ergebnis: features und state für spätere Fixtures.

from collections import defaultdict, deque
import math
import numpy as np
import polars as pl
from datetime import date

BASE_GOALS_PER_TEAM = 1.35
ATT_DEF_ALPHA = 0.055

if "sb_map" not in globals():
    sb_map = {}

if "shootout_map" not in globals():
    shootout_map = {}

def safe_div(a, b):
    return float(a / b) if b else np.nan

def result_points(gf, ga):
    if gf > ga:
        return 1.0
    if gf == ga:
        return 0.5
    return 0.0

def elo_expected(a, b):
    return 1.0 / (1.0 + 10 ** ((b - a) / 400.0))

def seq_avg(seq, fn, n):
    xs = [fn(x) for x in list(seq)[-n:]]
    xs = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(xs)) if xs else np.nan

def days_since_last(seq, current_date):
    if not seq:
        return np.nan
    return float((current_date - seq[-1]["date"]).days)

def tournament_weight(t):
    s = str(t).lower()
    if s == "fifa world cup":
        return 3.0
    if "qualification" in s:
        return 1.35
    if any(x in s for x in ["uefa euro", "copa america", "african cup", "asian cup", "gold cup", "nations league"]):
        return 2.0
    if s == "friendly":
        return 0.8
    return 1.0

def empty_goal_stats():
    return {"scorers": set(), "pen": 0, "own": 0}

def get_goal_stats(d, h, a, team):
    try:
        return goal_map[(d, h, a)][team]
    except Exception:
        return empty_goal_stats()

def init_state_v2():
    return {
        "elo": defaultdict(lambda: 1500.0),
        "attack": defaultdict(lambda: 1.0),
        "def_weak": defaultdict(lambda: 1.0),
        "team_hist": defaultdict(lambda: deque(maxlen=60)),
        "xg_hist": defaultdict(lambda: deque(maxlen=30)),
        "scorer_hist": defaultdict(lambda: deque(maxlen=30)),
        "h2h": defaultdict(lambda: deque(maxlen=25)),
        "shoot_w": defaultdict(int),
        "shoot_l": defaultdict(int),
    }

def snapshot_features_v2(home, away, d, tournament="FIFA World Cup", country="United States", neutral=True, state=None):
    hn, an = norm_team(home), norm_team(away)

    elo = state["elo"]
    attack = state["attack"]
    def_weak = state["def_weak"]
    team_hist = state["team_hist"]
    xg_hist = state["xg_hist"]
    scorer_hist = state["scorer_hist"]
    h2h = state["h2h"]
    shoot_w = state["shoot_w"]
    shoot_l = state["shoot_l"]

    hh, ah = team_hist[hn], team_hist[an]
    hx, ax = xg_hist[hn], xg_hist[an]
    hsc, asc = scorer_hist[hn], scorer_hist[an]
    h2h_ha = h2h[(hn, an)]

    h_rank, h_rank_pts, h_conf = fifa_before(hn, d)
    a_rank, a_rank_pts, a_conf = fifa_before(an, d)

    row = {
        "date": d,
        "home_team": home,
        "away_team": away,

        "year": d.year,
        "month": d.month,
        "days_from_1980": (d - date(1980, 1, 1)).days,

        "neutral": int(bool(neutral)),
        "home_country_host": int(norm_team(country) == hn and not neutral),
        "away_country_host": int(norm_team(country) == an and not neutral),

        "is_world_cup": int(tournament == "FIFA World Cup"),
        "is_qualifier": int("qualification" in str(tournament).lower()),
        "is_friendly": int(str(tournament).lower() == "friendly"),
        "is_major_tournament": int(any(x in str(tournament).lower() for x in [
            "fifa world cup", "uefa euro", "copa america", "african cup",
            "asian cup", "gold cup", "nations league"
        ])),

        "elo_home_pre": elo[hn],
        "elo_away_pre": elo[an],
        "elo_diff": elo[hn] - elo[an],
        "elo_ratio": safe_div(elo[hn], elo[an]),

        "home_attack_pre": attack[hn],
        "away_attack_pre": attack[an],
        "home_def_weak_pre": def_weak[hn],
        "away_def_weak_pre": def_weak[an],
        "attack_diff": attack[hn] - attack[an],
        "def_weak_diff": def_weak[hn] - def_weak[an],
        "home_attack_vs_away_def": attack[hn] * def_weak[an],
        "away_attack_vs_home_def": attack[an] * def_weak[hn],

        "home_fifa_rank": h_rank,
        "away_fifa_rank": a_rank,
        "fifa_rank_diff": h_rank - a_rank if not math.isnan(h_rank) and not math.isnan(a_rank) else np.nan,
        "home_fifa_points": h_rank_pts,
        "away_fifa_points": a_rank_pts,
        "fifa_points_diff": h_rank_pts - a_rank_pts if not math.isnan(h_rank_pts) and not math.isnan(a_rank_pts) else np.nan,
        "same_confed": int(h_conf != "" and h_conf == a_conf),

        "home_games_pre": len(hh),
        "away_games_pre": len(ah),
        "home_days_rest": days_since_last(hh, d),
        "away_days_rest": days_since_last(ah, d),
        "rest_diff": days_since_last(hh, d) - days_since_last(ah, d) if hh and ah else np.nan,

        "home_shootout_wins_pre": shoot_w[hn],
        "away_shootout_wins_pre": shoot_w[an],
        "home_shootout_losses_pre": shoot_l[hn],
        "away_shootout_losses_pre": shoot_l[an],
    }

    for n in [3, 5, 10, 20]:
        for side, hist in [("home", hh), ("away", ah)]:
            row[f"{side}_gf_l{n}"] = seq_avg(hist, lambda x: x["gf"], n)
            row[f"{side}_ga_l{n}"] = seq_avg(hist, lambda x: x["ga"], n)
            row[f"{side}_gd_l{n}"] = seq_avg(hist, lambda x: x["gf"] - x["ga"], n)
            row[f"{side}_pts_l{n}"] = seq_avg(hist, lambda x: x["pts"], n)
            row[f"{side}_adj_gf_l{n}"] = seq_avg(hist, lambda x: x["adj_gf"], n)
            row[f"{side}_adj_ga_l{n}"] = seq_avg(hist, lambda x: x["adj_ga"], n)
            row[f"{side}_adj_gd_l{n}"] = seq_avg(hist, lambda x: x["adj_gf"] - x["adj_ga"], n)
            row[f"{side}_adj_pts_l{n}"] = seq_avg(hist, lambda x: x["adj_pts"], n)
            row[f"{side}_win_rate_l{n}"] = seq_avg(hist, lambda x: int(x["pts"] == 1.0), n)
            row[f"{side}_loss_rate_l{n}"] = seq_avg(hist, lambda x: int(x["pts"] == 0.0), n)
            row[f"{side}_clean_sheet_l{n}"] = seq_avg(hist, lambda x: int(x["ga"] == 0), n)
            row[f"{side}_failed_score_l{n}"] = seq_avg(hist, lambda x: int(x["gf"] == 0), n)

        row[f"form_pts_diff_l{n}"] = row[f"home_pts_l{n}"] - row[f"away_pts_l{n}"] if not math.isnan(row[f"home_pts_l{n}"]) and not math.isnan(row[f"away_pts_l{n}"]) else np.nan
        row[f"form_gd_diff_l{n}"] = row[f"home_gd_l{n}"] - row[f"away_gd_l{n}"] if not math.isnan(row[f"home_gd_l{n}"]) and not math.isnan(row[f"away_gd_l{n}"]) else np.nan
        row[f"adj_form_pts_diff_l{n}"] = row[f"home_adj_pts_l{n}"] - row[f"away_adj_pts_l{n}"] if not math.isnan(row[f"home_adj_pts_l{n}"]) and not math.isnan(row[f"away_adj_pts_l{n}"]) else np.nan
        row[f"adj_form_gd_diff_l{n}"] = row[f"home_adj_gd_l{n}"] - row[f"away_adj_gd_l{n}"] if not math.isnan(row[f"home_adj_gd_l{n}"]) and not math.isnan(row[f"away_adj_gd_l{n}"]) else np.nan

    for n in [3, 5, 10]:
        for side, hist in [("home", hx), ("away", ax)]:
            row[f"{side}_xg_for_l{n}"] = seq_avg(hist, lambda x: x["xg_for"], n)
            row[f"{side}_xg_against_l{n}"] = seq_avg(hist, lambda x: x["xg_against"], n)
            row[f"{side}_xg_diff_l{n}"] = seq_avg(hist, lambda x: x["xg_for"] - x["xg_against"], n)
            row[f"{side}_shots_l{n}"] = seq_avg(hist, lambda x: x["shots"], n)
            row[f"{side}_sot_l{n}"] = seq_avg(hist, lambda x: x["sot"], n)
            row[f"{side}_passes_l{n}"] = seq_avg(hist, lambda x: x["passes"], n)
            row[f"{side}_pressures_l{n}"] = seq_avg(hist, lambda x: x["pressures"], n)

    for n in [5, 10, 20]:
        row[f"home_unique_scorers_l{n}"] = seq_avg(hsc, lambda x: x["unique_scorers"], n)
        row[f"away_unique_scorers_l{n}"] = seq_avg(asc, lambda x: x["unique_scorers"], n)
        row[f"home_pen_goals_l{n}"] = seq_avg(hsc, lambda x: x["pen"], n)
        row[f"away_pen_goals_l{n}"] = seq_avg(asc, lambda x: x["pen"], n)
        row[f"home_own_goals_l{n}"] = seq_avg(hsc, lambda x: x["own"], n)
        row[f"away_own_goals_l{n}"] = seq_avg(asc, lambda x: x["own"], n)

    for n in [3, 5, 10]:
        row[f"h2h_home_pts_l{n}"] = seq_avg(h2h_ha, lambda x: x["pts"], n)
        row[f"h2h_home_gd_l{n}"] = seq_avg(h2h_ha, lambda x: x["gd"], n)
        row[f"h2h_games_l{n}"] = min(len(h2h_ha), n)

    row["sample_weight"] = tournament_weight(tournament) * (0.75 + (d.year - 1980) / 60.0)

    return row

def update_state_after_match_v2(r, state):
    d = r["date"]
    h = r["home_team"]
    a = r["away_team"]
    hn = norm_team(h)
    an = norm_team(a)
    hs = int(r["home_score"])
    aw = int(r["away_score"])
    t = r["tournament"]

    elo = state["elo"]
    attack = state["attack"]
    def_weak = state["def_weak"]
    team_hist = state["team_hist"]
    xg_hist = state["xg_hist"]
    scorer_hist = state["scorer_hist"]
    h2h = state["h2h"]
    shoot_w = state["shoot_w"]
    shoot_l = state["shoot_l"]

    h_elo_pre = elo[hn]
    a_elo_pre = elo[an]

    hp = result_points(hs, aw)
    ap = result_points(aw, hs)

    h_adj_gf = hs * (a_elo_pre / 1500.0)
    h_adj_ga = aw * (1500.0 / max(1.0, a_elo_pre))
    a_adj_gf = aw * (h_elo_pre / 1500.0)
    a_adj_ga = hs * (1500.0 / max(1.0, h_elo_pre))

    team_hist[hn].append({
        "date": d, "gf": hs, "ga": aw, "pts": hp,
        "adj_gf": h_adj_gf, "adj_ga": h_adj_ga, "adj_pts": hp * (a_elo_pre / 1500.0),
    })
    team_hist[an].append({
        "date": d, "gf": aw, "ga": hs, "pts": ap,
        "adj_gf": a_adj_gf, "adj_ga": a_adj_ga, "adj_pts": ap * (h_elo_pre / 1500.0),
    })

    h_sb = sb_map.get((d, hn, an))
    a_sb = sb_map.get((d, an, hn))
    if h_sb and a_sb:
        xg_hist[hn].append({"xg_for": h_sb["sb_xg_for"], "xg_against": a_sb["sb_xg_for"], "shots": h_sb["sb_shots_for"], "sot": h_sb["sb_sot_for"], "passes": h_sb["sb_passes_for"], "pressures": h_sb["sb_pressures_for"]})
        xg_hist[an].append({"xg_for": a_sb["sb_xg_for"], "xg_against": h_sb["sb_xg_for"], "shots": a_sb["sb_shots_for"], "sot": a_sb["sb_sot_for"], "passes": a_sb["sb_passes_for"], "pressures": a_sb["sb_pressures_for"]})

    gh = get_goal_stats(d, h, a, h)
    ga = get_goal_stats(d, h, a, a)
    scorer_hist[hn].append({"unique_scorers": len(gh["scorers"]), "pen": gh["pen"], "own": gh["own"]})
    scorer_hist[an].append({"unique_scorers": len(ga["scorers"]), "pen": ga["pen"], "own": ga["own"]})

    h2h[(hn, an)].append({"pts": hp, "gd": hs - aw})
    h2h[(an, hn)].append({"pts": ap, "gd": aw - hs})

    winner = shootout_map.get((d, h, a))
    if winner:
        wn = norm_team(winner)
        loser = an if wn == hn else hn
        shoot_w[wn] += 1
        shoot_l[loser] += 1

    attack[hn] = (1 - ATT_DEF_ALPHA) * attack[hn] + ATT_DEF_ALPHA * max(0.15, h_adj_gf / BASE_GOALS_PER_TEAM)
    attack[an] = (1 - ATT_DEF_ALPHA) * attack[an] + ATT_DEF_ALPHA * max(0.15, a_adj_gf / BASE_GOALS_PER_TEAM)
    def_weak[hn] = (1 - ATT_DEF_ALPHA) * def_weak[hn] + ATT_DEF_ALPHA * max(0.15, h_adj_ga / BASE_GOALS_PER_TEAM)
    def_weak[an] = (1 - ATT_DEF_ALPHA) * def_weak[an] + ATT_DEF_ALPHA * max(0.15, a_adj_ga / BASE_GOALS_PER_TEAM)

    k = 20 * tournament_weight(t) * (1 + abs(hs - aw) / 4)
    eh = elo_expected(elo[hn], elo[an])
    elo[hn] += k * (hp - eh)
    elo[an] += k * (ap - (1 - eh))

state = init_state_v2()
rows = []

for r in results.sort("date").iter_rows(named=True):
    row = snapshot_features_v2(
        r["home_team"], r["away_team"], r["date"],
        tournament=r["tournament"],
        country=r["country"],
        neutral=bool(r["neutral"]),
        state=state,
    )
    row["home_score"] = int(r["home_score"])
    row["away_score"] = int(r["away_score"])
    rows.append(row)
    update_state_after_match_v2(r, state)

features = pl.DataFrame(rows).sort("date")
features.write_parquet(PROCESSED / "features_1980_plus_v2.parquet")

print("Features v2:", features.shape)
print("New feature examples:", [c for c in features.columns if "adj_" in c or "attack" in c or "def_weak" in c][:40])
features.head()


Features v2: (37332, 225)
New feature examples: ['home_attack_pre', 'away_attack_pre', 'home_def_weak_pre', 'away_def_weak_pre', 'attack_diff', 'def_weak_diff', 'home_attack_vs_away_def', 'away_attack_vs_home_def', 'home_adj_gf_l3', 'home_adj_ga_l3', 'home_adj_gd_l3', 'home_adj_pts_l3', 'away_adj_gf_l3', 'away_adj_ga_l3', 'away_adj_gd_l3', 'away_adj_pts_l3', 'adj_form_pts_diff_l3', 'adj_form_gd_diff_l3', 'home_adj_gf_l5', 'home_adj_ga_l5', 'home_adj_gd_l5', 'home_adj_pts_l5', 'away_adj_gf_l5', 'away_adj_ga_l5', 'away_adj_gd_l5', 'away_adj_pts_l5', 'adj_form_pts_diff_l5', 'adj_form_gd_diff_l5', 'home_adj_gf_l10', 'home_adj_ga_l10', 'home_adj_gd_l10', 'home_adj_pts_l10', 'away_adj_gf_l10', 'away_adj_ga_l10', 'away_adj_gd_l10', 'away_adj_pts_l10', 'adj_form_pts_diff_l10', 'adj_form_gd_diff_l10', 'home_adj_gf_l20', 'home_adj_ga_l20']


date,home_team,away_team,year,month,days_from_1980,neutral,home_country_host,away_country_host,is_world_cup,is_qualifier,is_friendly,is_major_tournament,elo_home_pre,elo_away_pre,…,away_pen_goals_l20,home_own_goals_l20,away_own_goals_l20,h2h_home_pts_l3,h2h_home_gd_l3,h2h_games_l3,h2h_home_pts_l5,h2h_home_gd_l5,h2h_games_l5,h2h_home_pts_l10,h2h_home_gd_l10,h2h_games_l10,sample_weight,home_score,away_score
date,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,…,f64,f64,f64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,i64
1980-01-06,"""Sierra Leone""","""Ghana""",1980,1,5,0,1,0,0,0,1,0,1500.0,1500.0,…,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,0.6,2,4
1980-01-16,"""Cyprus""","""Greece""",1980,1,15,0,1,0,0,0,1,0,1500.0,1500.0,…,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,0.6,1,1
1980-01-19,"""Congo""","""Ivory Coast""",1980,1,18,0,1,0,0,0,1,0,1500.0,1500.0,…,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,0.6,2,0
1980-01-20,"""Congo""","""Ivory Coast""",1980,1,19,0,1,0,0,0,1,0,1512.0,1488.0,…,0.0,0.0,0.0,1.0,2.0,1,1.0,2.0,1,1.0,2.0,1,0.6,0,2
1980-01-23,"""Spain""","""Netherlands""",1980,1,22,0,1,0,0,0,1,0,1500.0,1500.0,…,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,0.6,1,0


## 10. v2 Training und ehrlicher Backtest

Trainiert das XGBoost-v2-Ensemble, macht Random Search, Rolling Backtest, 2022+-Holdout-Test und speichert das Production-Modell.


In [37]:
# Was diese Zelle macht:
# Trainiert das XGBoost-v2-Ensemble, macht Random Search, Rolling Backtest, 2022+-Holdout-Test und speichert das Production-Modell.

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, accuracy_score, log_loss
from xgboost import XGBRegressor, XGBClassifier
import numpy as np
import polars as pl
import joblib
import math
import time

N_TUNING_TRIALS = 35
N_ENSEMBLE = 7
BACKTEST_ENSEMBLE_N = 3
RANDOM_SEED = 42
MAX_GOALS_EVAL = 10

exclude = {"date", "home_team", "away_team", "home_score", "away_score", "sample_weight"}
candidate_cols = [c for c in features.columns if c not in exclude]

feature_cols = []
dropped_cols = []

for c in candidate_cols:
    try:
        arr = features[c].to_numpy().astype(float)
        if np.isfinite(arr).sum() == 0:
            dropped_cols.append(c)
        else:
            feature_cols.append(c)
    except Exception:
        dropped_cols.append(c)

df = features.drop_nulls(["home_score", "away_score"]).sort("date")

print("Using features:", len(feature_cols))
print("FIFA features:", [c for c in feature_cols if "fifa" in c])
print("Attack/def features:", [c for c in feature_cols if "attack" in c or "def_weak" in c])
print("Adjusted form features:", [c for c in feature_cols if "adj_" in c][:20])

def y_outcome_from_scores(home_scores, away_scores):
    y = []
    for h, a in zip(home_scores, away_scores):
        if h > a:
            y.append(0)
        elif h == a:
            y.append(1)
        else:
            y.append(2)
    return np.array(y, dtype=int)

def poisson_grid_single(home_lam, away_lam, max_goals=10, lambda_scale=1.0, draw_boost=1.0):
    hlam = float(max(0.03, min(7.0, home_lam * lambda_scale)))
    alam = float(max(0.03, min(7.0, away_lam * lambda_scale)))

    hp = np.array([math.exp(-hlam) * (hlam ** k) / math.factorial(k) for k in range(max_goals + 1)], dtype=float)
    ap = np.array([math.exp(-alam) * (alam ** k) / math.factorial(k) for k in range(max_goals + 1)], dtype=float)
    hp /= hp.sum()
    ap /= ap.sum()

    grid = np.outer(hp, ap)

    if draw_boost != 1.0:
        for i in range(max_goals + 1):
            grid[i, i] *= draw_boost
        grid /= grid.sum()

    p_home = float(np.tril(grid, -1).sum())
    p_draw = float(np.trace(grid))
    p_away = float(np.triu(grid, 1).sum())
    return np.array([p_home, p_draw, p_away], dtype=float)

def poisson_outcome_matrix(home_pred, away_pred, lambda_scale=1.0, draw_boost=1.0):
    return np.vstack([
        poisson_grid_single(h, a, MAX_GOALS_EVAL, lambda_scale, draw_boost)
        for h, a in zip(home_pred, away_pred)
    ])

def multiclass_brier(y_true, probs):
    y_one = np.zeros_like(probs)
    y_one[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((probs - y_one) ** 2, axis=1)))

def calibration_table(y_true, probs, bins=10):
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y_true).astype(float)
    rows = []
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if mask.sum() == 0:
            continue
        rows.append({
            "bin": f"{lo:.1f}-{hi:.1f}",
            "n": int(mask.sum()),
            "avg_conf": round(float(conf[mask].mean()), 4),
            "actual_acc": round(float(correct[mask].mean()), 4),
            "gap": round(float(conf[mask].mean() - correct[mask].mean()), 4),
        })
    return pl.DataFrame(rows)

def eval_probs(name, y_true, probs, home_true, away_true, home_pred, away_pred):
    return {
        "split": name,
        "n": len(y_true),
        "home_mae": float(mean_absolute_error(home_true, home_pred)),
        "away_mae": float(mean_absolute_error(away_true, away_pred)),
        "1x2_accuracy": float(accuracy_score(y_true, probs.argmax(axis=1))),
        "brier": multiclass_brier(y_true, probs),
        "log_loss": float(log_loss(y_true, probs, labels=[0, 1, 2])),
    }

def sample_params(rng):
    return {
        "objective": "count:poisson",
        "eval_metric": "poisson-nloglik",
        "n_estimators": int(rng.choice([650, 850, 1050, 1250, 1500])),
        "max_depth": int(rng.choice([2, 3, 4, 5])),
        "learning_rate": float(rng.choice([0.01, 0.015, 0.02, 0.03, 0.04])),
        "subsample": float(rng.uniform(0.70, 0.95)),
        "colsample_bytree": float(rng.uniform(0.70, 0.95)),
        "reg_lambda": float(10 ** rng.uniform(0.2, 1.15)),
        "reg_alpha": float(10 ** rng.uniform(-3, 0.5)),
        "min_child_weight": float(rng.choice([1, 2, 3, 5, 8])),
        "tree_method": "hist",
        "n_jobs": -1,
    }

def clf_params_from_reg(params, seed):
    return {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "n_estimators": min(int(params["n_estimators"]), 900),
        "max_depth": int(params["max_depth"]),
        "learning_rate": float(params["learning_rate"]),
        "subsample": float(params["subsample"]),
        "colsample_bytree": float(params["colsample_bytree"]),
        "reg_lambda": float(params["reg_lambda"]),
        "reg_alpha": float(params["reg_alpha"]),
        "min_child_weight": float(params["min_child_weight"]),
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": seed,
    }

def fit_ensemble(train_df, feature_cols, params, n_models, seed0=42):
    imp = SimpleImputer(strategy="median")
    X = imp.fit_transform(train_df.select(feature_cols).to_numpy())

    yh = train_df["home_score"].to_numpy()
    ya = train_df["away_score"].to_numpy()
    yo = y_outcome_from_scores(yh, ya)
    sw = train_df["sample_weight"].to_numpy()

    home_models, away_models, outcome_models = [], [], []

    for i in range(n_models):
        seed = seed0 + i * 17
        p = dict(params)
        p["random_state"] = seed

        hm = XGBRegressor(**p)
        am = XGBRegressor(**p)
        cm = XGBClassifier(**clf_params_from_reg(params, seed))

        hm.fit(X, yh, sample_weight=sw, verbose=False)
        am.fit(X, ya, sample_weight=sw, verbose=False)
        cm.fit(X, yo, sample_weight=sw, verbose=False)

        home_models.append(hm)
        away_models.append(am)
        outcome_models.append(cm)

    return imp, home_models, away_models, outcome_models

def predict_ensemble(imp, home_models, away_models, outcome_models, data_df, feature_cols):
    X = imp.transform(data_df.select(feature_cols).to_numpy())

    hp = np.mean([m.predict(X) for m in home_models], axis=0)
    ap = np.mean([m.predict(X) for m in away_models], axis=0)
    hp = np.clip(hp, 0.03, 7.0)
    ap = np.clip(ap, 0.03, 7.0)

    cp = np.mean([m.predict_proba(X) for m in outcome_models], axis=0)
    cp = np.clip(cp, 1e-6, 1.0)
    cp = cp / cp.sum(axis=1, keepdims=True)

    return hp, ap, cp

def tune_calibration(y_true, home_pred, away_pred, clf_probs):
    best = None
    for lambda_scale in [0.90, 0.95, 1.00, 1.05, 1.10]:
        for draw_boost in [0.85, 1.0, 1.10, 1.20, 1.35, 1.50]:
            p_pois = poisson_outcome_matrix(home_pred, away_pred, lambda_scale, draw_boost)
            for poisson_weight in [0.45, 0.55, 0.65, 0.75, 0.85, 0.95]:
                probs = poisson_weight * p_pois + (1.0 - poisson_weight) * clf_probs
                probs = np.clip(probs, 1e-6, 1.0)
                probs = probs / probs.sum(axis=1, keepdims=True)
                ll = log_loss(y_true, probs, labels=[0, 1, 2])
                if best is None or ll < best["log_loss"]:
                    best = {
                        "lambda_scale": lambda_scale,
                        "draw_boost": draw_boost,
                        "poisson_weight": poisson_weight,
                        "log_loss": float(ll),
                    }
    return best

def final_probs_from_parts(home_pred, away_pred, clf_probs, calibration):
    p_pois = poisson_outcome_matrix(
        home_pred,
        away_pred,
        calibration["lambda_scale"],
        calibration["draw_boost"],
    )
    w = calibration["poisson_weight"]
    probs = w * p_pois + (1.0 - w) * clf_probs
    probs = np.clip(probs, 1e-6, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs

# Random Search auf 2018-2021

train_tune = df.filter(pl.col("date") < pl.date(2018, 1, 1))
val_tune = df.filter((pl.col("date") >= pl.date(2018, 1, 1)) & (pl.col("date") < pl.date(2022, 1, 1)))
test_main = df.filter(pl.col("date") >= pl.date(2022, 1, 1))

rng = np.random.default_rng(RANDOM_SEED)

imp0 = SimpleImputer(strategy="median")
X_train = imp0.fit_transform(train_tune.select(feature_cols).to_numpy())
X_val = imp0.transform(val_tune.select(feature_cols).to_numpy())

y_train_h = train_tune["home_score"].to_numpy()
y_train_a = train_tune["away_score"].to_numpy()
sw_train = train_tune["sample_weight"].to_numpy()
y_val = y_outcome_from_scores(val_tune["home_score"].to_numpy(), val_tune["away_score"].to_numpy())

best = None
trial_rows = []

start = time.time()
for trial in range(1, N_TUNING_TRIALS + 1):
    params = sample_params(rng)

    hm = XGBRegressor(**params, random_state=1000 + trial)
    am = XGBRegressor(**params, random_state=2000 + trial)

    hm.fit(X_train, y_train_h, sample_weight=sw_train, verbose=False)
    am.fit(X_train, y_train_a, sample_weight=sw_train, verbose=False)

    ph = np.clip(hm.predict(X_val), 0.03, 7.0)
    pa = np.clip(am.predict(X_val), 0.03, 7.0)
    probs = poisson_outcome_matrix(ph, pa)

    ll = log_loss(y_val, probs, labels=[0, 1, 2])
    brier = multiclass_brier(y_val, probs)
    acc = accuracy_score(y_val, probs.argmax(axis=1))

    row = {"trial": trial, "log_loss": ll, "brier": brier, "acc": acc, **params}
    trial_rows.append(row)

    if best is None or ll < best["log_loss"]:
        best = {"log_loss": float(ll), "params": params, "trial": trial}

    print(f"Trial {trial:02d}/{N_TUNING_TRIALS} | logloss={ll:.4f} | brier={brier:.4f} | acc={acc:.4f} | best={best['log_loss']:.4f}")

print("Tuning fertig in", round(time.time() - start, 1), "s")
print("Best trial:", best["trial"])
print("Best params:", best["params"])

tuning_results = pl.DataFrame(trial_rows).sort("log_loss")
print(tuning_results.head(10))

# Rolling Backtest

def evaluate_period(train_end_year, test_start_year, test_end_year):
    test_start = date(test_start_year, 1, 1)
    test_end = date(test_end_year + 1, 1, 1)

    inner_val_start = date(test_start_year - 4, 1, 1)

    inner_train = df.filter(pl.col("date") < inner_val_start)
    inner_val = df.filter((pl.col("date") >= inner_val_start) & (pl.col("date") < test_start))
    test = df.filter((pl.col("date") >= test_start) & (pl.col("date") < test_end))

    imp, hms, ams, cms = fit_ensemble(inner_train, feature_cols, best["params"], BACKTEST_ENSEMBLE_N, seed0=500 + test_start_year)

    vh, va, vclf = predict_ensemble(imp, hms, ams, cms, inner_val, feature_cols)
    yv = y_outcome_from_scores(inner_val["home_score"].to_numpy(), inner_val["away_score"].to_numpy())
    cal = tune_calibration(yv, vh, va, vclf)

    th, ta, tclf = predict_ensemble(imp, hms, ams, cms, test, feature_cols)
    yt = y_outcome_from_scores(test["home_score"].to_numpy(), test["away_score"].to_numpy())
    probs = final_probs_from_parts(th, ta, tclf, cal)

    return eval_probs(
        f"{test_start_year}-{test_end_year}",
        yt,
        probs,
        test["home_score"].to_numpy(),
        test["away_score"].to_numpy(),
        th,
        ta,
    ) | {
        "lambda_scale": cal["lambda_scale"],
        "draw_boost": cal["draw_boost"],
        "poisson_weight": cal["poisson_weight"],
    }

folds = [
    (2010, 2011, 2014),
    (2014, 2015, 2018),
    (2018, 2019, 2022),
    (2022, 2023, 2026),
]

backtest_rows = []
for train_end, ts, te in folds:
    print("Backtest fold:", ts, "-", te)
    backtest_rows.append(evaluate_period(train_end, ts, te))

backtest = pl.DataFrame(backtest_rows)
print("=== Rolling Backtest ===")
print(backtest)

# Finaler ehrlicher Test 2022+

imp_eval, hms_eval, ams_eval, cms_eval = fit_ensemble(train_tune, feature_cols, best["params"], N_ENSEMBLE, seed0=900)

vh, va, vclf = predict_ensemble(imp_eval, hms_eval, ams_eval, cms_eval, val_tune, feature_cols)
yv = y_outcome_from_scores(val_tune["home_score"].to_numpy(), val_tune["away_score"].to_numpy())
calibration = tune_calibration(yv, vh, va, vclf)

th, ta, tclf = predict_ensemble(imp_eval, hms_eval, ams_eval, cms_eval, test_main, feature_cols)
yt = y_outcome_from_scores(test_main["home_score"].to_numpy(), test_main["away_score"].to_numpy())
test_probs = final_probs_from_parts(th, ta, tclf, calibration)

main_metrics = eval_probs(
    "test_2022_plus",
    yt,
    test_probs,
    test_main["home_score"].to_numpy(),
    test_main["away_score"].to_numpy(),
    th,
    ta,
)

print("=== Main Test Metrics ===")
print(pl.DataFrame([main_metrics]))
print("Calibration:", calibration)

print("=== Confidence Calibration Test 2022+ ===")
print(calibration_table(yt, test_probs))

# Finalmodell auf allen bekannten Daten trainieren

print("Training final v2 ensemble auf allen Daten...")
final_imp, final_home_models, final_away_models, final_outcome_models = fit_ensemble(
    df,
    feature_cols,
    best["params"],
    N_ENSEMBLE,
    seed0=1200,
)

model_path = MODELS / "xgb_goal_models_v2_ensemble.joblib"
joblib.dump({
    "model_version": "v2_ensemble_attack_def_adjform_classifier_calibrated",
    "feature_cols": feature_cols,
    "dropped_cols": dropped_cols,
    "imputer": final_imp,
    "home_models": final_home_models,
    "away_models": final_away_models,
    "outcome_models": final_outcome_models,
    "params": best["params"],
    "calibration": calibration,
    "tuning_results": tuning_results,
    "backtest": backtest,
    "main_metrics": main_metrics,
}, model_path)

print("Saved:", model_path)


Using features: 177
FIFA features: ['home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff', 'home_fifa_points', 'away_fifa_points', 'fifa_points_diff']
Attack/def features: ['home_attack_pre', 'away_attack_pre', 'home_def_weak_pre', 'away_def_weak_pre', 'attack_diff', 'def_weak_diff', 'home_attack_vs_away_def', 'away_attack_vs_home_def']
Adjusted form features: ['home_adj_gf_l3', 'home_adj_ga_l3', 'home_adj_gd_l3', 'home_adj_pts_l3', 'away_adj_gf_l3', 'away_adj_ga_l3', 'away_adj_gd_l3', 'away_adj_pts_l3', 'adj_form_pts_diff_l3', 'adj_form_gd_diff_l3', 'home_adj_gf_l5', 'home_adj_ga_l5', 'home_adj_gd_l5', 'home_adj_pts_l5', 'away_adj_gf_l5', 'away_adj_ga_l5', 'away_adj_gd_l5', 'away_adj_pts_l5', 'adj_form_pts_diff_l5', 'adj_form_gd_diff_l5']
Trial 01/35 | logloss=0.8639 | brier=0.5075 | acc=0.6023 | best=0.8639
Trial 02/35 | logloss=0.8701 | brier=0.5114 | acc=0.5997 | best=0.8639
Trial 03/35 | logloss=0.8616 | brier=0.5062 | acc=0.6071 | best=0.8616
Trial 04/35 | logloss=0.8628 | brier=0

## 12. v2 Modellcheck

Lädt das gespeicherte v2-Modell und zeigt Metrics, Backtest und aktive Kernel-Variablen. Diese Zelle ist nur Diagnose, verändert das Modell nicht.


In [38]:
# Was diese Zelle macht:
# Lädt das gespeicherte v2-Modell und zeigt Metrics, Backtest und aktive Kernel-Variablen. Diese Zelle ist nur Diagnose, verändert das Modell nicht.

import polars as pl
import numpy as np
import joblib
from pathlib import Path

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_width_chars(280)
pl.Config.set_fmt_str_lengths(120)

model_path = MODELS / "xgb_goal_models_v2_ensemble.joblib"
bundle_check = joblib.load(model_path)

print("=== Aktuell erwartetes Modell ===")
print("Model path:", model_path)
print("Model version:", bundle_check.get("model_version"))
print("Features:", len(bundle_check.get("feature_cols", [])))
print("Calibration:", bundle_check.get("calibration"))

print("\n=== Sind die v2-Variablen im Notebook aktiv? ===")
checks = {
    "home_models": "home_models" in globals(),
    "away_models": "away_models" in globals(),
    "outcome_models": "outcome_models" in globals(),
    "calibration": "calibration" in globals(),
    "predict_neutral": "predict_neutral" in globals(),
    "raw_predict_home_away": "raw_predict_home_away" in globals(),
}

for k, v in checks.items():
    print(f"{k}: {v}")

if "home_models" in globals():
    print("Aktive home_models:", len(home_models))
if "outcome_models" in globals():
    print("Aktive outcome_models:", len(outcome_models))
if "calibration" in globals():
    print("Aktive calibration:", calibration)

print("\n=== Main Test Metrics 2022+ ===")
main_metrics_from_bundle = bundle_check.get("main_metrics")

if main_metrics_from_bundle is not None:
    print(pl.DataFrame([main_metrics_from_bundle]))
else:
    print("Keine main_metrics im Bundle gefunden.")

print("\n=== Rolling Backtest ===")
backtest_from_bundle = bundle_check.get("backtest")

if backtest_from_bundle is not None:
    print(backtest_from_bundle)
else:
    print("Kein backtest im Bundle gefunden.")

print("\n=== Tuning Top 10 ===")
tuning_results_from_bundle = bundle_check.get("tuning_results")

if tuning_results_from_bundle is not None:
    print(tuning_results_from_bundle.head(10))
else:
    print("Keine tuning_results im Bundle gefunden.")

print("\n=== Kurzinterpretation ===")

if main_metrics_from_bundle is not None:
    mm = main_metrics_from_bundle

    print(f"Home MAE: {mm['home_mae']:.4f}")
    print(f"Away MAE: {mm['away_mae']:.4f}")
    print(f"1X2 Accuracy: {100 * mm['1x2_accuracy']:.2f}%")
    print(f"Brier Score: {mm['brier']:.4f}")
    print(f"Log Loss: {mm['log_loss']:.4f}")

    print("\nLesart:")
    print("- 1X2 Accuracy höher = besser.")
    print("- Brier niedriger = bessere Wahrscheinlichkeiten.")
    print("- Log Loss niedriger = bessere und weniger überconfidente Wahrscheinlichkeiten.")
    print("- MAE niedriger = bessere Torprognose.")

if backtest_from_bundle is not None:
    bt = backtest_from_bundle

    print("\nRolling Backtest Mittelwerte:")
    numeric_cols = [
        "home_mae",
        "away_mae",
        "1x2_accuracy",
        "brier",
        "log_loss",
    ]

    available = [c for c in numeric_cols if c in bt.columns]

    means = {}
    for c in available:
        means[c] = float(bt[c].mean())

    print(pl.DataFrame([means]))

    print("\nFold-Stabilität:")
    for c in available:
        vals = bt[c].to_list()
        print(
            f"{c}: min={min(vals):.4f}, mean={np.mean(vals):.4f}, max={max(vals):.4f}"
        )


=== Aktuell erwartetes Modell ===
Model path: C:\ml\projects\WeltmeisterKI\models\xgb_goal_models_v2_ensemble.joblib
Model version: v2_ensemble_attack_def_adjform_classifier_calibrated
Features: 177
Calibration: {'lambda_scale': 1.1, 'draw_boost': 1.1, 'poisson_weight': 0.75, 'log_loss': 0.8577838343889644}

=== Sind die v2-Variablen im Notebook aktiv? ===
home_models: True
away_models: True
outcome_models: True
calibration: True
predict_neutral: True
raw_predict_home_away: True
Aktive home_models: 7
Aktive outcome_models: 7
Aktive calibration: {'lambda_scale': 1.1, 'draw_boost': 1.1, 'poisson_weight': 0.75, 'log_loss': 0.8577838343889644}

=== Main Test Metrics 2022+ ===
shape: (1, 7)
┌────────────────┬──────┬──────────┬──────────┬──────────────┬─────────┬──────────┐
│ split          ┆ n    ┆ home_mae ┆ away_mae ┆ 1x2_accuracy ┆ brier   ┆ log_loss │
│ ---            ┆ ---  ┆ ---      ┆ ---      ┆ ---          ┆ ---     ┆ ---      │
│ str            ┆ i64  ┆ f64      ┆ f64      ┆ f64  

## 14. v2 Prediction Helper

Lädt v2 aus der Joblib-Datei und definiert predict_match/predict_neutral plus score_grid. Diese Namen sind die stabile Schnittstelle für Forecasts und Monte Carlo.


In [39]:
# Was diese Zelle macht:
# Lädt v2 aus der Joblib-Datei und definiert predict_match/predict_neutral plus score_grid. Diese Namen sind die stabile Schnittstelle für Forecasts und Monte Carlo.

import joblib
import numpy as np
import math

bundle = joblib.load(MODELS / "xgb_goal_models_v2_ensemble.joblib")

feature_cols = bundle["feature_cols"]
imp = bundle["imputer"]
home_models = bundle["home_models"]
away_models = bundle["away_models"]
outcome_models = bundle["outcome_models"]
calibration = bundle["calibration"]

print("Loaded:", bundle["model_version"])
print("Features:", len(feature_cols))
print("Calibration:", calibration)

def predict_row_parts(row):
    x = np.array([[row.get(c, np.nan) for c in feature_cols]], dtype=float)
    x = imp.transform(x)

    home_xg = float(np.mean([m.predict(x)[0] for m in home_models]))
    away_xg = float(np.mean([m.predict(x)[0] for m in away_models]))

    home_xg = float(np.clip(home_xg, 0.03, 7.0))
    away_xg = float(np.clip(away_xg, 0.03, 7.0))

    clf_probs = np.mean([m.predict_proba(x)[0] for m in outcome_models], axis=0)
    clf_probs = np.clip(clf_probs, 1e-6, 1.0)
    clf_probs = clf_probs / clf_probs.sum()

    return home_xg, away_xg, clf_probs

def calibrated_score_grid(home_xg, away_xg, clf_probs=None, max_goals=10):
    lam_scale = calibration["lambda_scale"]
    draw_boost = calibration["draw_boost"]
    poisson_weight = calibration["poisson_weight"]

    home_xg = float(np.clip(home_xg, 0.03, 7.0))
    away_xg = float(np.clip(away_xg, 0.03, 7.0))

    hlam = float(np.clip(home_xg * lam_scale, 0.03, 7.0))
    alam = float(np.clip(away_xg * lam_scale, 0.03, 7.0))

    hp = np.array(
        [math.exp(-hlam) * hlam**k / math.factorial(k) for k in range(max_goals + 1)],
        dtype=float,
    )
    ap = np.array(
        [math.exp(-alam) * alam**k / math.factorial(k) for k in range(max_goals + 1)],
        dtype=float,
    )

    hp /= hp.sum()
    ap /= ap.sum()

    grid = np.outer(hp, ap)

    for i in range(max_goals + 1):
        grid[i, i] *= draw_boost

    grid /= grid.sum()

    p_home = float(np.tril(grid, -1).sum())
    p_draw = float(np.trace(grid))
    p_away = float(np.triu(grid, 1).sum())

    p_pois = np.array([p_home, p_draw, p_away], dtype=float)
    p_pois = np.clip(p_pois, 1e-9, 1.0)
    p_pois = p_pois / p_pois.sum()

    if clf_probs is None:
        final_outcome = p_pois
    else:
        clf_probs = np.asarray(clf_probs, dtype=float)
        clf_probs = np.clip(clf_probs, 1e-6, 1.0)
        clf_probs = clf_probs / clf_probs.sum()

        final_outcome = poisson_weight * p_pois + (1.0 - poisson_weight) * clf_probs
        final_outcome = np.clip(final_outcome, 1e-6, 1.0)
        final_outcome = final_outcome / final_outcome.sum()

        ratios = final_outcome / np.clip(p_pois, 1e-9, 1.0)

        for h in range(max_goals + 1):
            for a in range(max_goals + 1):
                if h > a:
                    grid[h, a] *= ratios[0]
                elif h == a:
                    grid[h, a] *= ratios[1]
                else:
                    grid[h, a] *= ratios[2]

        grid /= grid.sum()

    return grid, final_outcome

def raw_predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    """
    Kompatibel mit deiner alten Monte-Carlo-Zelle:
    Gibt weiterhin nur (home_xg, away_xg) zurück.
    """
    row = snapshot_features_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        state=state,
    )

    home_xg, away_xg, clf_probs = predict_row_parts(row)
    return home_xg, away_xg

def predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10):
    """
    Neuer interner Helper, aber mit neutralem Namen.
    Gibt vollen Detail-Output für home vs away.
    """
    row = snapshot_features_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        state=state,
    )

    home_xg, away_xg, clf_probs = predict_row_parts(row)
    grid, outcome_probs = calibrated_score_grid(
        home_xg,
        away_xg,
        clf_probs=clf_probs,
        max_goals=max_goals,
    )

    top = []
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            top.append((h, a, float(grid[h, a])))

    top = sorted(top, key=lambda x: x[2], reverse=True)

    return {
        "home": home,
        "away": away,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "p_home_win": float(outcome_probs[0]),
        "p_draw": float(outcome_probs[1]),
        "p_away_win": float(outcome_probs[2]),
        "most_likely_score": f"{top[0][0]}:{top[0][1]}",
        "most_likely_score_prob": top[0][2],
        "top_scorelines": top[:12],
        "score_grid": grid,
        "clf_probs": clf_probs,
    }

def predict_match(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10):
    """
    Alias für deine alte/naheliegende Schreibweise.
    """
    return predict_home_away(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
    )

def predict_neutral(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10):
    """
    Kompatibel mit deinen bestehenden Vorhersage-Zellen.

    Deine alten Zellen erwarten Keys wie:
    - team_xg_pred
    - opponent_xg_pred
    - p_team_win
    - p_draw
    - p_opponent_win
    - most_likely_score
    - most_likely_score_prob
    - top_scorelines
    """
    p1 = predict_home_away(
        team,
        opponent,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    p2 = predict_home_away(
        opponent,
        team,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    team_xg = (p1["home_xg"] + p2["away_xg"]) / 2
    opp_xg = (p1["away_xg"] + p2["home_xg"]) / 2

    p_team_win = (p1["p_home_win"] + p2["p_away_win"]) / 2
    p_draw = (p1["p_draw"] + p2["p_draw"]) / 2
    p_opp_win = (p1["p_away_win"] + p2["p_home_win"]) / 2

    probs = np.array([p_team_win, p_draw, p_opp_win], dtype=float)
    probs = np.clip(probs, 1e-6, 1.0)
    probs = probs / probs.sum()

    grid, _ = calibrated_score_grid(
        team_xg,
        opp_xg,
        clf_probs=probs,
        max_goals=max_goals,
    )

    top = []
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            top.append((h, a, float(grid[h, a])))

    top = sorted(top, key=lambda x: x[2], reverse=True)

    return {
        "team": team,
        "opponent": opponent,
        "team_xg_pred": float(team_xg),
        "opponent_xg_pred": float(opp_xg),
        "home_xg_pred": float(team_xg),
        "away_xg_pred": float(opp_xg),
        "p_team_win": float(probs[0]),
        "p_draw": float(probs[1]),
        "p_opponent_win": float(probs[2]),
        "p_home_win": float(probs[0]),
        "p_away_win": float(probs[2]),
        "most_likely_score": f"{top[0][0]}:{top[0][1]}",
        "most_likely_score_prob": top[0][2],
        "top_scorelines": top[:12],
        "score_grid": grid,
    }

# Alte v2-Namen bleiben optional auch verfügbar, falls irgendwo noch eine Zelle sie nutzt.
predict_match_v2 = predict_match
predict_neutral_v2 = predict_neutral

print("Prediction helpers ready:")
print("- raw_predict_home_away(...)")
print("- predict_home_away(...)")
print("- predict_match(...)")
print("- predict_neutral(...)")


Loaded: v2_ensemble_attack_def_adjform_classifier_calibrated
Features: 177
Calibration: {'lambda_scale': 1.1, 'draw_boost': 1.1, 'poisson_weight': 0.75, 'log_loss': 0.8577838343889644}
Prediction helpers ready:
- raw_predict_home_away(...)
- predict_home_away(...)
- predict_match(...)
- predict_neutral(...)


## 16. Deep v1 Daten und Sequenzen

Baut die Deep-Learning-Eingaben: numeric v2 features, Team-/Turnier-/Confed-IDs und Sequenzen der letzten Spiele pro Team.


In [40]:
# Was diese Zelle macht:
# Baut die Deep-Learning-Eingaben: numeric v2 features, Team-/Turnier-/Confed-IDs und Sequenzen der letzten Spiele pro Team.

# WeltmeisterKI Deep v1 - Daten, Embeddings, Sequenzen
# Muss nach der v2-Feature-Zelle laufen.
# Erwartet: features, PROCESSED, MODELS, norm_team

import math, random, joblib
import numpy as np
import polars as pl
from pathlib import Path
from datetime import date
from collections import defaultdict, deque

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if "features" not in globals():
    features = pl.read_parquet(PROCESSED / "features_1980_plus_v2.parquet")

if "norm_team" not in globals():
    import re, unicodedata
    def norm_team(x):
        x = str(x).strip()
        x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
        x = re.sub(r"\s+", " ", x)
        aliases = {
            "United States": "USA",
            "Korea Republic": "South Korea",
            "Cote dIvoire": "Ivory Coast",
            "Cote d'Ivoire": "Ivory Coast",
            "Curaçao": "Curacao",
        }
        return aliases.get(x, x)

def result_points(gf, ga):
    return 1.0 if gf > ga else 0.5 if gf == ga else 0.0

deep_df = (
    features
    .drop_nulls(["home_score", "away_score"])
    .sort("date")
    .with_row_index("deep_row_id")
)

# Falls dein Feature-DF keinen Turnier-String mehr hat, bauen wir eine Embedding-Kategorie aus Flags.
def tournament_bucket_from_row(r):
    if int(r.get("is_world_cup", 0) or 0) == 1:
        return "FIFA World Cup"
    if int(r.get("is_qualifier", 0) or 0) == 1:
        return "Qualifier"
    if int(r.get("is_friendly", 0) or 0) == 1:
        return "Friendly"
    if int(r.get("is_major_tournament", 0) or 0) == 1:
        return "Major Tournament"
    return str(r.get("tournament", "Other") or "Other")

# Confederation Mapping aus FIFA-Ranking, falls vorhanden.
try:
    fifa_rankings
except NameError:
    fifa_path = PROCESSED / "fifa_rankings.parquet"
    fifa_rankings = pl.read_parquet(fifa_path) if fifa_path.exists() else pl.DataFrame()

conf_map = {}
if fifa_rankings.height and "confederation" in fifa_rankings.columns:
    conf_rows = (
        fifa_rankings
        .filter(pl.col("confederation").is_not_null())
        .filter(pl.col("confederation") != "")
        .sort("rank_date")
        .unique(subset=["country_full"], keep="last")
        .select(["country_full", "confederation"])
        .iter_rows(named=True)
    )
    conf_map = {norm_team(r["country_full"]): str(r["confederation"]) for r in conf_rows}

def conf_of(team):
    return conf_map.get(norm_team(team), "UNKNOWN")

# V2 Numeric Features benutzen, wenn Modell-Bundle existiert.
v2_candidates = [
    MODELS / "xgb_goal_models_v2_ensemble.joblib",
    MODELS / "xgb_goal_models_max_features.joblib",
]
v2_bundle = None
for p in v2_candidates:
    if p.exists():
        try:
            v2_bundle = joblib.load(p)
            print("Loaded v2 feature list from:", p)
            break
        except Exception as e:
            print("Could not load:", p, repr(e))

exclude = {"deep_row_id", "date", "home_team", "away_team", "home_score", "away_score", "sample_weight"}

if v2_bundle is not None and "feature_cols" in v2_bundle:
    deep_feature_cols = [c for c in v2_bundle["feature_cols"] if c in deep_df.columns]
else:
    deep_feature_cols = []
    for c in deep_df.columns:
        if c in exclude:
            continue
        try:
            arr = deep_df[c].to_numpy().astype(float)
            if np.isfinite(arr).sum() > 0:
                deep_feature_cols.append(c)
        except Exception:
            pass

print("Deep numeric features:", len(deep_feature_cols))

teams = sorted(set(deep_df["home_team"].to_list()) | set(deep_df["away_team"].to_list()))
teams = sorted(set(norm_team(t) for t in teams))
team_to_id = {"__UNK__": 0, **{t: i + 1 for i, t in enumerate(teams)}}

tournament_values = []
home_conf_values = []
away_conf_values = []

for r in deep_df.iter_rows(named=True):
    tournament_values.append(tournament_bucket_from_row(r))
    home_conf_values.append(conf_of(r["home_team"]))
    away_conf_values.append(conf_of(r["away_team"]))

tournaments = sorted(set(tournament_values))
confs = sorted(set(home_conf_values) | set(away_conf_values) | {"UNKNOWN"})

tournament_to_id = {"__UNK__": 0, **{t: i + 1 for i, t in enumerate(tournaments)}}
conf_to_id = {"UNKNOWN": 0, **{c: i + 1 for i, c in enumerate(confs) if c != "UNKNOWN"}}

home_team_ids = np.array([team_to_id.get(norm_team(x), 0) for x in deep_df["home_team"].to_list()], dtype=np.int64)
away_team_ids = np.array([team_to_id.get(norm_team(x), 0) for x in deep_df["away_team"].to_list()], dtype=np.int64)
tournament_ids = np.array([tournament_to_id.get(x, 0) for x in tournament_values], dtype=np.int64)
home_conf_ids = np.array([conf_to_id.get(x, 0) for x in home_conf_values], dtype=np.int64)
away_conf_ids = np.array([conf_to_id.get(x, 0) for x in away_conf_values], dtype=np.int64)

SEQ_LEN = 16
SEQ_COLS = [
    "gf", "ga", "gd", "pts",
    "is_home", "neutral", "is_world_cup", "is_qualifier",
    "elo_for", "elo_against", "fifa_rank_for", "fifa_rank_against",
    "fifa_points_for", "fifa_points_against",
    "days_rest", "opp_elo_gap",
]

def safe_float(x):
    try:
        y = float(x)
        return y if math.isfinite(y) else np.nan
    except Exception:
        return np.nan

def pad_history(hist):
    arr = np.full((SEQ_LEN, len(SEQ_COLS)), np.nan, dtype=np.float32)
    xs = list(hist)[-SEQ_LEN:]
    if xs:
        arr[-len(xs):, :] = np.array(xs, dtype=np.float32)
    return arr

team_seq_hist = defaultdict(lambda: deque(maxlen=SEQ_LEN))
home_seq_raw = []
away_seq_raw = []

for r in deep_df.iter_rows(named=True):
    d = r["date"]
    hn = norm_team(r["home_team"])
    an = norm_team(r["away_team"])
    hs = int(r["home_score"])
    aw = int(r["away_score"])

    home_seq_raw.append(pad_history(team_seq_hist[hn]))
    away_seq_raw.append(pad_history(team_seq_hist[an]))

    eh = safe_float(r.get("elo_home_pre", np.nan))
    ea = safe_float(r.get("elo_away_pre", np.nan))
    rh = safe_float(r.get("home_fifa_rank", np.nan))
    ra = safe_float(r.get("away_fifa_rank", np.nan))
    ph = safe_float(r.get("home_fifa_points", np.nan))
    pa = safe_float(r.get("away_fifa_points", np.nan))

    team_seq_hist[hn].append([
        hs, aw, hs - aw, result_points(hs, aw),
        1, int(r.get("neutral", 0) or 0), int(r.get("is_world_cup", 0) or 0), int(r.get("is_qualifier", 0) or 0),
        eh, ea, rh, ra, ph, pa,
        safe_float(r.get("home_days_rest", np.nan)),
        eh - ea if math.isfinite(eh) and math.isfinite(ea) else np.nan,
    ])

    team_seq_hist[an].append([
        aw, hs, aw - hs, result_points(aw, hs),
        0, int(r.get("neutral", 0) or 0), int(r.get("is_world_cup", 0) or 0), int(r.get("is_qualifier", 0) or 0),
        ea, eh, ra, rh, pa, ph,
        safe_float(r.get("away_days_rest", np.nan)),
        ea - eh if math.isfinite(eh) and math.isfinite(ea) else np.nan,
    ])

home_seq_raw = np.stack(home_seq_raw)
away_seq_raw = np.stack(away_seq_raw)

train_mask = deep_df["date"] < date(2018, 1, 1)
val_mask = (deep_df["date"] >= date(2018, 1, 1)) & (deep_df["date"] < date(2022, 1, 1))
test_mask = deep_df["date"] >= date(2022, 1, 1)

train_idx = np.where(train_mask.to_numpy())[0]
val_idx = np.where(val_mask.to_numpy())[0]
test_idx = np.where(test_mask.to_numpy())[0]

X_raw = deep_df.select(deep_feature_cols).to_numpy().astype(np.float32)

num_imputer = SimpleImputer(strategy="median")
num_scaler = StandardScaler()

X_train_num = num_scaler.fit_transform(num_imputer.fit_transform(X_raw[train_idx])).astype(np.float32)
X_val_num = num_scaler.transform(num_imputer.transform(X_raw[val_idx])).astype(np.float32)
X_test_num = num_scaler.transform(num_imputer.transform(X_raw[test_idx])).astype(np.float32)

seq_train_flat = np.concatenate([home_seq_raw[train_idx], away_seq_raw[train_idx]], axis=0).reshape(-1, len(SEQ_COLS))
seq_mean = np.nanmean(seq_train_flat, axis=0).astype(np.float32)
seq_std = np.nanstd(seq_train_flat, axis=0).astype(np.float32)
seq_std = np.where(seq_std < 1e-6, 1.0, seq_std).astype(np.float32)

def transform_seq(x):
    z = (x - seq_mean) / seq_std
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

home_seq = transform_seq(home_seq_raw)
away_seq = transform_seq(away_seq_raw)

y_home = deep_df["home_score"].to_numpy().astype(np.float32)
y_away = deep_df["away_score"].to_numpy().astype(np.float32)
y_total = (y_home + y_away).astype(np.float32)
y_diff = (y_home - y_away).astype(np.float32)
y_outcome = np.where(y_home > y_away, 0, np.where(y_home == y_away, 1, 2)).astype(np.int64)
y_draw = (y_home == y_away).astype(np.float32)
sample_weight = deep_df["sample_weight"].to_numpy().astype(np.float32) if "sample_weight" in deep_df.columns else np.ones(len(deep_df), dtype=np.float32)

print("Train/Val/Test:", len(train_idx), len(val_idx), len(test_idx))
print("Teams:", len(team_to_id), "Tournaments:", len(tournament_to_id), "Confs:", len(conf_to_id))


Loaded v2 feature list from: C:\ml\projects\WeltmeisterKI\models\xgb_goal_models_v2_ensemble.joblib
Deep numeric features: 177
Train/Val/Test: 29204 3540 4588
Teams: 331 Tournaments: 6 Confs: 7


## 18. Deep v1 Training

Trainiert das PyTorch-Multi-Task-Modell mit Embeddings, GRU-Sequenzencoder und Heads für Tore, 1X2, Draw, Total Goals und Goal Difference.


In [41]:
# Was diese Zelle macht:
# Trainiert das PyTorch-Multi-Task-Modell mit Embeddings, GRU-Sequenzencoder und Heads für Tore, 1X2, Draw, Total Goals und Goal Difference.

# WeltmeisterKI Deep v1 - PyTorch Training, V4-regularisiert

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

def make_loader(idx, X_num_scaled, batch_size=512, shuffle=False):
    # X_num_scaled ist nur für den Split, daher idx_split lokal mappen.
    return None

def split_tensors(idx, X_num_scaled):
    return TensorDataset(
        torch.tensor(X_num_scaled, dtype=torch.float32),
        torch.tensor(home_team_ids[idx], dtype=torch.long),
        torch.tensor(away_team_ids[idx], dtype=torch.long),
        torch.tensor(tournament_ids[idx], dtype=torch.long),
        torch.tensor(home_conf_ids[idx], dtype=torch.long),
        torch.tensor(away_conf_ids[idx], dtype=torch.long),
        torch.tensor(home_seq[idx], dtype=torch.float32),
        torch.tensor(away_seq[idx], dtype=torch.float32),
        torch.tensor(y_home[idx], dtype=torch.float32),
        torch.tensor(y_away[idx], dtype=torch.float32),
        torch.tensor(y_total[idx], dtype=torch.float32),
        torch.tensor(y_diff[idx], dtype=torch.float32),
        torch.tensor(y_outcome[idx], dtype=torch.long),
        torch.tensor(y_draw[idx], dtype=torch.float32),
        torch.tensor(sample_weight[idx], dtype=torch.float32),
    )

train_ds = split_tensors(train_idx, X_train_num)
val_ds = split_tensors(val_idx, X_val_num)
test_ds = split_tensors(test_idx, X_test_num)

train_loader = DataLoader(train_ds, batch_size=768, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False, num_workers=0)

class FootballDeepV1(nn.Module):
    def __init__(
        self,
        num_dim,
        n_teams,
        n_tournaments,
        n_confs,
        seq_dim,
        team_emb_dim=16,
        tourn_emb_dim=6,
        conf_emb_dim=4,
        seq_hidden=24,
        hidden=160,
        dropout=0.34,
    ):
        super().__init__()
        self.home_team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.away_team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.tournament_emb = nn.Embedding(n_tournaments, tourn_emb_dim)
        self.home_conf_emb = nn.Embedding(n_confs, conf_emb_dim)
        self.away_conf_emb = nn.Embedding(n_confs, conf_emb_dim)

        self.seq_gru = nn.GRU(
            input_size=seq_dim,
            hidden_size=seq_hidden,
            batch_first=True,
            bidirectional=True,
        )

        cat_dim = team_emb_dim * 2 + tourn_emb_dim + conf_emb_dim * 2
        seq_out_dim = seq_hidden * 4
        input_dim = num_dim + cat_dim + seq_out_dim

        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.SiLU(),
        )

        self.goal_head = nn.Linear(hidden // 2, 2)
        self.outcome_head = nn.Linear(hidden // 2, 3)
        self.draw_head = nn.Linear(hidden // 2, 1)
        self.total_head = nn.Linear(hidden // 2, 1)
        self.diff_head = nn.Linear(hidden // 2, 1)

    def encode_seq(self, seq):
        _, h = self.seq_gru(seq)
        return h.transpose(0, 1).reshape(seq.shape[0], -1)

    def forward(self, x_num, home_team, away_team, tournament, home_conf, away_conf, home_seq_x, away_seq_x):
        home_s = self.encode_seq(home_seq_x)
        away_s = self.encode_seq(away_seq_x)

        x = torch.cat([
            x_num,
            self.home_team_emb(home_team),
            self.away_team_emb(away_team),
            self.tournament_emb(tournament),
            self.home_conf_emb(home_conf),
            self.away_conf_emb(away_conf),
            home_s,
            away_s,
        ], dim=1)

        z = self.trunk(x)
        goals = F.softplus(self.goal_head(z)) + 1e-4
        outcome_logits = self.outcome_head(z)
        draw_logit = self.draw_head(z).squeeze(1)
        total = F.softplus(self.total_head(z).squeeze(1))
        diff = self.diff_head(z).squeeze(1)

        return {
            "goals": goals,
            "outcome_logits": outcome_logits,
            "draw_logit": draw_logit,
            "total": total,
            "diff": diff,
        }

model = FootballDeepV1(
    num_dim=X_train_num.shape[1],
    n_teams=len(team_to_id),
    n_tournaments=len(tournament_to_id),
    n_confs=len(conf_to_id),
    seq_dim=len(SEQ_COLS),
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=9e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.55, patience=3)

def batch_to_device(batch):
    return [x.to(device) for x in batch]

def multitask_loss(out, batch):
    (
        x_num, ht, at, tid, hc, ac, hsx, asx,
        yh, ya, yt, yd, yo, ydraw, w
    ) = batch

    pred_goals = out["goals"]

    loss_h = F.poisson_nll_loss(pred_goals[:, 0], yh, log_input=False, reduction="none")
    loss_a = F.poisson_nll_loss(pred_goals[:, 1], ya, log_input=False, reduction="none")
    loss_out = F.cross_entropy(out["outcome_logits"], yo, reduction="none")
    loss_draw = F.binary_cross_entropy_with_logits(out["draw_logit"], ydraw, reduction="none")
    loss_total = F.smooth_l1_loss(out["total"], yt, reduction="none")
    loss_diff = F.smooth_l1_loss(out["diff"], yd, reduction="none")

    loss = (
        loss_h + loss_a
        + 0.35 * loss_out
        + 0.18 * loss_draw
        + 0.08 * loss_total
        + 0.08 * loss_diff
    )

    w = w / torch.clamp(w.mean(), min=1e-6)
    return (loss * w).mean()

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    losses = []
    for batch in loader:
        batch = batch_to_device(batch)
        x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]
        out = model(x_num, ht, at, tid, hc, ac, hsx, asx)
        losses.append(multitask_loss(out, batch).item())
    return float(np.mean(losses))

best_val = float("inf")
best_state = None
patience = 8
bad = 0
EPOCHS = 60

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        batch = batch_to_device(batch)
        x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]

        optimizer.zero_grad(set_to_none=True)
        out = model(x_num, ht, at, tid, hc, ac, hsx, asx)
        loss = multitask_loss(out, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        train_losses.append(loss.item())

    val_loss = eval_epoch(val_loader)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch={epoch:03d} train_loss={np.mean(train_losses):.4f} val_loss={val_loss:.4f}")

    if bad >= patience:
        print("Early stopping at epoch", epoch)
        break

model.load_state_dict(best_state)
print("Best val loss:", best_val)


Device: cuda
epoch=001 train_loss=2.3932 val_loss=2.0592
epoch=005 train_loss=2.0029 val_loss=1.9019
epoch=010 train_loss=1.9474 val_loss=1.9037
epoch=015 train_loss=1.9366 val_loss=1.8950
epoch=020 train_loss=1.9210 val_loss=1.9073
Early stopping at epoch 23
Best val loss: 1.8950234055519104


## 20. Deep v1 Evaluation und Kalibrierung

Kalibriert die Deep-Scorematrix auf dem Validation-Split und bewertet Deep v1 ehrlich auf dem 2022+-Testsplit.


In [42]:
# Was diese Zelle macht:
# Kalibriert die Deep-Scorematrix auf dem Validation-Split und bewertet Deep v1 ehrlich auf dem 2022+-Testsplit.

# WeltmeisterKI Deep v1 - Evaluation + Kalibrierung

from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error

@torch.no_grad()
def predict_loader_outputs(loader):
    model.eval()
    all_goals, all_logits, all_draw, all_total, all_diff = [], [], [], [], []

    for batch in loader:
        batch = batch_to_device(batch)
        x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]
        out = model(x_num, ht, at, tid, hc, ac, hsx, asx)

        all_goals.append(out["goals"].detach().cpu().numpy())
        all_logits.append(out["outcome_logits"].detach().cpu().numpy())
        all_draw.append(torch.sigmoid(out["draw_logit"]).detach().cpu().numpy())
        all_total.append(out["total"].detach().cpu().numpy())
        all_diff.append(out["diff"].detach().cpu().numpy())

    return {
        "goals": np.concatenate(all_goals),
        "logits": np.concatenate(all_logits),
        "draw": np.concatenate(all_draw),
        "total": np.concatenate(all_total),
        "diff": np.concatenate(all_diff),
    }

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def poisson_pmf(k, lam):
    return np.exp(-lam) * (lam ** k) / math.factorial(k)

def raw_poisson_matrix(home_lam, away_lam, max_goals=10, lambda_scale=1.0, draw_boost=1.0):
    hl = max(float(home_lam) * lambda_scale, 0.03)
    al = max(float(away_lam) * lambda_scale, 0.03)

    mat = np.zeros((max_goals + 1, max_goals + 1), dtype=np.float64)
    hp = np.array([poisson_pmf(i, hl) for i in range(max_goals + 1)])
    ap = np.array([poisson_pmf(i, al) for i in range(max_goals + 1)])
    mat = np.outer(hp, ap)

    for i in range(max_goals + 1):
        mat[i, i] *= draw_boost

    mat = mat / mat.sum()
    return mat

def outcome_from_matrix(mat):
    home = float(np.tril(mat, -1).sum())
    draw = float(np.trace(mat))
    away = float(np.triu(mat, 1).sum())
    return np.array([home, draw, away], dtype=np.float64)

def reconcile_score_matrix(mat, target_probs):
    target = np.asarray(target_probs, dtype=np.float64)
    target = np.clip(target, 1e-6, 1.0)
    target = target / target.sum()

    masks = [
        np.tril(np.ones_like(mat), -1).astype(bool),
        np.eye(mat.shape[0], dtype=bool),
        np.triu(np.ones_like(mat), 1).astype(bool),
    ]

    out = mat.copy()
    current = [out[m].sum() for m in masks]

    for i, m in enumerate(masks):
        if current[i] > 1e-12:
            out[m] *= target[i] / current[i]

    out = out / out.sum()
    return out

def blended_probs_from_outputs(goals, logits, draw_prob, calibration):
    clf_probs = softmax_np(logits)
    out = []

    for i in range(len(goals)):
        mat = raw_poisson_matrix(
            goals[i, 0],
            goals[i, 1],
            max_goals=10,
            lambda_scale=calibration["lambda_scale"],
            draw_boost=calibration["draw_boost"],
        )
        pois = outcome_from_matrix(mat)
        d = float(draw_prob[i])
        non_draw = max(1.0 - d, 1e-6)
        side_sum = max(pois[0] + pois[2], 1e-6)

        draw_vec = np.array([
            non_draw * pois[0] / side_sum,
            d,
            non_draw * pois[2] / side_sum,
        ])

        p = (
            calibration["poisson_weight"] * pois
            + calibration["classifier_weight"] * clf_probs[i]
            + calibration["draw_weight"] * draw_vec
        )
        p = np.clip(p, 1e-6, 1.0)
        p = p / p.sum()
        out.append(p)

    return np.vstack(out)

def multiclass_brier(y_true, probs):
    y = np.zeros_like(probs)
    y[np.arange(len(y_true)), y_true] = 1
    return float(np.mean(np.sum((probs - y) ** 2, axis=1)))

val_out = predict_loader_outputs(val_loader)
test_out = predict_loader_outputs(test_loader)

y_val_outcome = y_outcome[val_idx]
y_test_outcome = y_outcome[test_idx]

grid = []
for lambda_scale in [0.90, 1.00, 1.10, 1.20]:
    for draw_boost in [1.00, 1.10, 1.20, 1.30, 1.40]:
        for pw in [0.40, 0.50, 0.60, 0.70]:
            for cw in [0.20, 0.30, 0.40, 0.50]:
                dw = 1.0 - pw - cw
                if dw < 0 or dw > 0.30:
                    continue

                cal = {
                    "lambda_scale": lambda_scale,
                    "draw_boost": draw_boost,
                    "poisson_weight": pw,
                    "classifier_weight": cw,
                    "draw_weight": dw,
                }
                probs = blended_probs_from_outputs(val_out["goals"], val_out["logits"], val_out["draw"], cal)
                ll = log_loss(y_val_outcome, probs, labels=[0, 1, 2])
                br = multiclass_brier(y_val_outcome, probs)
                acc = accuracy_score(y_val_outcome, probs.argmax(axis=1))
                grid.append((ll, br, acc, cal))

grid = sorted(grid, key=lambda x: x[0])
deep_calibration = grid[0][3]
print("Best deep calibration:", deep_calibration)
print("Val log_loss/brier/acc:", grid[0][0], grid[0][1], grid[0][2])

test_probs = blended_probs_from_outputs(test_out["goals"], test_out["logits"], test_out["draw"], deep_calibration)

deep_metrics = {
    "home_mae": mean_absolute_error(y_home[test_idx], test_out["goals"][:, 0]),
    "away_mae": mean_absolute_error(y_away[test_idx], test_out["goals"][:, 1]),
    "total_mae": mean_absolute_error(y_total[test_idx], test_out["total"]),
    "diff_mae": mean_absolute_error(y_diff[test_idx], test_out["diff"]),
    "1x2_accuracy": accuracy_score(y_test_outcome, test_probs.argmax(axis=1)),
    "brier": multiclass_brier(y_test_outcome, test_probs),
    "log_loss": log_loss(y_test_outcome, test_probs, labels=[0, 1, 2]),
}

print("=== Deep v1 Test 2022+ ===")
for k, v in deep_metrics.items():
    print(f"{k}: {v:.6f}")

conf = test_probs.max(axis=1)
correct = (test_probs.argmax(axis=1) == y_test_outcome).astype(float)

cal_rows = []
bins = np.arange(0.3, 1.01, 0.1)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf < hi if hi < 1.0 else conf <= hi)
    if m.sum() == 0:
        continue
    cal_rows.append({
        "bin": f"{lo:.1f}-{hi:.1f}",
        "n": int(m.sum()),
        "avg_conf": round(float(conf[m].mean()), 4),
        "actual_acc": round(float(correct[m].mean()), 4),
        "gap": round(float(conf[m].mean() - correct[m].mean()), 4),
    })

print("=== Deep v1 Calibration ===")
print(pl.DataFrame(cal_rows))


Best deep calibration: {'lambda_scale': 0.9, 'draw_boost': 1.1, 'poisson_weight': 0.5, 'classifier_weight': 0.5, 'draw_weight': 0.0}
Val log_loss/brier/acc: 0.8605184155117909 0.5044948501780344 0.6048022598870056
=== Deep v1 Test 2022+ ===
home_mae: 1.018581
away_mae: 0.833620
total_mae: 1.351022
diff_mae: 1.310898
1x2_accuracy: 0.604621
brier: 0.512302
log_loss: 0.873484
=== Deep v1 Calibration ===
shape: (7, 5)
┌─────────┬──────┬──────────┬────────────┬─────────┐
│ bin     ┆ n    ┆ avg_conf ┆ actual_acc ┆ gap     │
│ ---     ┆ ---  ┆ ---      ┆ ---        ┆ ---     │
│ str     ┆ i64  ┆ f64      ┆ f64        ┆ f64     │
╞═════════╪══════╪══════════╪════════════╪═════════╡
│ 0.3-0.4 ┆ 637  ┆ 0.37     ┆ 0.3878     ┆ -0.0177 │
│ 0.4-0.5 ┆ 1003 ┆ 0.4482   ┆ 0.4766     ┆ -0.0284 │
│ 0.5-0.6 ┆ 782  ┆ 0.5498   ┆ 0.546      ┆ 0.0037  │
│ 0.6-0.7 ┆ 683  ┆ 0.6494   ┆ 0.6076     ┆ 0.0418  │
│ 0.7-0.8 ┆ 583  ┆ 0.749    ┆ 0.7393     ┆ 0.0097  │
│ 0.8-0.9 ┆ 525  ┆ 0.848    ┆ 0.821      ┆ 0.027   │

## 22. Deep v1 speichern

Speichert Deep-Modell, Scaler, Imputer, Mappings, Sequenzzustand, Kalibrierung und Testmetriken als .pt Bundle.


In [43]:
# Was diese Zelle macht:
# Speichert Deep-Modell, Scaler, Imputer, Mappings, Sequenzzustand, Kalibrierung und Testmetriken als .pt Bundle.

# WeltmeisterKI Deep v1 - Save Bundle

deep_model_path = MODELS / "weltmeisterki_deep_v1.pt"

deep_bundle = {
    "model_name": "WeltmeisterKI Deep v1",
    "state_dict": model.state_dict(),
    "model_config": {
        "num_dim": X_train_num.shape[1],
        "n_teams": len(team_to_id),
        "n_tournaments": len(tournament_to_id),
        "n_confs": len(conf_to_id),
        "seq_dim": len(SEQ_COLS),
    },
    "feature_cols": deep_feature_cols,
    "team_to_id": team_to_id,
    "tournament_to_id": tournament_to_id,
    "conf_to_id": conf_to_id,
    "conf_map": conf_map,
    "seq_cols": SEQ_COLS,
    "seq_len": SEQ_LEN,
    "seq_mean": seq_mean,
    "seq_std": seq_std,
    "team_seq_history": {k: list(v) for k, v in team_seq_hist.items()},
    "num_imputer": num_imputer,
    "num_scaler": num_scaler,
    "calibration": deep_calibration,
    "metrics_2022_plus": deep_metrics,
}

torch.save(deep_bundle, deep_model_path)
print("Saved:", deep_model_path)


Saved: C:\ml\projects\WeltmeisterKI\models\weltmeisterki_deep_v1.pt


## 24. Bridge für Deep/Hybrid

Definiert fixture_row(...) als saubere Brücke von der v2-Snapshot-Logik zum Deep-Hybrid-Helper. Verhindert Namenschaos im Kernel.


In [44]:
# Was diese Zelle macht:
# Definiert fixture_row(...) als saubere Brücke von der v2-Snapshot-Logik zum Deep-Hybrid-Helper. Verhindert Namenschaos im Kernel.

# Bridge: fixture_row für Deep/Hybrid
# Diese Zelle gibt dem Deep-Helper einen stabilen Namen für die v2-Snapshot-Funktion.
needed = ["snapshot_features_v2", "state", "norm_team", "fifa_before"]
missing = [x for x in needed if x not in globals()]
if missing:
    raise RuntimeError(f"Diese Sachen fehlen noch: {missing}. Erst v2 Feature Builder laufen lassen.")
def fixture_row(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    return snapshot_features_v2(home, away, match_date, tournament=tournament, country=country, neutral=neutral, state=state)
print("fixture_row ready.")
print("Germany FIFA sanity:", fixture_row("Germany", "Curacao", date(2026, 6, 14)).get("home_fifa_rank"))


fixture_row ready.
Germany FIFA sanity: 10.0


## 26. Deep- und Hybrid-Prediction Helper

Lädt Deep v1 und definiert predict_match_deep, predict_neutral_deep sowie predict_match_hybrid/predict_neutral_hybrid. Die Hybrid-Funktionen akzeptieren max_goals und sind Monte-Carlo-kompatibel.


In [45]:
# Was diese Zelle macht:
# Lädt Deep v1 und definiert predict_match_deep, predict_neutral_deep sowie predict_match_hybrid/predict_neutral_hybrid. Die Hybrid-Funktionen akzeptieren max_goals und sind Monte-Carlo-kompatibel.

# WeltmeisterKI Deep v1 - Prediction Helper
# Läuft nach deiner v2 Prediction Helper Zelle.
# Definiert:
#   predict_match_deep
#   predict_neutral_deep
#   predict_match_hybrid
#   predict_neutral_hybrid
# Optional: USE_DEEP_HYBRID_AS_DEFAULT = True

import numpy as np
import torch
import torch.nn.functional as F

V2_PREDICT_MATCH = globals().get("predict_match")
V2_PREDICT_NEUTRAL = globals().get("predict_neutral")

if "fixture_row" not in globals():
    if "snapshot_features_v2" in globals() and "state" in globals():
        def fixture_row(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
            return snapshot_features_v2(home, away, match_date, tournament=tournament, country=country, neutral=neutral, state=state)
    else:
        raise RuntimeError("Bitte zuerst v2 Feature Builder und v2 Prediction Helper ausführen, damit fixture_row/snapshot_features_v2 existiert.")

deep_bundle = torch.load(MODELS / "weltmeisterki_deep_v1.pt", map_location=device, weights_only=False)

deep_model = FootballDeepV1(**deep_bundle["model_config"]).to(device)
deep_model.load_state_dict(deep_bundle["state_dict"])
deep_model.eval()

deep_feature_cols = deep_bundle["feature_cols"]
team_to_id = deep_bundle["team_to_id"]
tournament_to_id = deep_bundle["tournament_to_id"]
conf_to_id = deep_bundle["conf_to_id"]
conf_map = deep_bundle["conf_map"]
SEQ_LEN = deep_bundle["seq_len"]
SEQ_COLS = deep_bundle["seq_cols"]
seq_mean = deep_bundle["seq_mean"]
seq_std = deep_bundle["seq_std"]
team_seq_history = deep_bundle["team_seq_history"]
num_imputer = deep_bundle["num_imputer"]
num_scaler = deep_bundle["num_scaler"]
deep_calibration = deep_bundle["calibration"]

def deep_conf_of(team):
    return conf_map.get(norm_team(team), "UNKNOWN")

def tournament_to_deep_id(tournament):
    tournament = str(tournament or "Other")
    if tournament in tournament_to_id:
        return tournament_to_id[tournament]
    if "World Cup" in tournament:
        return tournament_to_id.get("FIFA World Cup", 0)
    return tournament_to_id.get("Other", 0)

def deep_transform_one_seq(team):
    hist = team_seq_history.get(norm_team(team), [])
    arr = np.full((SEQ_LEN, len(SEQ_COLS)), np.nan, dtype=np.float32)
    xs = hist[-SEQ_LEN:]
    if xs:
        arr[-len(xs):, :] = np.array(xs, dtype=np.float32)
    z = (arr - seq_mean) / seq_std
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def deep_fixture_tensors(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    row = fixture_row(home, away, match_date, tournament=tournament, country=country, neutral=neutral)

    x_num = np.array([[row.get(c, np.nan) for c in deep_feature_cols]], dtype=np.float32)
    x_num = num_scaler.transform(num_imputer.transform(x_num)).astype(np.float32)

    ht = team_to_id.get(norm_team(home), 0)
    at = team_to_id.get(norm_team(away), 0)
    tid = tournament_to_deep_id(tournament)
    hc = conf_to_id.get(deep_conf_of(home), 0)
    ac = conf_to_id.get(deep_conf_of(away), 0)

    hsx = deep_transform_one_seq(home)[None, :, :]
    asx = deep_transform_one_seq(away)[None, :, :]

    return [
        torch.tensor(x_num, dtype=torch.float32).to(device),
        torch.tensor([ht], dtype=torch.long).to(device),
        torch.tensor([at], dtype=torch.long).to(device),
        torch.tensor([tid], dtype=torch.long).to(device),
        torch.tensor([hc], dtype=torch.long).to(device),
        torch.tensor([ac], dtype=torch.long).to(device),
        torch.tensor(hsx, dtype=torch.float32).to(device),
        torch.tensor(asx, dtype=torch.float32).to(device),
    ]

def calibrated_deep_score_matrix(home_lam, away_lam, clf_probs, draw_prob, max_goals=10):
    mat = raw_poisson_matrix(
        home_lam,
        away_lam,
        max_goals=max_goals,
        lambda_scale=deep_calibration["lambda_scale"],
        draw_boost=deep_calibration["draw_boost"],
    )

    pois = outcome_from_matrix(mat)
    d = float(draw_prob)
    side_sum = max(pois[0] + pois[2], 1e-6)

    draw_vec = np.array([
        (1.0 - d) * pois[0] / side_sum,
        d,
        (1.0 - d) * pois[2] / side_sum,
    ])

    target = (
        deep_calibration["poisson_weight"] * pois
        + deep_calibration["classifier_weight"] * clf_probs
        + deep_calibration["draw_weight"] * draw_vec
    )
    target = np.clip(target, 1e-6, 1.0)
    target = target / target.sum()

    return reconcile_score_matrix(mat, target)

def score_summary_from_grid(mat):
    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())

    pairs = []
    for h in range(mat.shape[0]):
        for a in range(mat.shape[1]):
            pairs.append((h, a, float(mat[h, a])))

    pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
    ml = pairs[0]

    return {
        "p_home_win": p_home,
        "p_draw": p_draw,
        "p_away_win": p_away,
        "most_likely_score": f"{ml[0]}:{ml[1]}",
        "most_likely_score_prob": ml[2],
        "top_scorelines": pairs[:12],
        "score_grid": mat,
    }

DEEP_UNCERTAINTY_SAMPLES = 12

@torch.no_grad()
def deep_raw_predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    if uncertainty_samples is None:
        uncertainty_samples = DEEP_UNCERTAINTY_SAMPLES
    xs = deep_fixture_tensors(home, away, match_date, tournament=tournament, country=country, neutral=neutral)

    mats = []
    home_lams = []
    away_lams = []

    # Dropout + LogNormal-Lambda-Jitter = Modellunsicherheit für Monte Carlo.
    deep_model.train() if uncertainty_samples > 1 else deep_model.eval()

    for _ in range(max(1, uncertainty_samples)):
        out = deep_model(*xs)
        goals = out["goals"].detach().cpu().numpy()[0]
        logits = out["outcome_logits"].detach().cpu().numpy()
        clf = softmax_np(logits)[0]
        draw_p = torch.sigmoid(out["draw_logit"]).detach().cpu().numpy()[0]

        jitter_h = np.random.lognormal(mean=0.0, sigma=0.10) if uncertainty_samples > 1 else 1.0
        jitter_a = np.random.lognormal(mean=0.0, sigma=0.10) if uncertainty_samples > 1 else 1.0

        hl = float(goals[0]) * jitter_h
        al = float(goals[1]) * jitter_a

        mats.append(calibrated_deep_score_matrix(hl, al, clf, draw_p, max_goals=max_goals))
        home_lams.append(hl)
        away_lams.append(al)

    deep_model.eval()

    mat = np.mean(mats, axis=0)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["home_xg_pred"] = float(np.mean(home_lams))
    out["away_xg_pred"] = float(np.mean(away_lams))
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "deep_v1"
    return out

def predict_match_deep(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    return deep_raw_predict_home_away(home, away, match_date, tournament, country, neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

def predict_neutral_deep(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10, uncertainty_samples=None):
    p1 = deep_raw_predict_home_away(team, opponent, match_date, tournament, country, True, max_goals=max_goals, uncertainty_samples=uncertainty_samples)
    p2 = deep_raw_predict_home_away(opponent, team, match_date, tournament, country, True, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = 0.5 * p1["score_grid"] + 0.5 * p2["score_grid"].T
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["team_xg_pred"] = 0.5 * (p1["home_xg_pred"] + p2["away_xg_pred"])
    out["opponent_xg_pred"] = 0.5 * (p1["away_xg_pred"] + p2["home_xg_pred"])
    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["model"] = "deep_v1_neutral"
    return out

def grid_from_prediction(pred):
    if "score_grid" in pred:
        mat = np.asarray(pred["score_grid"], dtype=np.float64)
        return mat / mat.sum()

    # Fallback, falls altes v2 kein score_grid enthält.
    h = pred.get("home_xg_pred", pred.get("team_xg_pred"))
    a = pred.get("away_xg_pred", pred.get("opponent_xg_pred"))
    return raw_poisson_matrix(h, a, max_goals=10, lambda_scale=1.0, draw_boost=1.15)

HYBRID_DEEP_WEIGHT = 0.30

def predict_match_hybrid(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    if V2_PREDICT_MATCH is None:
        raise RuntimeError("V2_PREDICT_MATCH fehlt. Bitte vorher deine v2 Prediction Helper Zelle laufen lassen.")

    if HYBRID_DEEP_WEIGHT <= 0:
        return V2_PREDICT_MATCH(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals)
    if HYBRID_DEEP_WEIGHT >= 1:
        return predict_match_deep(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    v2 = V2_PREDICT_MATCH(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals)
    dp = predict_match_deep(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = (1.0 - HYBRID_DEEP_WEIGHT) * grid_from_prediction(v2) + HYBRID_DEEP_WEIGHT * grid_from_prediction(dp)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    v2_home_xg = v2.get("home_xg_pred", v2.get("home_xg"))
    v2_away_xg = v2.get("away_xg_pred", v2.get("away_xg"))
    out["home_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2_home_xg + HYBRID_DEEP_WEIGHT * dp["home_xg_pred"]
    out["away_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2_away_xg + HYBRID_DEEP_WEIGHT * dp["away_xg_pred"]
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "hybrid_v2_plus_deep_v1"
    return out

def predict_neutral_hybrid(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10, uncertainty_samples=None):
    if V2_PREDICT_NEUTRAL is None:
        raise RuntimeError("V2_PREDICT_NEUTRAL fehlt. Bitte vorher deine v2 Prediction Helper Zelle laufen lassen.")

    if HYBRID_DEEP_WEIGHT <= 0:
        return V2_PREDICT_NEUTRAL(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals)
    if HYBRID_DEEP_WEIGHT >= 1:
        return predict_neutral_deep(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    v2 = V2_PREDICT_NEUTRAL(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals)
    dp = predict_neutral_deep(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = (1.0 - HYBRID_DEEP_WEIGHT) * grid_from_prediction(v2) + HYBRID_DEEP_WEIGHT * grid_from_prediction(dp)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["team_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2["team_xg_pred"] + HYBRID_DEEP_WEIGHT * dp["team_xg_pred"]
    out["opponent_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2["opponent_xg_pred"] + HYBRID_DEEP_WEIGHT * dp["opponent_xg_pred"]
    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["model"] = "hybrid_v2_plus_deep_v1_neutral"
    return out

USE_DEEP_HYBRID_AS_DEFAULT = False

if USE_DEEP_HYBRID_AS_DEFAULT:
    predict_match = predict_match_hybrid
    predict_neutral = predict_neutral_hybrid
    print("Default predictions now use HYBRID v2 + Deep v1.")
else:
    print("Deep ready. Nutze predict_neutral_deep(...) oder predict_neutral_hybrid(...).")


Deep ready. Nutze predict_neutral_deep(...) oder predict_neutral_hybrid(...).


## 28. V4 OOF-Stacking Daten

Trainiert v2- und Deep-Modelle in mehreren Zeit-Folds und sammelt echte Out-of-Fold-Prognosen. Das ist die Grundlage für ein Meta-Modell ohne direkte Leakage.


In [46]:
# Was diese Zelle macht:
# Baut echte Out-of-Fold-Prognosen fuer v2 und Deep. Jedes historische Spiel bekommt Prognosen von Modellen, die dieses Spiel noch nicht gesehen haben.
# Daraus lernt spaeter das Meta-Modell, wann v2, Deep oder ein Mix die besseren Wahrscheinlichkeiten liefert.

import gc
import time
import joblib
import numpy as np
import polars as pl
import torch
from datetime import date
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from xgboost import XGBClassifier
from torch.utils.data import TensorDataset, DataLoader

V4_OOF_V2_ENSEMBLE_N = 3
V4_OOF_DEEP_EPOCHS = 38
V4_OOF_DEEP_PATIENCE = 6
V4_OOF_BATCH_SIZE = 768
V4_META_SEED = 20260610
V4_MAX_GOALS = 10

if "df" not in globals():
    df = features.drop_nulls(["home_score", "away_score"]).sort("date")
if "best" not in globals():
    bundle_tmp = joblib.load(MODELS / "xgb_goal_models_v2_ensemble.joblib")
    best = {"params": bundle_tmp["params"]}

print("V4 OOF config:")
print("- v2 ensemble per fold:", V4_OOF_V2_ENSEMBLE_N)
print("- deep epochs per fold:", V4_OOF_DEEP_EPOCHS)
print("- device:", device)

all_dates = np.array(deep_df["date"].to_list(), dtype=object)

OOF_FOLDS = [
    ("2011-2014", date(2011, 1, 1), date(2015, 1, 1)),
    ("2015-2018", date(2015, 1, 1), date(2019, 1, 1)),
    ("2019-2022", date(2019, 1, 1), date(2023, 1, 1)),
    ("2023-2026", date(2023, 1, 1), date(2027, 1, 1)),
]

def date_indices(start, end):
    return np.array([i for i, d in enumerate(all_dates) if d >= start and d < end], dtype=np.int64)

def before_indices(end):
    return np.array([i for i, d in enumerate(all_dates) if d < end], dtype=np.int64)

def df_between(start, end):
    return df.filter((pl.col("date") >= pl.lit(start)) & (pl.col("date") < pl.lit(end)))

def df_before(end):
    return df.filter(pl.col("date") < pl.lit(end))

def fold_seq_stats(train_ids):
    flat = np.concatenate([home_seq_raw[train_ids], away_seq_raw[train_ids]], axis=0).reshape(-1, len(SEQ_COLS))
    mu = np.nanmean(flat, axis=0).astype(np.float32)
    sd = np.nanstd(flat, axis=0).astype(np.float32)
    sd = np.where(sd < 1e-6, 1.0, sd).astype(np.float32)
    return mu, sd

def transform_seq_fold(x, mu, sd):
    z = (x - mu) / sd
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def make_deep_fold_dataset(ids, x_num, hseq, aseq):
    return TensorDataset(
        torch.tensor(x_num, dtype=torch.float32),
        torch.tensor(home_team_ids[ids], dtype=torch.long),
        torch.tensor(away_team_ids[ids], dtype=torch.long),
        torch.tensor(tournament_ids[ids], dtype=torch.long),
        torch.tensor(home_conf_ids[ids], dtype=torch.long),
        torch.tensor(away_conf_ids[ids], dtype=torch.long),
        torch.tensor(hseq, dtype=torch.float32),
        torch.tensor(aseq, dtype=torch.float32),
        torch.tensor(y_home[ids], dtype=torch.float32),
        torch.tensor(y_away[ids], dtype=torch.float32),
        torch.tensor(y_total[ids], dtype=torch.float32),
        torch.tensor(y_diff[ids], dtype=torch.float32),
        torch.tensor(y_outcome[ids], dtype=torch.long),
        torch.tensor(y_draw[ids], dtype=torch.float32),
        torch.tensor(sample_weight[ids], dtype=torch.float32),
    )

def fit_deep_fold(train_ids, val_ids, seed):
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)

    imp_fold = SimpleImputer(strategy="median")
    scaler_fold = StandardScaler()
    x_train = scaler_fold.fit_transform(imp_fold.fit_transform(X_raw[train_ids])).astype(np.float32)
    x_val = scaler_fold.transform(imp_fold.transform(X_raw[val_ids])).astype(np.float32)

    seq_mu, seq_sd = fold_seq_stats(train_ids)
    h_train = transform_seq_fold(home_seq_raw[train_ids], seq_mu, seq_sd)
    a_train = transform_seq_fold(away_seq_raw[train_ids], seq_mu, seq_sd)
    h_val = transform_seq_fold(home_seq_raw[val_ids], seq_mu, seq_sd)
    a_val = transform_seq_fold(away_seq_raw[val_ids], seq_mu, seq_sd)

    tr_loader = DataLoader(make_deep_fold_dataset(train_ids, x_train, h_train, a_train), batch_size=V4_OOF_BATCH_SIZE, shuffle=True, num_workers=0)
    va_loader = DataLoader(make_deep_fold_dataset(val_ids, x_val, h_val, a_val), batch_size=1536, shuffle=False, num_workers=0)

    m = FootballDeepV1(
        num_dim=x_train.shape[1],
        n_teams=len(team_to_id),
        n_tournaments=len(tournament_to_id),
        n_confs=len(conf_to_id),
        seq_dim=len(SEQ_COLS),
        team_emb_dim=16,
        tourn_emb_dim=6,
        conf_emb_dim=4,
        seq_hidden=24,
        hidden=160,
        dropout=0.36,
    ).to(device)

    opt = torch.optim.AdamW(m.parameters(), lr=5e-4, weight_decay=1.1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.55, patience=2)

    best_loss = float("inf")
    best_state = None
    bad = 0

    for epoch in range(1, V4_OOF_DEEP_EPOCHS + 1):
        m.train()
        for batch in tr_loader:
            batch = batch_to_device(batch)
            x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]
            opt.zero_grad(set_to_none=True)
            out = m(x_num, ht, at, tid, hc, ac, hsx, asx)
            loss = multitask_loss(out, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.5)
            opt.step()

        m.eval()
        losses = []
        with torch.no_grad():
            for batch in va_loader:
                batch = batch_to_device(batch)
                x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]
                out = m(x_num, ht, at, tid, hc, ac, hsx, asx)
                losses.append(multitask_loss(out, batch).item())
        val_loss = float(np.mean(losses))
        sched.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if bad >= V4_OOF_DEEP_PATIENCE:
            break

    m.load_state_dict(best_state)
    return {"model": m, "imputer": imp_fold, "scaler": scaler_fold, "seq_mean": seq_mu, "seq_std": seq_sd, "best_val_loss": best_loss}

@torch.no_grad()
def predict_deep_fold(fold_bundle, ids):
    m = fold_bundle["model"]
    m.eval()
    x = fold_bundle["scaler"].transform(fold_bundle["imputer"].transform(X_raw[ids])).astype(np.float32)
    h = transform_seq_fold(home_seq_raw[ids], fold_bundle["seq_mean"], fold_bundle["seq_std"])
    a = transform_seq_fold(away_seq_raw[ids], fold_bundle["seq_mean"], fold_bundle["seq_std"])
    loader = DataLoader(make_deep_fold_dataset(ids, x, h, a), batch_size=1536, shuffle=False, num_workers=0)
    outs = []
    for batch in loader:
        batch = batch_to_device(batch)
        x_num, ht, at, tid, hc, ac, hsx, asx = batch[:8]
        o = m(x_num, ht, at, tid, hc, ac, hsx, asx)
        outs.append({
            "goals": o["goals"].detach().cpu().numpy(),
            "logits": o["outcome_logits"].detach().cpu().numpy(),
            "draw": torch.sigmoid(o["draw_logit"]).detach().cpu().numpy(),
        })
    return {
        "goals": np.concatenate([x["goals"] for x in outs]),
        "logits": np.concatenate([x["logits"] for x in outs]),
        "draw": np.concatenate([x["draw"] for x in outs]),
    }

def tune_deep_calibration_outputs(y_true, outputs):
    best_local = None
    for lambda_scale in [1.0, 1.1, 1.2, 1.3]:
        for draw_boost in [1.0, 1.15, 1.3, 1.45]:
            for pw in [0.30, 0.40, 0.50, 0.60]:
                for cw in [0.25, 0.35, 0.45, 0.55]:
                    dw = 1.0 - pw - cw
                    if dw < 0 or dw > 0.30:
                        continue
                    cal = {"lambda_scale": lambda_scale, "draw_boost": draw_boost, "poisson_weight": pw, "classifier_weight": cw, "draw_weight": dw}
                    probs = blended_probs_from_outputs(outputs["goals"], outputs["logits"], outputs["draw"], cal)
                    ll = log_loss(y_true, probs, labels=[0, 1, 2])
                    if best_local is None or ll < best_local["log_loss"]:
                        best_local = {**cal, "log_loss": float(ll)}
    return best_local

def deep_grids_from_outputs(outputs, cal, max_goals=10):
    clf = softmax_np(outputs["logits"])
    grids = []
    for i in range(len(clf)):
        grids.append(calibrated_deep_score_matrix(outputs["goals"][i, 0], outputs["goals"][i, 1], clf[i], outputs["draw"][i], max_goals=max_goals))
    return np.stack(grids)

def v2_grids_from_parts(home_pred, away_pred, clf_probs, max_goals=10):
    grids = []
    for i in range(len(home_pred)):
        g, _ = calibrated_score_grid(home_pred[i], away_pred[i], clf_probs=clf_probs[i], max_goals=max_goals)
        grids.append(g)
    return np.stack(grids)

def entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1.0)
    return -np.sum(p * np.log(p), axis=1)

META_CONTEXT_COLS = [
    "neutral", "is_world_cup", "is_qualifier", "is_friendly", "is_major_tournament",
    "elo_diff", "elo_ratio", "fifa_rank_diff", "fifa_points_diff", "same_confed",
    "home_games_pre", "away_games_pre", "rest_diff", "attack_diff", "def_weak_diff",
    "home_attack_vs_away_def", "away_attack_vs_home_def",
    "form_gd_diff_l10", "form_gd_diff_l20", "adj_form_gd_diff_l10", "adj_form_gd_diff_l20",
]
META_CONTEXT_COLS = [c for c in META_CONTEXT_COLS if c in df.columns]

def build_meta_matrix(data_df, v2_home, v2_away, v2_probs, deep_goals, deep_probs):
    base_parts = [
        v2_probs,
        deep_probs,
        deep_probs - v2_probs,
        np.abs(deep_probs - v2_probs),
        np.column_stack([
            v2_home, v2_away, v2_home - v2_away, v2_home + v2_away,
            deep_goals[:, 0], deep_goals[:, 1], deep_goals[:, 0] - deep_goals[:, 1], deep_goals[:, 0] + deep_goals[:, 1],
            np.max(v2_probs, axis=1), np.max(deep_probs, axis=1),
            entropy(v2_probs), entropy(deep_probs),
        ])
    ]
    names = [
        "v2_p_home", "v2_p_draw", "v2_p_away",
        "deep_p_home", "deep_p_draw", "deep_p_away",
        "diff_p_home", "diff_p_draw", "diff_p_away",
        "absdiff_p_home", "absdiff_p_draw", "absdiff_p_away",
        "v2_home_xg", "v2_away_xg", "v2_xg_diff", "v2_xg_total",
        "deep_home_xg", "deep_away_xg", "deep_xg_diff", "deep_xg_total",
        "v2_conf", "deep_conf", "v2_entropy", "deep_entropy",
    ]
    ctx = data_df.select(META_CONTEXT_COLS).to_numpy().astype(float) if META_CONTEXT_COLS else np.empty((data_df.height, 0))
    if ctx.shape[1]:
        base_parts.append(ctx)
        names += META_CONTEXT_COLS
    return np.concatenate(base_parts, axis=1).astype(np.float32), names

def metrics_from_probs(name, y, probs, home_true, away_true, home_pred, away_pred):
    return {
        "model": name,
        "n": len(y),
        "home_mae": float(mean_absolute_error(home_true, home_pred)),
        "away_mae": float(mean_absolute_error(away_true, away_pred)),
        "1x2_accuracy": float(accuracy_score(y, probs.argmax(axis=1))),
        "brier": multiclass_brier(y, probs),
        "log_loss": float(log_loss(y, probs, labels=[0, 1, 2])),
    }

fold_records = []
oof_X_parts, oof_y_parts, oof_dates = [], [], []
oof_v2_probs, oof_deep_probs = [], []
oof_v2_home, oof_v2_away, oof_deep_goals = [], [], []
oof_v2_grids, oof_deep_grids = [], []
oof_home_true, oof_away_true = [], []
meta_feature_names = None

start_all = time.time()
for fold_i, (fold_name, test_start, test_end) in enumerate(OOF_FOLDS, start=1):
    val_start = date(test_start.year - 4, 1, 1)
    train_df_fold = df_before(val_start)
    val_df_fold = df_between(val_start, test_start)
    hold_df_fold = df_between(test_start, test_end)
    train_ids = before_indices(val_start)
    val_ids = date_indices(val_start, test_start)
    hold_ids = date_indices(test_start, test_end)

    print(f"\n=== OOF Fold {fold_i}/{len(OOF_FOLDS)}: {fold_name} ===")
    print("train/val/hold:", train_df_fold.height, val_df_fold.height, hold_df_fold.height)

    imp_f, hms_f, ams_f, cms_f = fit_ensemble(train_df_fold, feature_cols, best["params"], V4_OOF_V2_ENSEMBLE_N, seed0=7100 + fold_i * 100)
    vh, va, vclf = predict_ensemble(imp_f, hms_f, ams_f, cms_f, val_df_fold, feature_cols)
    yv = y_outcome_from_scores(val_df_fold["home_score"].to_numpy(), val_df_fold["away_score"].to_numpy())
    cal_v2_f = tune_calibration(yv, vh, va, vclf)

    hh, ha, hclf = predict_ensemble(imp_f, hms_f, ams_f, cms_f, hold_df_fold, feature_cols)
    yh = y_outcome_from_scores(hold_df_fold["home_score"].to_numpy(), hold_df_fold["away_score"].to_numpy())
    v2p = final_probs_from_parts(hh, ha, hclf, cal_v2_f)
    v2g = v2_grids_from_parts(hh, ha, hclf, max_goals=V4_MAX_GOALS)

    deep_f = fit_deep_fold(train_ids, val_ids, seed=8100 + fold_i * 100)
    deep_val = predict_deep_fold(deep_f, val_ids)
    cal_deep_f = tune_deep_calibration_outputs(yv, deep_val)
    deep_hold = predict_deep_fold(deep_f, hold_ids)
    deepp = blended_probs_from_outputs(deep_hold["goals"], deep_hold["logits"], deep_hold["draw"], cal_deep_f)
    deepg = deep_grids_from_outputs(deep_hold, cal_deep_f, max_goals=V4_MAX_GOALS)

    X_meta, names = build_meta_matrix(hold_df_fold, hh, ha, v2p, deep_hold["goals"], deepp)
    if meta_feature_names is None:
        meta_feature_names = names

    fold_records.append({
        "fold": fold_name,
        "n": len(yh),
        "v2_log_loss": float(log_loss(yh, v2p, labels=[0, 1, 2])),
        "deep_log_loss": float(log_loss(yh, deepp, labels=[0, 1, 2])),
        "v2_acc": float(accuracy_score(yh, v2p.argmax(axis=1))),
        "deep_acc": float(accuracy_score(yh, deepp.argmax(axis=1))),
        "deep_best_val_loss": deep_f["best_val_loss"],
        "v2_lambda_scale": cal_v2_f["lambda_scale"],
        "v2_draw_boost": cal_v2_f["draw_boost"],
        "deep_lambda_scale": cal_deep_f["lambda_scale"],
        "deep_draw_boost": cal_deep_f["draw_boost"],
    })

    oof_X_parts.append(X_meta)
    oof_y_parts.append(yh)
    oof_dates += hold_df_fold["date"].to_list()
    oof_v2_probs.append(v2p)
    oof_deep_probs.append(deepp)
    oof_v2_home.append(hh)
    oof_v2_away.append(ha)
    oof_deep_goals.append(deep_hold["goals"])
    oof_v2_grids.append(v2g)
    oof_deep_grids.append(deepg)
    oof_home_true.append(hold_df_fold["home_score"].to_numpy())
    oof_away_true.append(hold_df_fold["away_score"].to_numpy())

    del imp_f, hms_f, ams_f, cms_f, deep_f
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

v4_oof_X = np.vstack(oof_X_parts)
v4_oof_y = np.concatenate(oof_y_parts)
v4_oof_dates = np.array(oof_dates, dtype=object)
v4_oof_v2_probs = np.vstack(oof_v2_probs)
v4_oof_deep_probs = np.vstack(oof_deep_probs)
v4_oof_v2_home = np.concatenate(oof_v2_home)
v4_oof_v2_away = np.concatenate(oof_v2_away)
v4_oof_deep_goals = np.vstack(oof_deep_goals)
v4_oof_v2_grids = np.concatenate(oof_v2_grids, axis=0)
v4_oof_deep_grids = np.concatenate(oof_deep_grids, axis=0)
v4_oof_home_true = np.concatenate(oof_home_true)
v4_oof_away_true = np.concatenate(oof_away_true)

v4_fold_table = pl.DataFrame(fold_records)
print("\n=== V4 OOF Fold Summary ===")
print(v4_fold_table)
print("OOF rows:", len(v4_oof_y), "elapsed_s:", round(time.time() - start_all, 1))


V4 OOF config:
- v2 ensemble per fold: 3
- deep epochs per fold: 38
- device: cuda

=== OOF Fold 1/4: 2011-2014 ===
train/val/hold: 18481 3877 3963

=== OOF Fold 2/4: 2015-2018 ===
train/val/hold: 22358 3963 3812

=== OOF Fold 3/4: 2019-2022 ===
train/val/hold: 26321 3812 3581

=== OOF Fold 4/4: 2023-2026 ===
train/val/hold: 30133 3581 3618

=== V4 OOF Fold Summary ===
shape: (4, 11)
┌───────────┬──────┬─────────────┬───────────────┬──────────┬──────────┬────────────────────┬─────────────────┬───────────────┬───────────────────┬─────────────────┐
│ fold      ┆ n    ┆ v2_log_loss ┆ deep_log_loss ┆ v2_acc   ┆ deep_acc ┆ deep_best_val_loss ┆ v2_lambda_scale ┆ v2_draw_boost ┆ deep_lambda_scale ┆ deep_draw_boost │
│ ---       ┆ ---  ┆ ---         ┆ ---           ┆ ---      ┆ ---      ┆ ---                ┆ ---             ┆ ---           ┆ ---               ┆ ---             │
│ str       ┆ i64  ┆ f64         ┆ f64           ┆ f64      ┆ f64      ┆ f64                ┆ f64             ┆ f64

## 30. V4 Meta-Modell und Performance-Vergleich

Trainiert ein dynamisches Meta-Modell auf OOF-Prognosen und vergleicht v2, Deep, fixen Hybrid und V4 auf 2022+.


In [47]:
# Was diese Zelle macht:
# Trainiert das dynamische Meta-Modell auf OOF-Prognosen und vergleicht v2, Deep, fixen Hybrid und V4 ehrlich auf 2022+.
# Das Meta-Modell entscheidet pro Match anhand von Kontext und Modell-Signalen, welche Quelle wie stark zaehlt.

from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from xgboost import XGBClassifier
import joblib

meta_train_mask = np.array([d < date(2022, 1, 1) for d in v4_oof_dates])
meta_eval_mask = np.array([d >= date(2022, 1, 1) for d in v4_oof_dates])

meta_imputer = SimpleImputer(strategy="median")
X_meta_train = meta_imputer.fit_transform(v4_oof_X[meta_train_mask])
X_meta_eval = meta_imputer.transform(v4_oof_X[meta_eval_mask])
y_meta_train = v4_oof_y[meta_train_mask]
y_meta_eval = v4_oof_y[meta_eval_mask]

v4_meta_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=420,
    max_depth=2,
    learning_rate=0.025,
    subsample=0.82,
    colsample_bytree=0.82,
    reg_lambda=8.0,
    reg_alpha=1.2,
    min_child_weight=8,
    tree_method="hist",
    n_jobs=-1,
    random_state=V4_META_SEED,
)
v4_meta_model.fit(X_meta_train, y_meta_train, verbose=False)

v4_meta_probs_eval = v4_meta_model.predict_proba(X_meta_eval)
v4_meta_probs_eval = np.clip(v4_meta_probs_eval, 1e-6, 1.0)
v4_meta_probs_eval = v4_meta_probs_eval / v4_meta_probs_eval.sum(axis=1, keepdims=True)

# Best fixed hybrid auf dem Meta-Train-OOF lernen, dann auf 2022+ vergleichen.
weights = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 1.0]
best_fixed = None
for w in weights:
    p_train = (1.0 - w) * v4_oof_v2_probs[meta_train_mask] + w * v4_oof_deep_probs[meta_train_mask]
    p_train = np.clip(p_train, 1e-6, 1.0)
    p_train = p_train / p_train.sum(axis=1, keepdims=True)
    ll = log_loss(y_meta_train, p_train, labels=[0, 1, 2])
    if best_fixed is None or ll < best_fixed["train_log_loss"]:
        best_fixed = {"deep_weight": float(w), "train_log_loss": float(ll)}

v4_best_fixed_deep_weight = best_fixed["deep_weight"]
fixed_probs_eval = (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_probs[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_probs[meta_eval_mask]
fixed_probs_eval = np.clip(fixed_probs_eval, 1e-6, 1.0)
fixed_probs_eval = fixed_probs_eval / fixed_probs_eval.sum(axis=1, keepdims=True)

home_true_eval = v4_oof_home_true[meta_eval_mask]
away_true_eval = v4_oof_away_true[meta_eval_mask]

def eval_oof_model(label, probs, home_pred, away_pred):
    return metrics_from_probs(label, y_meta_eval, probs, home_true_eval, away_true_eval, home_pred, away_pred)

v4_compare_rows = []
v4_compare_rows.append(eval_oof_model("v2_oof", v4_oof_v2_probs[meta_eval_mask], v4_oof_v2_home[meta_eval_mask], v4_oof_v2_away[meta_eval_mask]))
v4_compare_rows.append(eval_oof_model("deep_oof", v4_oof_deep_probs[meta_eval_mask], v4_oof_deep_goals[meta_eval_mask, 0], v4_oof_deep_goals[meta_eval_mask, 1]))
v4_compare_rows.append(eval_oof_model(
    f"fixed_hybrid_{int(v4_best_fixed_deep_weight * 100)}pct_deep_oof",
    fixed_probs_eval,
    (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_home[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_goals[meta_eval_mask, 0],
    (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_away[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_goals[meta_eval_mask, 1],
))
v4_compare_rows.append(eval_oof_model(
    "v4_oof_meta_model",
    v4_meta_probs_eval,
    (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_home[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_goals[meta_eval_mask, 0],
    (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_away[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_goals[meta_eval_mask, 1],
))

v4_performance_table = pl.DataFrame(v4_compare_rows).sort("log_loss")
print("=== V4 Performance Vergleich, OOF 2022+ ===")
print(v4_performance_table)
print("Best fixed deep weight from OOF train:", v4_best_fixed_deep_weight)

# Meta-Modell final auf allen OOF-Zeilen trainieren fuer Future Prediction.
v4_meta_imputer_final = SimpleImputer(strategy="median")
X_meta_all = v4_meta_imputer_final.fit_transform(v4_oof_X)
v4_meta_model_final = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=420,
    max_depth=2,
    learning_rate=0.025,
    subsample=0.82,
    colsample_bytree=0.82,
    reg_lambda=8.0,
    reg_alpha=1.2,
    min_child_weight=8,
    tree_method="hist",
    n_jobs=-1,
    random_state=V4_META_SEED + 1,
)
v4_meta_model_final.fit(X_meta_all, v4_oof_y, verbose=False)

try:
    imp_gain = v4_meta_model_final.get_booster().get_score(importance_type="gain")
    importance_rows = []
    for k, v in imp_gain.items():
        idx = int(k[1:]) if k.startswith("f") else None
        name = meta_feature_names[idx] if idx is not None and idx < len(meta_feature_names) else k
        importance_rows.append({"feature": name, "gain": float(v)})
    v4_meta_importance = pl.DataFrame(importance_rows).sort("gain", descending=True)
    print("\n=== V4 Meta Feature Importance Top 30 ===")
    print(v4_meta_importance.head(30))
except Exception as e:
    print("Meta importance skipped:", repr(e))


=== V4 Performance Vergleich, OOF 2022+ ===
shape: (4, 7)
┌─────────────────────────────┬──────┬──────────┬──────────┬──────────────┬──────────┬──────────┐
│ model                       ┆ n    ┆ home_mae ┆ away_mae ┆ 1x2_accuracy ┆ brier    ┆ log_loss │
│ ---                         ┆ ---  ┆ ---      ┆ ---      ┆ ---          ┆ ---      ┆ ---      │
│ str                         ┆ i64  ┆ f64      ┆ f64      ┆ f64          ┆ f64      ┆ f64      │
╞═════════════════════════════╪══════╪══════════╪══════════╪══════════════╪══════════╪══════════╡
│ fixed_hybrid_40pct_deep_oof ┆ 4588 ┆ 1.00884  ┆ 0.826993 ┆ 0.602005     ┆ 0.508935 ┆ 0.865739 │
│ v4_oof_meta_model           ┆ 4588 ┆ 1.00884  ┆ 0.826993 ┆ 0.602005     ┆ 0.508977 ┆ 0.866024 │
│ v2_oof                      ┆ 4588 ┆ 1.01655  ┆ 0.827621 ┆ 0.599608     ┆ 0.511439 ┆ 0.8694   │
│ deep_oof                    ┆ 4588 ┆ 1.010186 ┆ 0.83458  ┆ 0.604621     ┆ 0.510947 ┆ 0.870414 │
└─────────────────────────────┴──────┴──────────┴──────────┴

## 32. V4 Scoreline-Kalibrierung

Kalibriert die exakte Score-Matrix mit Low-Score- und Dixon-Coles-artigen Korrekturen, ohne die 1X2-Meta-Wahrscheinlichkeiten zu verlieren.


In [48]:
# Was diese Zelle macht:
# Kalibriert die exakten Scorelines: mehr Kontrolle fuer 0:0, 1:0, 0:1, 1:1 und Low-Score-Spiele.
# Die 1X2-Wahrscheinlichkeiten kommen vom Meta-Modell; die Score-Matrix wird danach passend darauf reconciled.

def apply_scoreline_adjustments(mat, cal):
    out = np.asarray(mat, dtype=np.float64).copy()
    max_g = out.shape[0] - 1
    low_boost = cal.get("low_score_boost", 1.0)
    if low_boost != 1.0:
        for h in range(max_g + 1):
            for a in range(max_g + 1):
                if h + a <= 2:
                    out[h, a] *= low_boost
    out[0, 0] *= cal.get("dc_00", 1.0)
    if max_g >= 1:
        out[1, 0] *= cal.get("dc_10", 1.0)
        out[0, 1] *= cal.get("dc_01", 1.0)
        out[1, 1] *= cal.get("dc_11", 1.0)
    out = np.clip(out, 1e-12, None)
    out /= out.sum()
    return out

def score_nll_for_grids(grids, home_scores, away_scores):
    losses = []
    max_g = grids.shape[1] - 1
    for g, hs, aw in zip(grids, home_scores, away_scores):
        h = int(min(max_g, hs))
        a = int(min(max_g, aw))
        losses.append(-np.log(max(float(g[h, a]), 1e-12)))
    return float(np.mean(losses))

def calibrated_meta_score_grids(base_grids, target_probs, cal):
    out = []
    for g, p in zip(base_grids, target_probs):
        gg = apply_scoreline_adjustments(g, cal)
        gg = reconcile_score_matrix(gg, p)
        out.append(gg)
    return np.stack(out)

base_train_grids = (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_grids[meta_train_mask] + v4_best_fixed_deep_weight * v4_oof_deep_grids[meta_train_mask]
base_eval_grids = (1.0 - v4_best_fixed_deep_weight) * v4_oof_v2_grids[meta_eval_mask] + v4_best_fixed_deep_weight * v4_oof_deep_grids[meta_eval_mask]

meta_probs_train = v4_meta_model.predict_proba(X_meta_train)
meta_probs_train = np.clip(meta_probs_train, 1e-6, 1.0)
meta_probs_train = meta_probs_train / meta_probs_train.sum(axis=1, keepdims=True)

home_train_scores = v4_oof_home_true[meta_train_mask]
away_train_scores = v4_oof_away_true[meta_train_mask]

score_grid = []
for low_score_boost in [0.95, 1.0, 1.05, 1.10, 1.15]:
    for dc_00 in [0.95, 1.0, 1.05, 1.10, 1.20]:
        for dc_11 in [0.95, 1.0, 1.05, 1.10, 1.20]:
            for one_goal in [0.95, 1.0, 1.05, 1.10]:
                cal = {"low_score_boost": low_score_boost, "dc_00": dc_00, "dc_10": one_goal, "dc_01": one_goal, "dc_11": dc_11}
                grids = calibrated_meta_score_grids(base_train_grids, meta_probs_train, cal)
                nll = score_nll_for_grids(grids, home_train_scores, away_train_scores)
                score_grid.append({**cal, "score_nll_train": nll})

v4_scoreline_tuning = pl.DataFrame(score_grid).sort("score_nll_train")
v4_scoreline_calibration = v4_scoreline_tuning.row(0, named=True)
v4_scoreline_calibration = {k: float(v) for k, v in v4_scoreline_calibration.items() if k != "score_nll_train"}

v4_meta_grids_eval_uncal = calibrated_meta_score_grids(base_eval_grids, v4_meta_probs_eval, {"low_score_boost": 1.0, "dc_00": 1.0, "dc_10": 1.0, "dc_01": 1.0, "dc_11": 1.0})
v4_meta_grids_eval_cal = calibrated_meta_score_grids(base_eval_grids, v4_meta_probs_eval, v4_scoreline_calibration)

scoreline_rows = [
    {"model": "v4_meta_score_uncalibrated", "score_nll": score_nll_for_grids(v4_meta_grids_eval_uncal, home_true_eval, away_true_eval)},
    {"model": "v4_meta_score_calibrated", "score_nll": score_nll_for_grids(v4_meta_grids_eval_cal, home_true_eval, away_true_eval)},
]
v4_scoreline_performance = pl.DataFrame(scoreline_rows).sort("score_nll")

print("=== V4 Scoreline Calibration Best Params ===")
print(v4_scoreline_calibration)
print("\n=== V4 Scoreline NLL 2022+ ===")
print(v4_scoreline_performance)
print("\nTop 10 Scoreline Params auf OOF-Train:")
print(v4_scoreline_tuning.head(10))


=== V4 Scoreline Calibration Best Params ===
{'low_score_boost': 1.0, 'dc_00': 1.2, 'dc_10': 1.1, 'dc_01': 1.1, 'dc_11': 1.05}

=== V4 Scoreline NLL 2022+ ===
shape: (2, 2)
┌────────────────────────────┬───────────┐
│ model                      ┆ score_nll │
│ ---                        ┆ ---       │
│ str                        ┆ f64       │
╞════════════════════════════╪═══════════╡
│ v4_meta_score_uncalibrated ┆ 2.814944  │
│ v4_meta_score_calibrated   ┆ 2.815027  │
└────────────────────────────┴───────────┘

Top 10 Scoreline Params auf OOF-Train:
shape: (10, 6)
┌─────────────────┬───────┬───────┬───────┬───────┬─────────────────┐
│ low_score_boost ┆ dc_00 ┆ dc_10 ┆ dc_01 ┆ dc_11 ┆ score_nll_train │
│ ---             ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---             │
│ f64             ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ f64             │
╞═════════════════╪═══════╪═══════╪═══════╪═══════╪═════════════════╡
│ 1.0             ┆ 1.2   ┆ 1.1   ┆ 1.1   ┆ 1.05  ┆ 2.7996          │
│ 1.0   

## 34. V4 Final Helper

Speichert das V4-Bundle und setzt die globalen Prediction Helper auf das finale OOF-Meta-Stacking-Modell.


In [49]:
# Was diese Zelle macht:
# Speichert das V4-SOTA-Bundle und setzt predict_match/predict_neutral auf das finale Meta-Stacking-Modell.
# Ab hier nutzen alle Forecast- und Monte-Carlo-Zellen V4 statt v2 oder fixem Hybrid.

import joblib

v4_bundle_path = MODELS / "weltmeisterki4_sota_meta.joblib"
joblib.dump({
    "model_version": "weltmeisterki4_sota_oof_meta_deep_v2_scoreline_calibrated",
    "meta_model": v4_meta_model_final,
    "meta_imputer": v4_meta_imputer_final,
    "meta_feature_names": meta_feature_names,
    "meta_context_cols": META_CONTEXT_COLS,
    "best_fixed_deep_weight": v4_best_fixed_deep_weight,
    "scoreline_calibration": v4_scoreline_calibration,
    "performance_table": v4_performance_table,
    "scoreline_performance": v4_scoreline_performance,
    "fold_table": v4_fold_table,
}, v4_bundle_path)
print("Saved V4 bundle:", v4_bundle_path)

v4_bundle = joblib.load(v4_bundle_path)
v4_meta_model_final = v4_bundle["meta_model"]
v4_meta_imputer_final = v4_bundle["meta_imputer"]
meta_feature_names = v4_bundle["meta_feature_names"]
META_CONTEXT_COLS = v4_bundle["meta_context_cols"]
v4_best_fixed_deep_weight = float(v4_bundle["best_fixed_deep_weight"])
v4_scoreline_calibration = v4_bundle["scoreline_calibration"]

V4_DEEP_UNCERTAINTY_SAMPLES = 8

def _as_grid(pred):
    g = np.asarray(pred["score_grid"], dtype=np.float64).copy()
    g = np.clip(g, 1e-12, None)
    return g / g.sum()

def _prediction_probs_home_order(pred):
    return np.array([pred["p_home_win"], pred["p_draw"], pred["p_away_win"]], dtype=np.float64)

def _meta_features_for_single(row, v2_pred, deep_pred):
    v2_probs = _prediction_probs_home_order(v2_pred)[None, :]
    deep_probs = _prediction_probs_home_order(deep_pred)[None, :]
    v2_home = np.array([v2_pred.get("home_xg_pred", v2_pred.get("home_xg"))], dtype=np.float64)
    v2_away = np.array([v2_pred.get("away_xg_pred", v2_pred.get("away_xg"))], dtype=np.float64)
    deep_goals = np.array([[deep_pred["home_xg_pred"], deep_pred["away_xg_pred"]]], dtype=np.float64)
    ctx_vals = []
    for c in META_CONTEXT_COLS:
        try:
            ctx_vals.append(float(row.get(c, np.nan)))
        except Exception:
            ctx_vals.append(np.nan)
    data_df_dummy = pl.DataFrame([{c: ctx_vals[i] for i, c in enumerate(META_CONTEXT_COLS)}]) if META_CONTEXT_COLS else pl.DataFrame({})
    X, _ = build_meta_matrix(data_df_dummy, v2_home, v2_away, v2_probs, deep_goals, deep_probs)
    return X

def _summary_from_final_grid(mat):
    mat = np.asarray(mat, dtype=np.float64)
    mat = mat / mat.sum()
    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())
    pairs = []
    for h in range(mat.shape[0]):
        for a in range(mat.shape[1]):
            pairs.append((h, a, float(mat[h, a])))
    pairs.sort(key=lambda x: x[2], reverse=True)
    return {
        "p_home_win": p_home,
        "p_draw": p_draw,
        "p_away_win": p_away,
        "most_likely_score": f"{pairs[0][0]}:{pairs[0][1]}",
        "most_likely_score_prob": pairs[0][2],
        "top_scorelines": pairs[:12],
        "score_grid": mat,
    }

def predict_match_v4(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10):
    row = fixture_row(home, away, match_date, tournament=tournament, country=country, neutral=neutral)
    v2_pred = predict_match_v2(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals)
    deep_pred = predict_match_deep(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals, uncertainty_samples=V4_DEEP_UNCERTAINTY_SAMPLES)

    X = _meta_features_for_single(row, v2_pred, deep_pred)
    X = v4_meta_imputer_final.transform(X)
    meta_probs = v4_meta_model_final.predict_proba(X)[0]
    meta_probs = np.clip(meta_probs, 1e-6, 1.0)
    meta_probs = meta_probs / meta_probs.sum()

    base_grid = (1.0 - v4_best_fixed_deep_weight) * _as_grid(v2_pred) + v4_best_fixed_deep_weight * _as_grid(deep_pred)
    base_grid = apply_scoreline_adjustments(base_grid, v4_scoreline_calibration)
    final_grid = reconcile_score_matrix(base_grid, meta_probs)

    out = _summary_from_final_grid(final_grid)
    out["home"] = home
    out["away"] = away
    out["home_xg_pred"] = (1.0 - v4_best_fixed_deep_weight) * v2_pred.get("home_xg_pred", v2_pred.get("home_xg")) + v4_best_fixed_deep_weight * deep_pred["home_xg_pred"]
    out["away_xg_pred"] = (1.0 - v4_best_fixed_deep_weight) * v2_pred.get("away_xg_pred", v2_pred.get("away_xg")) + v4_best_fixed_deep_weight * deep_pred["away_xg_pred"]
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["meta_probs"] = meta_probs
    out["model"] = "weltmeisterki4_sota"
    return out

def predict_neutral_v4(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10):
    p1 = predict_match_v4(team, opponent, match_date, tournament=tournament, country=country, neutral=True, max_goals=max_goals)
    p2 = predict_match_v4(opponent, team, match_date, tournament=tournament, country=country, neutral=True, max_goals=max_goals)
    mat = 0.5 * p1["score_grid"] + 0.5 * p2["score_grid"].T
    mat = mat / mat.sum()
    out = _summary_from_final_grid(mat)
    out["team"] = team
    out["opponent"] = opponent
    out["team_xg_pred"] = 0.5 * (p1["home_xg_pred"] + p2["away_xg_pred"])
    out["opponent_xg_pred"] = 0.5 * (p1["away_xg_pred"] + p2["home_xg_pred"])
    out["home_xg_pred"] = out["team_xg_pred"]
    out["away_xg_pred"] = out["opponent_xg_pred"]
    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["p_home_win"] = out["p_team_win"]
    out["p_away_win"] = out["p_opponent_win"]
    out["model"] = "weltmeisterki4_sota_neutral"
    return out

predict_match = predict_match_v4
predict_neutral = predict_neutral_v4
ACTIVE_MODEL_NAME = "weltmeisterki4_sota_oof_meta_scoreline"

print("Aktives Modell für alle folgenden Prognosen:", ACTIVE_MODEL_NAME)
print("Best fixed base deep weight:", v4_best_fixed_deep_weight)
print("Scoreline calibration:", v4_scoreline_calibration)
print("Performance table:")
print(v4_performance_table)


Saved V4 bundle: C:\ml\projects\WeltmeisterKI\models\weltmeisterki4_sota_meta.joblib
Aktives Modell für alle folgenden Prognosen: weltmeisterki4_sota_oof_meta_scoreline
Best fixed base deep weight: 0.4
Scoreline calibration: {'low_score_boost': 1.0, 'dc_00': 1.2, 'dc_10': 1.1, 'dc_01': 1.1, 'dc_11': 1.05}
Performance table:
shape: (4, 7)
┌─────────────────────────────┬──────┬──────────┬──────────┬──────────────┬──────────┬──────────┐
│ model                       ┆ n    ┆ home_mae ┆ away_mae ┆ 1x2_accuracy ┆ brier    ┆ log_loss │
│ ---                         ┆ ---  ┆ ---      ┆ ---      ┆ ---          ┆ ---      ┆ ---      │
│ str                         ┆ i64  ┆ f64      ┆ f64      ┆ f64          ┆ f64      ┆ f64      │
╞═════════════════════════════╪══════╪══════════╪══════════╪══════════════╪══════════╪══════════╡
│ fixed_hybrid_40pct_deep_oof ┆ 4588 ┆ 1.00884  ┆ 0.826993 ┆ 0.602005     ┆ 0.508935 ┆ 0.865739 │
│ v4_oof_meta_model           ┆ 4588 ┆ 1.00884  ┆ 0.826993 ┆ 0.602005   

In [50]:
# FINAL PRODUCTION MODEL: fixed_hybrid_35pct_deep + scoreline calibration
# Diese Zelle setzt das beste Modell nach LogLoss/Brier als aktives Modell.
# Danach nutzen Deutschland, Südkorea und Monte Carlo automatisch dieses Modell.

from datetime import date
import numpy as np

FIXED_HYBRID_DEEP_WEIGHT = float(v4_best_fixed_deep_weight)  # bei dir: 0.35

def predict_match_fixed_hybrid_scoreline(
    home,
    away,
    match_date,
    tournament="FIFA World Cup",
    country="United States",
    neutral=True,
    max_goals=10,
):
    v2 = predict_match_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
    )

    deep = predict_match_deep(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
        uncertainty_samples=V4_DEEP_UNCERTAINTY_SAMPLES,
    )

    base_grid = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * _as_grid(v2)
        + FIXED_HYBRID_DEEP_WEIGHT * _as_grid(deep)
    )
    base_grid = base_grid / base_grid.sum()

    target_probs = outcome_from_matrix(base_grid)

    adjusted_grid = apply_scoreline_adjustments(base_grid, v4_scoreline_calibration)
    final_grid = reconcile_score_matrix(adjusted_grid, target_probs)

    out = _summary_from_final_grid(final_grid)

    v2_home_xg = v2.get("home_xg_pred", v2.get("home_xg"))
    v2_away_xg = v2.get("away_xg_pred", v2.get("away_xg"))

    out["home"] = home
    out["away"] = away
    out["home_xg_pred"] = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * v2_home_xg
        + FIXED_HYBRID_DEEP_WEIGHT * deep["home_xg_pred"]
    )
    out["away_xg_pred"] = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * v2_away_xg
        + FIXED_HYBRID_DEEP_WEIGHT * deep["away_xg_pred"]
    )
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "fixed_hybrid_35pct_deep_scoreline"

    return out


def predict_neutral_fixed_hybrid_scoreline(
    team,
    opponent,
    match_date,
    tournament="FIFA World Cup",
    country="United States",
    max_goals=10,
):
    p1 = predict_match_fixed_hybrid_scoreline(
        team,
        opponent,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    p2 = predict_match_fixed_hybrid_scoreline(
        opponent,
        team,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    mat = 0.5 * p1["score_grid"] + 0.5 * p2["score_grid"].T
    mat = mat / mat.sum()

    out = _summary_from_final_grid(mat)

    out["team"] = team
    out["opponent"] = opponent
    out["team_xg_pred"] = 0.5 * (p1["home_xg_pred"] + p2["away_xg_pred"])
    out["opponent_xg_pred"] = 0.5 * (p1["away_xg_pred"] + p2["home_xg_pred"])
    out["home_xg_pred"] = out["team_xg_pred"]
    out["away_xg_pred"] = out["opponent_xg_pred"]

    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["p_home_win"] = out["p_team_win"]
    out["p_away_win"] = out["p_opponent_win"]

    out["model"] = "fixed_hybrid_35pct_deep_scoreline_neutral"

    return out


predict_match = predict_match_fixed_hybrid_scoreline
predict_neutral = predict_neutral_fixed_hybrid_scoreline

ACTIVE_MODEL_NAME = f"fixed_hybrid_{int(FIXED_HYBRID_DEEP_WEIGHT * 100)}pct_deep_scoreline"

if "dist_cache" in globals():
    dist_cache.clear()

print("=" * 80)
print("AKTIVES MODELL FÜR ALLE FOLGENDEN ZELLEN")
print("=" * 80)
print("ACTIVE_MODEL_NAME:", ACTIVE_MODEL_NAME)
print("predict_match:", predict_match.__name__)
print("predict_neutral:", predict_neutral.__name__)
print("Deep weight:", FIXED_HYBRID_DEEP_WEIGHT)
print("v2 weight:", 1.0 - FIXED_HYBRID_DEEP_WEIGHT)
print("Scoreline calibration:", v4_scoreline_calibration)

print("\nPerformance-Vergleich:")
print(v4_performance_table)

# Sanity Check an einem Spiel
sanity = predict_neutral(
    "Germany",
    "Curacao",
    date(2026, 6, 14),
    tournament="FIFA World Cup",
    country="United States",
)

print("\nSanity Check: Germany vs Curacao")
print("model:", sanity["model"])
print("Germany xG:", round(sanity["team_xg_pred"], 3))
print("Curacao xG:", round(sanity["opponent_xg_pred"], 3))
print("Germany win %:", round(100 * sanity["p_team_win"], 1))
print("Draw %:", round(100 * sanity["p_draw"], 1))
print("Curacao win %:", round(100 * sanity["p_opponent_win"], 1))
print("Most likely score:", sanity["most_likely_score"])

print("\nJetzt kannst du Deutschland-Vorrunde, Südkorea und Monte Carlo neu starten.")


AKTIVES MODELL FÜR ALLE FOLGENDEN ZELLEN
ACTIVE_MODEL_NAME: fixed_hybrid_40pct_deep_scoreline
predict_match: predict_match_fixed_hybrid_scoreline
predict_neutral: predict_neutral_fixed_hybrid_scoreline
Deep weight: 0.4
v2 weight: 0.6
Scoreline calibration: {'low_score_boost': 1.0, 'dc_00': 1.2, 'dc_10': 1.1, 'dc_01': 1.1, 'dc_11': 1.05}

Performance-Vergleich:
shape: (4, 7)
┌─────────────────────────────┬──────┬──────────┬──────────┬──────────────┬──────────┬──────────┐
│ model                       ┆ n    ┆ home_mae ┆ away_mae ┆ 1x2_accuracy ┆ brier    ┆ log_loss │
│ ---                         ┆ ---  ┆ ---      ┆ ---      ┆ ---          ┆ ---      ┆ ---      │
│ str                         ┆ i64  ┆ f64      ┆ f64      ┆ f64          ┆ f64      ┆ f64      │
╞═════════════════════════════╪══════╪══════════╪══════════╪══════════════╪══════════╪══════════╡
│ fixed_hybrid_40pct_deep_oof ┆ 4588 ┆ 1.00884  ┆ 0.826993 ┆ 0.602005     ┆ 0.508935 ┆ 0.865739 │
│ v4_oof_meta_model           ┆ 458

## 37. 17b. Bereits gespielte WM-Spiele eintragen

Hier pflegst du echte WM-Ergebnisse nach. Danach werden diese Spiele als feste Realität verwendet: Gruppentabellen, Einzelbaum und Monte Carlo starten vom echten Turnierstand.


In [51]:
# Was diese Zelle macht:
# Hier werden alle 72 Gruppenspiele gepflegt. Bereits bekannte Ergebnisse sind eingetragen; bei offenen Spielen bleiben die Scores auf None.
# Wenn neue Ergebnisse dazukommen, nur home_score/away_score ersetzen, diese Zelle ausf?hren und danach die Prognosezellen erneut laufen lassen.

from collections import deque
from datetime import date
import unicodedata
import re
import numpy as np
import polars as pl

REAL_MATCH_STATE_REPEATS = 2

ALL_GROUP_MATCHES_INPUT = [
    {"match_number": 1,  "group": "A", "date": "2026-06-11", "time": None,    "home": "Mexico",       "away": "South Africa",          "home_score": 2,    "away_score": 0},
    {"match_number": 2,  "group": "A", "date": "2026-06-12", "time": None,    "home": "South Korea",  "away": "Czech Republic",         "home_score": 2,    "away_score": 1},
    {"match_number": 3,  "group": "B", "date": "2026-06-12", "time": None,    "home": "Canada",       "away": "Bosnia and Herzegovina", "home_score": 1,    "away_score": 1},
    {"match_number": 4,  "group": "D", "date": "2026-06-13", "time": None,    "home": "USA",          "away": "Paraguay",               "home_score": 4,    "away_score": 1},
    {"match_number": 5,  "group": "B", "date": "2026-06-13", "time": None,    "home": "Qatar",        "away": "Switzerland",            "home_score": 1,    "away_score": 1},
    {"match_number": 6,  "group": "C", "date": "2026-06-14", "time": None,    "home": "Brazil",       "away": "Morocco",                "home_score": 1,    "away_score": 1},
    {"match_number": 7,  "group": "C", "date": "2026-06-14", "time": None,    "home": "Haiti",        "away": "Scotland",               "home_score": 0,    "away_score": 1},
    {"match_number": 8,  "group": "D", "date": "2026-06-14", "time": None,    "home": "Australia",    "away": "Turkey",                 "home_score": 2,    "away_score": 0},
    {"match_number": 9,  "group": "E", "date": "2026-06-14", "time": None,    "home": "Germany",      "away": "Curacao",                "home_score": 7,    "away_score": 1},
    {"match_number": 10, "group": "F", "date": "2026-06-14", "time": None,    "home": "Netherlands",  "away": "Japan",                  "home_score": 2,    "away_score": 2},
    {"match_number": 11, "group": "E", "date": "2026-06-15", "time": None,    "home": "Ivory Coast",  "away": "Ecuador",                "home_score": 1,    "away_score": 0},
    {"match_number": 12, "group": "F", "date": "2026-06-15", "time": None,    "home": "Sweden",       "away": "Tunisia",                "home_score": 5,    "away_score": 1},
    {"match_number": 13, "group": "H", "date": "2026-06-15", "time": None,    "home": "Spain",        "away": "Cape Verde",             "home_score": 0,    "away_score": 0},
    {"match_number": 14, "group": "G", "date": "2026-06-15", "time": None,    "home": "Belgium",      "away": "Egypt",                  "home_score": 1,    "away_score": 1},
    {"match_number": 15, "group": "H", "date": "2026-06-16", "time": None,    "home": "Saudi Arabia", "away": "Uruguay",                "home_score": 1,    "away_score": 1},
    {"match_number": 16, "group": "G", "date": "2026-06-16", "time": None,    "home": "IR Iran",      "away": "New Zealand",            "home_score": 2,    "away_score": 2},
    {"match_number": 17, "group": "I", "date": "2026-06-16", "time": None,    "home": "France",       "away": "Senegal",                "home_score": 3,    "away_score": 1},
    {"match_number": 18, "group": "I", "date": "2026-06-17", "time": None,    "home": "Iraq",         "away": "Norway",                 "home_score": 1,    "away_score": 4},
    {"match_number": 19, "group": "J", "date": "2026-06-17", "time": None,    "home": "Argentina",    "away": "Algeria",                "home_score": 3,    "away_score": 0},
    {"match_number": 20, "group": "J", "date": "2026-06-17", "time": None,    "home": "Austria",      "away": "Jordan",                 "home_score": 3,    "away_score": 1},
    {"match_number": 21, "group": "K", "date": "2026-06-17", "time": None,    "home": "Portugal",     "away": "DR Congo",               "home_score": 1,    "away_score": 1},
    {"match_number": 22, "group": "L", "date": "2026-06-17", "time": None,    "home": "England",      "away": "Croatia",                "home_score": 4,    "away_score": 2},
    {"match_number": 23, "group": "L", "date": "2026-06-18", "time": None,    "home": "Ghana",        "away": "Panama",                 "home_score": 1,    "away_score": 0},
    {"match_number": 24, "group": "K", "date": "2026-06-18", "time": None,    "home": "Uzbekistan",   "away": "Colombia",               "home_score": 1,    "away_score": 3},
    {"match_number": 25, "group": "A", "date": "2026-06-18", "time": None,    "home": "Czech Republic", "away": "South Africa",        "home_score": 1,    "away_score": 1},
    {"match_number": 26, "group": "B", "date": "2026-06-18", "time": None,    "home": "Switzerland",  "away": "Bosnia and Herzegovina", "home_score": 4,    "away_score": 1},
    {"match_number": 27, "group": "B", "date": "2026-06-19", "time": None,    "home": "Canada",       "away": "Qatar",                  "home_score": 6,    "away_score": 0},
    {"match_number": 28, "group": "A", "date": "2026-06-19", "time": None,    "home": "Mexico",       "away": "South Korea",            "home_score": 1,    "away_score": 0},
    {"match_number": 29, "group": "D", "date": "2026-06-19", "time": None,    "home": "USA",          "away": "Australia",              "home_score": 2,    "away_score": 0},
    {"match_number": 30, "group": "C", "date": "2026-06-20", "time": None,    "home": "Scotland",     "away": "Morocco",                "home_score": 0,    "away_score": 1},
    {"match_number": 31, "group": "C", "date": "2026-06-20", "time": None,    "home": "Brazil",       "away": "Haiti",                  "home_score": 3,    "away_score": 0},
    {"match_number": 32, "group": "D", "date": "2026-06-20", "time": None,    "home": "Turkey",       "away": "Paraguay",               "home_score": 0,    "away_score": 1},
    {"match_number": 33, "group": "F", "date": "2026-06-20", "time": None,    "home": "Netherlands",  "away": "Sweden",                 "home_score": 5,    "away_score": 1},
    {"match_number": 34, "group": "E", "date": "2026-06-20", "time": None,    "home": "Germany",      "away": "Ivory Coast",            "home_score": 2,    "away_score": 1},
    {"match_number": 35, "group": "E", "date": "2026-06-21", "time": None,    "home": "Ecuador",      "away": "Curacao",                "home_score": 0,    "away_score": 0},
    {"match_number": 36, "group": "F", "date": "2026-06-21", "time": None,    "home": "Tunisia",      "away": "Japan",                  "home_score": 0,    "away_score": 4},
    {"match_number": 37, "group": "H", "date": "2026-06-21", "time": "18:00", "home": "Spain",        "away": "Saudi Arabia",          "home_score": None, "away_score": None},
    {"match_number": 38, "group": "G", "date": "2026-06-21", "time": "21:00", "home": "Belgium",      "away": "IR Iran",               "home_score": None, "away_score": None},
    {"match_number": 39, "group": "H", "date": "2026-06-22", "time": "00:00", "home": "Uruguay",      "away": "Cape Verde",            "home_score": None, "away_score": None},
    {"match_number": 40, "group": "G", "date": "2026-06-22", "time": "03:00", "home": "New Zealand",  "away": "Egypt",                 "home_score": None, "away_score": None},
    {"match_number": 41, "group": "J", "date": "2026-06-22", "time": "19:00", "home": "Argentina",    "away": "Austria",               "home_score": None, "away_score": None},
    {"match_number": 42, "group": "I", "date": "2026-06-22", "time": "23:00", "home": "France",       "away": "Iraq",                  "home_score": None, "away_score": None},
    {"match_number": 43, "group": "I", "date": "2026-06-23", "time": "02:00", "home": "Norway",       "away": "Senegal",               "home_score": None, "away_score": None},
    {"match_number": 44, "group": "J", "date": "2026-06-23", "time": "05:00", "home": "Jordan",       "away": "Algeria",               "home_score": None, "away_score": None},
    {"match_number": 45, "group": "K", "date": "2026-06-23", "time": "19:00", "home": "Portugal",     "away": "Uzbekistan",            "home_score": None, "away_score": None},
    {"match_number": 46, "group": "L", "date": "2026-06-23", "time": "22:00", "home": "England",      "away": "Ghana",                 "home_score": None, "away_score": None},
    {"match_number": 47, "group": "L", "date": "2026-06-24", "time": "01:00", "home": "Panama",       "away": "Croatia",               "home_score": None, "away_score": None},
    {"match_number": 48, "group": "K", "date": "2026-06-24", "time": "04:00", "home": "Colombia",     "away": "DR Congo",              "home_score": None, "away_score": None},
    {"match_number": 49, "group": "B", "date": "2026-06-24", "time": "21:00", "home": "Switzerland",  "away": "Canada",                "home_score": None, "away_score": None},
    {"match_number": 50, "group": "B", "date": "2026-06-24", "time": "21:00", "home": "Bosnia and Herzegovina", "away": "Qatar",       "home_score": None, "away_score": None},
    {"match_number": 51, "group": "C", "date": "2026-06-25", "time": "00:00", "home": "Scotland",     "away": "Brazil",                "home_score": None, "away_score": None},
    {"match_number": 52, "group": "C", "date": "2026-06-25", "time": "00:00", "home": "Morocco",      "away": "Haiti",                 "home_score": None, "away_score": None},
    {"match_number": 53, "group": "A", "date": "2026-06-25", "time": "03:00", "home": "Czech Republic", "away": "Mexico",            "home_score": None, "away_score": None},
    {"match_number": 54, "group": "A", "date": "2026-06-25", "time": "03:00", "home": "South Africa", "away": "South Korea",          "home_score": None, "away_score": None},
    {"match_number": 55, "group": "E", "date": "2026-06-25", "time": "22:00", "home": "Curacao",      "away": "Ivory Coast",           "home_score": None, "away_score": None},
    {"match_number": 56, "group": "E", "date": "2026-06-25", "time": "22:00", "home": "Ecuador",      "away": "Germany",               "home_score": None, "away_score": None},
    {"match_number": 57, "group": "F", "date": "2026-06-26", "time": "01:00", "home": "Japan",        "away": "Sweden",                "home_score": None, "away_score": None},
    {"match_number": 58, "group": "F", "date": "2026-06-26", "time": "01:00", "home": "Tunisia",      "away": "Netherlands",           "home_score": None, "away_score": None},
    {"match_number": 59, "group": "D", "date": "2026-06-26", "time": "04:00", "home": "Turkey",       "away": "USA",                   "home_score": None, "away_score": None},
    {"match_number": 60, "group": "D", "date": "2026-06-26", "time": "04:00", "home": "Paraguay",     "away": "Australia",             "home_score": None, "away_score": None},
    {"match_number": 61, "group": "I", "date": "2026-06-26", "time": "21:00", "home": "Norway",       "away": "France",                "home_score": None, "away_score": None},
    {"match_number": 62, "group": "I", "date": "2026-06-26", "time": "21:00", "home": "Senegal",      "away": "Iraq",                  "home_score": None, "away_score": None},
    {"match_number": 63, "group": "H", "date": "2026-06-27", "time": "02:00", "home": "Cape Verde",   "away": "Saudi Arabia",          "home_score": None, "away_score": None},
    {"match_number": 64, "group": "H", "date": "2026-06-27", "time": "02:00", "home": "Uruguay",      "away": "Spain",                 "home_score": None, "away_score": None},
    {"match_number": 65, "group": "G", "date": "2026-06-27", "time": "05:00", "home": "Egypt",        "away": "IR Iran",               "home_score": None, "away_score": None},
    {"match_number": 66, "group": "G", "date": "2026-06-27", "time": "05:00", "home": "New Zealand",  "away": "Belgium",               "home_score": None, "away_score": None},
    {"match_number": 67, "group": "L", "date": "2026-06-27", "time": "23:00", "home": "Panama",       "away": "England",               "home_score": None, "away_score": None},
    {"match_number": 68, "group": "L", "date": "2026-06-27", "time": "23:00", "home": "Croatia",      "away": "Ghana",                 "home_score": None, "away_score": None},
    {"match_number": 69, "group": "J", "date": "2026-06-28", "time": None,    "home": "Algeria",      "away": "Austria",               "home_score": None, "away_score": None},
    {"match_number": 70, "group": "J", "date": "2026-06-28", "time": None,    "home": "Jordan",       "away": "Argentina",             "home_score": None, "away_score": None},
    {"match_number": 71, "group": "K", "date": "2026-06-28", "time": None,    "home": "Colombia",     "away": "Portugal",              "home_score": None, "away_score": None},
    {"match_number": 72, "group": "K", "date": "2026-06-28", "time": None,    "home": "DR Congo",     "away": "Uzbekistan",            "home_score": None, "away_score": None},
]

TEAM_ALIASES_LIVE_EXACT = {
    "Cura?ao": "Curacao",
    "Curacao": "Curacao",
    "C?te d'Ivoire": "Ivory Coast",
    "C?te d?Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "Elfenbeink?ste": "Ivory Coast",
    "Ivory Coast": "Ivory Coast",
    "Republic of Korea": "South Korea",
    "Korea Republic": "South Korea",
    "Republik Korea": "South Korea",
    "South Korea": "South Korea",
    "Bosnien und Herzegowina": "Bosnia and Herzegovina",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Tschechien": "Czech Republic",
    "Czechia": "Czech Republic",
    "Czech Republic": "Czech Republic",
    "United States": "USA",
    "USA": "USA",
    "US": "USA",
    "IR Iran": "IR Iran",
    "Iran": "IR Iran",
    "DR Congo": "DR Congo",
    "DR Kongo": "DR Congo",
    "Congo DR": "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",
    "Cabo Verde": "Cape Verde",
    "Cape Verde": "Cape Verde",
    "Kap Verde": "Cape Verde",
    "Schweiz": "Switzerland",
    "Niederlande": "Netherlands",
    "T?rkei": "Turkey",
    "Schweden": "Sweden",
    "Tunesien": "Tunisia",
    "?gypten": "Egypt",
    "Saudi-Arabien": "Saudi Arabia",
    "Neuseeland": "New Zealand",
    "Frankreich": "France",
    "Norwegen": "Norway",
    "Argentinien": "Argentina",
    "?sterreich": "Austria",
    "Jordanien": "Jordan",
    "Irak": "Iraq",
    "Kanada": "Canada",
    "Mexiko": "Mexico",
    "S?dafrika": "South Africa",
    "Brasilien": "Brazil",
    "Marokko": "Morocco",
    "Schottland": "Scotland",
    "Belgien": "Belgium",
}

def _plain_team_key(x):
    x = "" if x is None else str(x).strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", " ", x).strip().lower()

TEAM_ALIASES_LIVE_PLAIN = {_plain_team_key(k): v for k, v in TEAM_ALIASES_LIVE_EXACT.items()}

def canonical_team_live(team):
    raw = "" if team is None else str(team).strip()
    if raw in TEAM_ALIASES_LIVE_EXACT:
        return TEAM_ALIASES_LIVE_EXACT[raw]
    return TEAM_ALIASES_LIVE_PLAIN.get(_plain_team_key(raw), raw)

if "norm_team" in globals():
    if "ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES" not in globals():
        ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES = norm_team

    def norm_team(s):
        base = ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES(s)
        return canonical_team_live(base)

def _score_known(row):
    return row.get("home_score") is not None and row.get("away_score") is not None

def _parse_live_date(row):
    value = row.get("date")
    if isinstance(value, date):
        return value
    try:
        y, m, d = map(int, str(value).split("-"))
        return date(y, m, d)
    except Exception:
        return date(2026, 6, 1)

def normalize_group_matches(rows):
    out, seen = [], set()
    for r in rows:
        rr = dict(r)
        rr["match_number"] = int(rr["match_number"])
        rr["group"] = str(rr.get("group") or "")
        rr["home"] = canonical_team_live(rr["home"])
        rr["away"] = canonical_team_live(rr["away"])
        rr["home_score"] = None if rr.get("home_score") is None else int(rr["home_score"])
        rr["away_score"] = None if rr.get("away_score") is None else int(rr["away_score"])
        if rr["match_number"] in seen:
            raise ValueError(f"match_number doppelt: {rr['match_number']}")
        seen.add(rr["match_number"])
        out.append(rr)
    if len(out) != 72:
        raise ValueError(f"Es muessen 72 Gruppenspiele eingetragen sein, gefunden: {len(out)}")
    return sorted(out, key=lambda x: x["match_number"])

ALL_GROUP_MATCHES = normalize_group_matches(ALL_GROUP_MATCHES_INPUT)
PLAYED_MATCHES = [r for r in ALL_GROUP_MATCHES if _score_known(r)]
OPEN_GROUP_MATCHES = [r for r in ALL_GROUP_MATCHES if not _score_known(r)]
PLAYED_MATCHES_BY_NUMBER = {r["match_number"]: r for r in PLAYED_MATCHES}
PLAYED_MATCHES_VERSION = tuple(
    sorted((r["match_number"], r["home"], r["away"], r["home_score"], r["away_score"]) for r in PLAYED_MATCHES)
)

def get_played_match_result(match_number, home=None, away=None):
    return PLAYED_MATCHES_BY_NUMBER.get(int(match_number))

def _orient_played_result(played, home, away):
    ph = canonical_team_live(played["home"])
    pa = canonical_team_live(played["away"])
    ch = canonical_team_live(home)
    ca = canonical_team_live(away)
    hs = int(played["home_score"])
    aw = int(played["away_score"])
    if ph == ch and pa == ca:
        return hs, aw
    if ph == ca and pa == ch:
        return aw, hs
    raise ValueError(
        "Played result passt nicht zum CSV-Match. "
        f"played={ph} vs {pa}, csv={ch} vs {ca}. "
        "Falls das dasselbe Team ist: Alias oben in TEAM_ALIASES_LIVE_EXACT ergaenzen."
    )

def played_result_as_res(played, home, away, match_date, country, knockout=False, distribution_func=None):
    hs, aw = _orient_played_result(played, home, away)
    model_home = canonical_team_live(home)
    model_away = canonical_team_live(away)
    dist = None
    if distribution_func is not None:
        try:
            dist = distribution_func(model_home, model_away, match_date, country)
        except Exception as e:
            print("Warnung: Modellverteilung fuer gespieltes Match nicht berechnet:", played["match_number"], repr(e))
    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        winner, loser = None, None
    if dist is None:
        p_home = p_draw = p_away = np.nan
        p_home_adv = p_away_adv = np.nan
        home_xg = away_xg = np.nan
        grid = None
        score_prob = np.nan
        most_likely_score = None
    else:
        p_home = float(dist["p_home_win"])
        p_draw = float(dist["p_draw"])
        p_away = float(dist["p_away_win"])
        home_xg = float(dist["home_xg"])
        away_xg = float(dist["away_xg"])
        grid = dist.get("grid")
        score_prob = float(grid[hs, aw]) if grid is not None and hs < grid.shape[0] and aw < grid.shape[1] else np.nan
        most_likely_score = dist.get("most_likely_score")
        denom = max(p_home + p_away, 1e-9)
        p_home_adv = p_home + p_draw * (p_home / denom)
        p_away_adv = p_away + p_draw * (p_away / denom)
    return {
        "home": home,
        "away": away,
        "home_score": int(hs),
        "away_score": int(aw),
        "score": f"{hs}:{aw}",
        "winner": winner,
        "loser": loser,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "p_home_win": p_home,
        "p_draw": p_draw,
        "p_away_win": p_away,
        "p_home_advance": p_home_adv,
        "p_away_advance": p_away_adv,
        "score_grid": grid,
        "most_likely_score": most_likely_score,
        "most_likely_score_prob": score_prob,
        "score_prob_%": round(100 * score_prob, 2) if np.isfinite(score_prob) else None,
        "status": "Gespielt",
    }

def clone_state_v2(src):
    dst = init_state_v2()
    for key in ["elo", "attack", "def_weak", "shoot_w", "shoot_l"]:
        dst[key].update(dict(src[key]))
    for key in ["team_hist", "xg_hist", "scorer_hist", "h2h"]:
        for k, v in src[key].items():
            dst[key][k] = deque(list(v), maxlen=v.maxlen)
    return dst

def apply_played_matches_to_forecast_state(base_state=None, repeats=REAL_MATCH_STATE_REPEATS):
    global state, FORECAST_STATE, BASE_FORECAST_STATE, LIVE_ORIGINAL_MODEL_STATE
    if "LIVE_ORIGINAL_MODEL_STATE" not in globals():
        LIVE_ORIGINAL_MODEL_STATE = clone_state_v2(state)
    if base_state is None:
        base_state = LIVE_ORIGINAL_MODEL_STATE
    BASE_FORECAST_STATE = clone_state_v2(base_state)
    FORECAST_STATE = clone_state_v2(base_state)
    for r in sorted(PLAYED_MATCHES, key=lambda x: x["match_number"]):
        update_row = {
            "date": _parse_live_date(r),
            "home_team": canonical_team_live(r["home"]),
            "away_team": canonical_team_live(r["away"]),
            "home_score": int(r["home_score"]),
            "away_score": int(r["away_score"]),
            "tournament": "FIFA World Cup",
        }
        for _ in range(int(repeats)):
            update_state_after_match_v2(update_row, FORECAST_STATE)
    state = FORECAST_STATE
    for cache_name in ["dist_cache", "single_dist_cache"]:
        if cache_name in globals() and hasattr(globals()[cache_name], "clear"):
            globals()[cache_name].clear()

apply_played_matches_to_forecast_state()

all_group_matches_table = pl.DataFrame([
    {
        "match_number": r["match_number"],
        "group": r["group"],
        "date": r.get("date"),
        "time": r.get("time"),
        "home": r["home"],
        "away": r["away"],
        "home_score": r["home_score"],
        "away_score": r["away_score"],
        "status": "Gespielt" if _score_known(r) else "Offen",
    }
    for r in ALL_GROUP_MATCHES
])

played_matches_table = all_group_matches_table.filter(pl.col("status") == "Gespielt")
open_matches_table = all_group_matches_table.filter(pl.col("status") == "Offen")

print("Gruppenspiele insgesamt:", len(ALL_GROUP_MATCHES))
print("Bereits gespielte Matches:", len(PLAYED_MATCHES))
print("Offene Matches:", len(OPEN_GROUP_MATCHES))
print("REAL_MATCH_STATE_REPEATS:", REAL_MATCH_STATE_REPEATS)
print("Forecast-State wurde aktualisiert.")
print("Alias sanity:")
print("  Cape Verde ->", canonical_team_live("Cape Verde"))
print("  Cabo Verde ->", canonical_team_live("Cabo Verde"))
print("  Kap Verde ->", canonical_team_live("Kap Verde"))
print("  Cura?ao ->", canonical_team_live("Cura?ao"))
print("  Republic of Korea ->", canonical_team_live("Republic of Korea"))
print("\nAlle 72 Gruppenspiele:")
print(all_group_matches_table)
print("\nNoch offene Gruppenspiele:")
print(open_matches_table)


Gruppenspiele insgesamt: 72
Bereits gespielte Matches: 36
Offene Matches: 36
REAL_MATCH_STATE_REPEATS: 2
Forecast-State wurde aktualisiert.
Alias sanity:
  Cape Verde -> Cape Verde
  Cabo Verde -> Cape Verde
  Kap Verde -> Cape Verde
  Cura?ao -> Curacao
  Republic of Korea -> South Korea

Alle 72 Gruppenspiele:
shape: (72, 9)
┌──────────────┬───────┬────────────┬───────┬────────────────────────┬────────────────────────┬────────────┬────────────┬──────────┐
│ match_number ┆ group ┆ date       ┆ time  ┆ home                   ┆ away                   ┆ home_score ┆ away_score ┆ status   │
│ ---          ┆ ---   ┆ ---        ┆ ---   ┆ ---                    ┆ ---                    ┆ ---        ┆ ---        ┆ ---      │
│ i64          ┆ str   ┆ str        ┆ str   ┆ str                    ┆ str                    ┆ i64        ┆ i64        ┆ str      │
╞══════════════╪═══════╪════════════╪═══════╪════════════════════════╪════════════════════════╪════════════╪════════════╪══════════╡
│ 1   

In [52]:
# EINZEL-VORHERSAGE: kompletter WM-2026-Turnierbaum
# Ausgabe ist kompakt:
# - Gruppenspiele ohne stage/date/city/matchup_original
# - KO-Runden ohne stage/label/date/city/matchup_original
# - KO-"Remis" wird als gleichstand_vor_ne_% angezeigt

import re
import numpy as np
import polars as pl
from pathlib import Path
from datetime import date

pl.Config.set_tbl_cols(50)
pl.Config.set_tbl_rows(220)
pl.Config.set_tbl_width_chars(460)
pl.Config.set_fmt_str_lengths(180)

PREFERRED_VIEW_TEAM = "Germany"

print("=" * 90)
print("EINZEL-VORHERSAGE NUTZT MODELL:")
print("ACTIVE_MODEL_NAME:", globals().get("ACTIVE_MODEL_NAME", "unknown"))
print("predict_match:", getattr(predict_match, "__name__", str(predict_match)))
print("predict_neutral:", getattr(predict_neutral, "__name__", str(predict_neutral)))
print("=" * 90)

REQUIRED_FILES = ["teams.csv", "matches.csv", "tournament_stages.csv", "host_cities.csv"]

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "archive",
    Path("D:/ml/projects/WeltmeisterKI"),
    Path("D:/ml/projects/WeltmeisterKI/data/raw/wc2026"),
    Path("C:/ml/projects/WeltmeisterKI"),
    Path("C:/Users/samue/Downloads/archive"),
]

CSV_DIR = None
for p in candidate_dirs:
    if all((p / f).exists() for f in REQUIRED_FILES):
        CSV_DIR = p
        break

if CSV_DIR is None:
    raise FileNotFoundError("CSV-Dateien nicht gefunden. Lege sie zum Notebook oder passe CSV_DIR an.")

print("CSV_DIR:", CSV_DIR)

teams = pl.read_csv(CSV_DIR / "teams.csv")
matches = pl.read_csv(CSV_DIR / "matches.csv")
stages = pl.read_csv(CSV_DIR / "tournament_stages.csv")
cities = pl.read_csv(CSV_DIR / "host_cities.csv")

PLACEHOLDER_REPLACEMENTS = {
    "Winner UEFA Playoff D": "Czech Republic",
    "Winner UEFA Playoff A": "Bosnia and Herzegovina",
    "Winner UEFA Playoff C": "Turkey",
    "Winner UEFA Playoff B": "Sweden",
    "Winner FIFA Playoff 2": "Iraq",
    "Winner FIFA Playoff 1": "DR Congo",
}

teams = teams.with_columns(
    pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.keys())))
    .then(pl.col("team_name").replace(PLACEHOLDER_REPLACEMENTS))
    .otherwise(pl.col("team_name"))
    .alias("team_name")
)

matches = matches.with_columns(
    pl.when((pl.col("match_number") == 100) & (pl.col("match_label").str.contains("W100")))
    .then(pl.lit("W95 vs W96"))
    .otherwise(pl.col("match_label"))
    .alias("match_label")
)

team_id_to_name = dict(teams.select(["id", "team_name"]).iter_rows())
team_to_group = dict(teams.select(["team_name", "group_letter"]).iter_rows())
stage_id_to_name = dict(stages.select(["id", "stage_name"]).iter_rows())
city_id_to_country = dict(cities.select(["id", "country"]).iter_rows())
city_id_to_city = dict(cities.select(["id", "city_name"]).iter_rows())

group_stage_id = next(
    sid for sid, sname in stage_id_to_name.items()
    if "group" in str(sname).lower()
)

group_matches_single = matches.filter(pl.col("stage_id") == group_stage_id).sort("match_number").to_dicts()
ko_matches_single = matches.filter(pl.col("stage_id") != group_stage_id).sort("match_number").to_dicts()
all_teams_single = sorted(team_to_group.keys())

print("Teams:", len(all_teams_single))
print("Group matches:", len(group_matches_single))
print("KO matches:", len(ko_matches_single))

HOST_COUNTRY_TO_TEAM = {
    "United States": "USA",
    "USA": "USA",
    "Mexico": "Mexico",
    "Canada": "Canada",
}

def parse_match_date_single(x):
    return date.fromisoformat(str(x)[:10])

def copy_grid(grid):
    g = np.asarray(grid, dtype=float).copy()
    g = np.clip(g, 1e-12, None)
    return g / g.sum()

def outcome_probs_from_grid_single(grid):
    return {
        "p_home_win": float(np.tril(grid, -1).sum()),
        "p_draw": float(np.trace(grid)),
        "p_away_win": float(np.triu(grid, 1).sum()),
    }

def top_scorelines_for_view_single(grid, view_is_home=True, top_n=8):
    if grid is None:
        return None
    g = np.asarray(grid, dtype=float)
    rows = []
    for h in range(g.shape[0]):
        for a in range(g.shape[1]):
            score = f"{h}:{a}" if view_is_home else f"{a}:{h}"
            rows.append((score, float(g[h, a])))
    rows.sort(key=lambda x: x[1], reverse=True)
    return ", ".join([f"{score} ({100*p:.1f}%)" for score, p in rows[:top_n]])


def ko_advance_probs_from_grid_single(grid):
    probs = outcome_probs_from_grid_single(grid)
    p_home = probs["p_home_win"]
    p_draw = probs["p_draw"]
    p_away = probs["p_away_win"]

    pen_home = p_home / max(p_home + p_away, 1e-9)

    return {
        "p_home_advance": float(p_home + p_draw * pen_home),
        "p_away_advance": float(p_away + p_draw * (1.0 - pen_home)),
    }

single_dist_cache = {}

def predict_distribution_single(home, away, match_date, country, max_goals=10):
    key = (home, away, str(match_date), country, globals().get("ACTIVE_MODEL_NAME", "active"))
    if key in single_dist_cache:
        return single_dist_cache[key]

    host_team = HOST_COUNTRY_TO_TEAM.get(country)

    if host_team == home:
        pred = predict_match(
            home, away, match_date,
            tournament="FIFA World Cup",
            country=country,
            neutral=False,
            max_goals=max_goals,
        )
        grid = copy_grid(pred["score_grid"])
        home_xg = float(pred.get("home_xg_pred", pred.get("home_xg")))
        away_xg = float(pred.get("away_xg_pred", pred.get("away_xg")))

    elif host_team == away:
        pred_rev = predict_match(
            away, home, match_date,
            tournament="FIFA World Cup",
            country=country,
            neutral=False,
            max_goals=max_goals,
        )
        grid = copy_grid(pred_rev["score_grid"]).T
        home_xg = float(pred_rev.get("away_xg_pred", pred_rev.get("away_xg")))
        away_xg = float(pred_rev.get("home_xg_pred", pred_rev.get("home_xg")))

    else:
        pred = predict_neutral(
            home, away, match_date,
            tournament="FIFA World Cup",
            country=country,
            max_goals=max_goals,
        )
        grid = copy_grid(pred["score_grid"])
        home_xg = float(pred.get("team_xg_pred", pred.get("home_xg_pred")))
        away_xg = float(pred.get("opponent_xg_pred", pred.get("away_xg_pred")))

    h, a = np.unravel_index(np.argmax(grid), grid.shape)

    out = {
        "home": home,
        "away": away,
        "date": match_date,
        "country": country,
        "grid": grid,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "most_likely_home_goals": int(h),
        "most_likely_away_goals": int(a),
        "most_likely_score_prob": float(grid[h, a]),
        **outcome_probs_from_grid_single(grid),
        **ko_advance_probs_from_grid_single(grid),
    }

    single_dist_cache[key] = out
    return out

def deterministic_match_single(home, away, match_date, country, knockout=False):
    dist = predict_distribution_single(home, away, match_date, country)

    hs = dist["most_likely_home_goals"]
    aw = dist["most_likely_away_goals"]

    tiebreak = ""

    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        if knockout:
            if dist["p_home_advance"] >= dist["p_away_advance"]:
                winner, loser = home, away
            else:
                winner, loser = away, home
            tiebreak = " n.E."
        else:
            winner, loser = None, None

    return {
        "home": home,
        "away": away,
        "home_score": hs,
        "away_score": aw,
        "score": f"{hs}:{aw}{tiebreak}",
        "winner": winner,
        "loser": loser,
        "home_xg": dist["home_xg"],
        "away_xg": dist["away_xg"],
        "p_home_win": dist["p_home_win"],
        "p_draw": dist["p_draw"],
        "p_away_win": dist["p_away_win"],
        "p_home_advance": dist["p_home_advance"],
        "p_away_advance": dist["p_away_advance"],
        "score_grid": dist["grid"],
        "score_prob_%": round(100 * dist["most_likely_score_prob"], 2),
        "status": "Prognose",
    }

def format_view_record(base, res, knockout=False):
    home = res["home"]
    away = res["away"]

    if PREFERRED_VIEW_TEAM in (home, away):
        view = PREFERRED_VIEW_TEAM
    else:
        view = home

    if view == home:
        opp = away
        view_score = res["score"]
        view_xg = res["home_xg"]
        opp_xg = res["away_xg"]
        p_win = res["p_home_win"]
        p_loss = res["p_away_win"]
        p_advance = res["p_home_advance"]
    else:
        opp = home
        raw_score = str(res["score"])
        pen = " n.E." if "n.E." in raw_score else ""
        clean = raw_score.replace(" n.E.", "")
        hs, aw = map(int, clean.split(":"))

        view_score = f"{aw}:{hs}{pen}"
        view_xg = res["away_xg"]
        opp_xg = res["home_xg"]
        p_win = res["p_away_win"]
        p_loss = res["p_home_win"]
        p_advance = res["p_away_advance"]

    out = dict(base)
    out.update({
        "sicht_matchup": f"{view} vs {opp}",
        "sicht_team": view,
        "gegner": opp,
        "sicht_score": view_score,
        "winner": res["winner"],
        "loser": res["loser"],
        "sicht_xg": round(view_xg, 3),
        "gegner_xg": round(opp_xg, 3),
        "sicht_sieg_%": round(100 * p_win, 1),
        "sicht_niederlage_%": round(100 * p_loss, 1),
        "score_prob_%": res["score_prob_%"],
        "top8_scorelines": top_scorelines_for_view_single(res.get("score_grid"), view_is_home=(view == home)) if res.get("score_grid") is not None else None,
        "status": res.get("status", "Prognose"),
    })

    if knockout:
        out["gleichstand_vor_ne_%"] = round(100 * res["p_draw"], 1)
        out["sicht_weiterkommen_%"] = round(100 * p_advance, 1)
    else:
        out["remis_%"] = round(100 * res["p_draw"], 1)

    return out

def empty_table_row_single(team):
    return {
        "team": team,
        "group": team_to_group[team],
        "mp": 0,
        "w": 0,
        "d": 0,
        "l": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "pts": 0,
    }

def add_group_result_single(table, home, away, hs, aw):
    table[home]["mp"] += 1
    table[away]["mp"] += 1

    table[home]["gf"] += hs
    table[home]["ga"] += aw
    table[away]["gf"] += aw
    table[away]["ga"] += hs

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if hs > aw:
        table[home]["w"] += 1
        table[away]["l"] += 1
        table[home]["pts"] += 3
    elif aw > hs:
        table[away]["w"] += 1
        table[home]["l"] += 1
        table[away]["pts"] += 3
    else:
        table[home]["d"] += 1
        table[away]["d"] += 1
        table[home]["pts"] += 1
        table[away]["pts"] += 1

def team_tiebreak_strength(team):
    score = 0.0

    try:
        score += float(state["elo"][norm_team(team)])
    except Exception:
        pass

    try:
        rank, points, conf = fifa_before(norm_team(team), date(2026, 6, 1))
        if not np.isnan(rank):
            score += 250.0 - float(rank)
        if not np.isnan(points):
            score += float(points) / 10.0
    except Exception:
        pass

    return score

def rank_group_rows_single(rows):
    return sorted(
        rows,
        key=lambda r: (
            r["pts"],
            r["gd"],
            r["gf"],
            r["w"],
            team_tiebreak_strength(r["team"]),
        ),
        reverse=True,
    )

def split_match_label_single(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

third_slot_pattern = re.compile(r"^3([A-L]+)$")

def collect_third_slots_single():
    slots = []
    for m in ko_matches_single:
        for p in split_match_label_single(m["match_label"]):
            if third_slot_pattern.fullmatch(p):
                slots.append(p)
    return list(dict.fromkeys(slots))

THIRD_SLOTS_SINGLE = collect_third_slots_single()

def allocate_third_place_slots_single(best_thirds):
    third_by_group = {r["group"]: r for r in best_thirds}
    available_groups = set(third_by_group.keys())
    third_rank_order = [r["group"] for r in best_thirds]
    allowed = {slot: set(slot[1:]) for slot in THIRD_SLOTS_SINGLE}
    ordered_slots = sorted(THIRD_SLOTS_SINGLE, key=lambda s: len(allowed[s] & available_groups))

    def rec(i, remaining, assignment):
        if i == len(ordered_slots):
            return assignment

        slot = ordered_slots[i]
        candidates = [
            g for g in third_rank_order
            if g in remaining and g in allowed[slot]
        ]

        for g in candidates:
            nxt = dict(assignment)
            nxt[slot] = third_by_group[g]["team"]
            result = rec(i + 1, remaining - {g}, nxt)
            if result is not None:
                return result

        return None

    assignment = rec(0, available_groups, {})
    if assignment is None:
        raise RuntimeError(f"Kann Third-place Slots nicht zuordnen: {sorted(available_groups)}")

    return assignment

def resolve_slot_single(slot, qualifiers, match_results):
    slot = str(slot).strip()

    if re.fullmatch(r"[12][A-L]", slot):
        return qualifiers[slot]

    if third_slot_pattern.fullmatch(slot):
        return qualifiers[slot]

    if re.fullmatch(r"W\d+", slot):
        return match_results[int(slot[1:])]["winner"]

    if re.fullmatch(r"RU\d+", slot):
        return match_results[int(slot[2:])]["loser"]

    raise RuntimeError(f"Unbekannter KO-Slot: {slot}")

table = {team: empty_table_row_single(team) for team in all_teams_single}
group_match_records = []

for m in group_matches_single:
    mn = int(m["match_number"])
    home = team_id_to_name[int(m["home_team_id"])]
    away = team_id_to_name[int(m["away_team_id"])]
    d = parse_match_date_single(m["kickoff_at"])
    country = city_id_to_country[int(m["city_id"])]

    fixed = get_played_match_result(mn, home, away) if "get_played_match_result" in globals() else None
    if fixed is not None:
        res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_distribution_single)
    else:
        res = deterministic_match_single(home, away, d, country, knockout=False)

    add_group_result_single(table, home, away, res["home_score"], res["away_score"])

    group_match_records.append(format_view_record({
        "match": mn,
        "group": team_to_group[home],
    }, res, knockout=False))

group_rankings = {}
qualifiers = {}

for g in sorted(set(team_to_group.values())):
    ranked = rank_group_rows_single([dict(v) for v in table.values() if v["group"] == g])
    group_rankings[g] = ranked
    qualifiers[f"1{g}"] = ranked[0]["team"]
    qualifiers[f"2{g}"] = ranked[1]["team"]

thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
best_thirds = rank_group_rows_single(thirds)[:8]
qualifiers.update(allocate_third_place_slots_single(best_thirds))

match_results = {}
ko_records = []

for m in ko_matches_single:
    mn = int(m["match_number"])
    stage = stage_id_to_name[int(m["stage_id"])]
    label = str(m["match_label"])
    parts = split_match_label_single(label)

    home = resolve_slot_single(parts[0], qualifiers, match_results)
    away = resolve_slot_single(parts[1], qualifiers, match_results)

    d = parse_match_date_single(m["kickoff_at"])
    country = city_id_to_country[int(m["city_id"])]

    res = deterministic_match_single(home, away, d, country, knockout=True)
    match_results[mn] = res

    ko_records.append(format_view_record({
        "match": mn,
        "runde": stage,
    }, res, knockout=True))

single_group_matches = pl.DataFrame(group_match_records).sort("match")

single_group_tables_rows = []
single_group_winners_rows = []
single_best_thirds_rows = []

for g in sorted(group_rankings.keys()):
    for pos, r in enumerate(group_rankings[g], start=1):
        single_group_tables_rows.append({
            "group": g,
            "pos": pos,
            "team": r["team"],
            "pts": r["pts"],
            "mp": r["mp"],
            "w": r["w"],
            "d": r["d"],
            "l": r["l"],
            "gf": r["gf"],
            "ga": r["ga"],
            "gd": r["gd"],
        })

    single_group_winners_rows.append({
        "group": g,
        "sieger": group_rankings[g][0]["team"],
        "sieger_pts": group_rankings[g][0]["pts"],
        "zweiter": group_rankings[g][1]["team"],
        "zweiter_pts": group_rankings[g][1]["pts"],
        "dritter": group_rankings[g][2]["team"],
        "dritter_pts": group_rankings[g][2]["pts"],
    })

for pos, r in enumerate(best_thirds, start=1):
    single_best_thirds_rows.append({
        "best_third_rank": pos,
        "group": r["group"],
        "team": r["team"],
        "pts": r["pts"],
        "gd": r["gd"],
        "gf": r["gf"],
    })

single_group_tables = pl.DataFrame(single_group_tables_rows).sort(["group", "pos"])
single_group_winners = pl.DataFrame(single_group_winners_rows).sort("group")
single_best_thirds = pl.DataFrame(single_best_thirds_rows)
single_ko_tree = pl.DataFrame(ko_records).sort("match")

final_rec = next((r for r in ko_records if str(r["runde"]).lower() == "final"), None)
third_rec = next((r for r in ko_records if "third" in str(r["runde"]).lower()), None)

single_medals = pl.DataFrame([{
    "gold": final_rec["winner"] if final_rec else None,
    "silver": final_rec["loser"] if final_rec else None,
    "bronze": third_rec["winner"] if third_rec else None,
    "fourth": third_rec["loser"] if third_rec else None,
}])

GAME_COLS_GROUP = [
    "match",
    "group",
    "sicht_matchup",
    "sicht_score",
    "sicht_sieg_%",
    "remis_%",
    "sicht_niederlage_%",
    "sicht_xg",
    "gegner_xg",
    "score_prob_%",
    "top8_scorelines",
    "status",
]

GAME_COLS_KO = [
    "match",
    "sicht_matchup",
    "sicht_score",
    "winner",
    "loser",
    "sicht_sieg_%",
    "sicht_niederlage_%",
    "gleichstand_vor_ne_%",
    "sicht_weiterkommen_%",
    "sicht_xg",
    "gegner_xg",
    "score_prob_%",
    "top8_scorelines",
    "status",
]

print("\n=== Alle Gruppenspiele: konkrete Einzel-Vorhersage ===")
print(single_group_matches.select(GAME_COLS_GROUP))

print("\n=== Gruppentabellen ===")
print(single_group_tables)

print("\n=== Gruppensieger / Gruppenzweite / Gruppendritte ===")
print(single_group_winners)

print("\n=== Beste Gruppendritte, die weiterkommen ===")
print(single_best_thirds)

print("\n=== KO-Baum: konkrete Einzel-Vorhersage ===")
for stage in ["Round of 32", "Round of 16", "Quarterfinals", "Semifinals", "Third Place Playoff", "Final"]:
    part = single_ko_tree.filter(pl.col("runde") == stage)
    if part.height == 0:
        continue

    print(f"\n--- {stage} ---")
    print(part.select(GAME_COLS_KO))

print("\n=== Medaillen: konkrete Einzel-Vorhersage ===")
print(single_medals)

print("\n=== Deutschland / Suedkorea im Einzel-Turnier ===")
for team in ["Germany", "South Korea"]:
    team_group = single_group_tables.filter(pl.col("team") == team)
    team_ko = single_ko_tree.filter(
        (pl.col("sicht_matchup").str.contains(team))
        | (pl.col("winner") == team)
        | (pl.col("loser") == team)
    )

    print(f"\n{team} Gruppentabelle:")
    print(team_group)

    print(f"\n{team} KO-Spiele:")
    if team_ko.height:
        print(team_ko.select([
            "runde",
            "sicht_matchup",
            "sicht_score",
            "winner",
            "loser",
            "sicht_sieg_%",
            "sicht_niederlage_%",
            "gleichstand_vor_ne_%",
            "sicht_weiterkommen_%",
        ]))
    else:
        print("Keine KO-Spiele im konkreten Einzelpfad.")


EINZEL-VORHERSAGE NUTZT MODELL:
ACTIVE_MODEL_NAME: fixed_hybrid_40pct_deep_scoreline
predict_match: predict_match_fixed_hybrid_scoreline
predict_neutral: predict_neutral_fixed_hybrid_scoreline
CSV_DIR: C:\ml\projects\WeltmeisterKI
Teams: 48
Group matches: 72
KO matches: 32

=== Alle Gruppenspiele: konkrete Einzel-Vorhersage ===
shape: (72, 12)
┌───────┬───────┬───────────────────────────────────────┬─────────────┬──────────────┬─────────┬────────────────────┬──────────┬───────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┐
│ match ┆ group ┆ sicht_matchup                         ┆ sicht_score ┆ sicht_sieg_% ┆ remis_% ┆ sicht_niederlage_% ┆ sicht_xg ┆ gegner_xg ┆ score_prob_% ┆ top8_scorelines                                                                                    ┆ status   │
│ ---   ┆ ---   ┆ ---                                   ┆ ---         ┆ ---          ┆ ---     ┆ ---                ┆ 

## 40. Deutschland Gruppenspiele live

Zeigt gespielte Deutschland-Spiele als echte Resultate und offene Deutschland-Spiele als Prognose mit Wahrscheinlichkeiten, xG und Top-8-Scorelines.


In [53]:
# Was diese Zelle macht:
# Zeigt Deutschlands Gruppenspiele mit Status: gespielte Spiele als echtes Ergebnis, offene Spiele als Modellprognose.
# Enthält Sieg/Remis/Niederlage, xG, wahrscheinlichsten Score und Top-8-Scorelines.

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(80)
pl.Config.set_fmt_str_lengths(120)

fixtures = [
    {"match_number": 9,  "team": "Germany", "opponent": "Curacao",     "display": "Deutschland vs Curacao",        "date": date(2026, 6, 14), "country": "United States"},
    {"match_number": 34, "team": "Germany", "opponent": "Ivory Coast", "display": "Deutschland vs Elfenbeinküste", "date": date(2026, 6, 20), "country": "United States"},
    {"match_number": 56, "team": "Germany", "opponent": "Ecuador",     "display": "Deutschland vs Ecuador",       "date": date(2026, 6, 25), "country": "United States"},
]

def top8_from_prediction(pred):
    return ", ".join([f"{h}:{a} ({100*p:.1f}%)" for h, a, p in pred.get("top_scorelines", [])[:8]])

preds, score_rows = [], []
for fx in fixtures:
    fixed = get_played_match_result(fx["match_number"], fx["team"], fx["opponent"]) if "get_played_match_result" in globals() else None
    pred = predict_neutral(fx["team"], fx["opponent"], fx["date"], fx["country"])
    gw, dr, ow = pred["p_team_win"], pred["p_draw"], pred["p_opponent_win"]
    if fixed is not None:
        hs, aw = _orient_played_result(fixed, fx["team"], fx["opponent"])
        status = "Gespielt"
        result_score = f"{hs}:{aw}"
        pick = "Deutschland gewinnt" if hs > aw else "Unentschieden" if hs == aw else "Deutschland verliert"
    else:
        status = "Prognose"
        result_score = pred["most_likely_score"]
        pick = "Deutschland gewinnt" if gw > max(dr, ow) else "Unentschieden" if dr > max(gw, ow) else "Gegner gewinnt"
    preds.append({
        "match": fx["display"], "status": status, "score": result_score, "date": fx["date"],
        "deutschland_xg": round(pred["team_xg_pred"], 3), "gegner_xg": round(pred["opponent_xg_pred"], 3),
        "deutschland_sieg_%": round(100 * gw, 1), "remis_%": round(100 * dr, 1), "deutschland_niederlage_%": round(100 * ow, 1),
        "modell_pick": pick, "wahrscheinlichster_modell_score": pred["most_likely_score"],
        "score_wahrscheinlichkeit_%": round(100 * pred["most_likely_score_prob"], 2),
        "top8_scorelines": top8_from_prediction(pred),
    })
    for tg, og, p in pred["top_scorelines"]:
        score_rows.append({"match": fx["display"], "score": f"{tg}:{og}", "wahrscheinlichkeit_%": round(100 * p, 2)})
summary = pl.DataFrame(preds)
top_scores = pl.DataFrame(score_rows)
print("Deutschland Gruppenspiele")
print(summary)
print("\nTop-Scorelines")
print(top_scores)


Deutschland Gruppenspiele
shape: (3, 13)
┌───────────────────────────────┬──────────┬───────┬────────────┬────────────────┬───────────┬────────────────────┬─────────┬──────────────────────────┬─────────────────────┬─────────────────────────────────┬────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────┐
│ match                         ┆ status   ┆ score ┆ date       ┆ deutschland_xg ┆ gegner_xg ┆ deutschland_sieg_% ┆ remis_% ┆ deutschland_niederlage_% ┆ modell_pick         ┆ wahrscheinlichster_modell_score ┆ score_wahrscheinlichkeit_% ┆ top8_scorelines                                                                                   │
│ ---                           ┆ ---      ┆ ---   ┆ ---        ┆ ---            ┆ ---       ┆ ---                ┆ ---     ┆ ---                      ┆ ---                 ┆ ---                             ┆ ---                        ┆ ---                                          

## 42. Deutschland Score-Zusammenfassung

Verdichtet die Score-Verteilungen: Expected Score, gerundeter Score, Top-8-gewichteter Score und praktischer Tipp.


In [54]:
# Was diese Zelle macht:
# Verdichtet die Score-Verteilungen und ergänzt Sieg-/Remis-/Niederlage-Wahrscheinlichkeiten.
# Robust gegen unterschiedliche Spaltennamen in summary.

import numpy as np
import polars as pl

def first_existing(row, candidates, default=None):
    for c in candidates:
        if c in row:
            return row[c]
    return default

def require_existing(row, candidates, label):
    value = first_existing(row, candidates, None)
    if value is None:
        raise KeyError(
            f"Keine passende Spalte fuer {label} gefunden. "
            f"Gesuchte Namen: {candidates}. "
            f"Vorhandene Spalten: {list(row.keys())}"
        )
    return value

score_summary_rows = []

for fx in fixtures:
    match_name = fx["display"]

    pred_row = summary.filter(pl.col("match") == match_name).row(0, named=True)
    rows_for_match = top_scores.filter(pl.col("match") == match_name)

    scores = []
    for r in rows_for_match.iter_rows(named=True):
        h, a = map(int, str(r["score"]).split(":"))
        p = float(r["wahrscheinlichkeit_%"]) / 100.0
        scores.append((h, a, p))

    if len(scores) < 2:
        raise ValueError(f"Zu wenige Scorelines fuer {match_name}: {scores}")

    probs = np.array([p for _, _, p in scores], dtype=float)
    probs = probs / probs.sum()

    avg_h = sum(h * w for (h, a, _), w in zip(scores, probs))
    avg_a = sum(a * w for (h, a, _), w in zip(scores, probs))

    top1 = scores[0]
    top2 = scores[1]
    diff_pp = abs(top1[2] - top2[2]) * 100

    practical_pick = (
        f"{top1[0]}:{top1[1]} oder {top2[0]}:{top2[1]}"
        if diff_pp < 0.75
        else f"{top1[0]}:{top1[1]}"
    )

    deutschland_sieg = require_existing(
        pred_row,
        ["deutschland_sieg_%", "team_win_%", "home_win_%", "germany_win_%"],
        "Deutschland-Sieg",
    )

    remis = require_existing(
        pred_row,
        ["remis_%", "draw_%", "draw_pct"],
        "Remis",
    )

    deutschland_niederlage = require_existing(
        pred_row,
        ["deutschland_niederlage_%", "gegner_sieg_%", "opponent_win_%", "opponent_sieg_%", "away_win_%", "gegner_win_%"],
        "Deutschland-Niederlage",
    )

    deutschland_xg = require_existing(
        pred_row,
        ["deutschland_xg", "team_xg", "home_xg", "germany_xg"],
        "Deutschland-xG",
    )

    gegner_xg = require_existing(
        pred_row,
        ["gegner_xg", "opponent_xg", "away_xg"],
        "Gegner-xG",
    )

    score_summary_rows.append({
        "match": match_name,
        "win_pick": pred_row.get("modell_pick", pred_row.get("model_pick", "")),

        "deutschland_sieg_%": deutschland_sieg,
        "remis_%": remis,
        "deutschland_niederlage_%": deutschland_niederlage,

        "expected_score": f"{float(deutschland_xg):.2f}:{float(gegner_xg):.2f}",
        "expected_rounded": f"{round(float(deutschland_xg))}:{round(float(gegner_xg))}",
        "top8_weighted": f"{avg_h:.2f}:{avg_a:.2f}",
        "top8_rounded": f"{round(avg_h)}:{round(avg_a)}",
        "practical_score_pick": practical_pick,
        "top1_top2_diff_pp": round(diff_pp, 3),
    })

score_summary = pl.DataFrame(score_summary_rows)

print("Summary-Spalten:", summary.columns)
print("\nScore-Zusammenfassung")
print(score_summary)


Summary-Spalten: ['match', 'status', 'score', 'date', 'deutschland_xg', 'gegner_xg', 'deutschland_sieg_%', 'remis_%', 'deutschland_niederlage_%', 'modell_pick', 'wahrscheinlichster_modell_score', 'score_wahrscheinlichkeit_%', 'top8_scorelines']

Score-Zusammenfassung
shape: (3, 11)
┌───────────────────────────────┬─────────────────────┬────────────────────┬─────────┬──────────────────────────┬────────────────┬──────────────────┬───────────────┬──────────────┬──────────────────────┬───────────────────┐
│ match                         ┆ win_pick            ┆ deutschland_sieg_% ┆ remis_% ┆ deutschland_niederlage_% ┆ expected_score ┆ expected_rounded ┆ top8_weighted ┆ top8_rounded ┆ practical_score_pick ┆ top1_top2_diff_pp │
│ ---                           ┆ ---                 ┆ ---                ┆ ---     ┆ ---                      ┆ ---            ┆ ---              ┆ ---           ┆ ---          ┆ ---                  ┆ ---               │
│ str                           ┆ str        

## 44. Südkorea Vorrunde

Erzeugt dieselbe kompakte Prognose für Südkorea gegen Tschechien, Mexiko und Südafrika.


In [55]:
# Was diese Zelle macht:
# Erzeugt dieselbe kompakte Prognose für Südkorea gegen Tschechien, Mexiko und Südafrika.

import polars as pl
import numpy as np
from datetime import date

pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(80)
pl.Config.set_fmt_str_lengths(40)

fixtures_korea = [
    {"team": "South Korea", "opponent": "Czech Republic", "display": "Südkorea vs Tschechien", "date": date(2026, 6, 15), "country": "United States"},
    {"team": "South Korea", "opponent": "Mexico", "display": "Südkorea vs Mexiko", "date": date(2026, 6, 20), "country": "United States"},
    {"team": "South Korea", "opponent": "South Africa", "display": "Südkorea vs Südafrika", "date": date(2026, 6, 25), "country": "United States"},
]

DISPLAY_NAMES_KOREA = {
    "Czech Republic": "Tschechien",
    "Mexico": "Mexiko",
    "South Africa": "Südafrika",
}

preds = []
score_rows = []

for fx in fixtures_korea:
    pred = predict_neutral(
        fx["team"],
        fx["opponent"],
        fx["date"],
        country=fx["country"],
    )

    team_win = pred["p_team_win"]
    draw = pred["p_draw"]
    opp_win = pred["p_opponent_win"]

    if team_win > draw and team_win > opp_win:
        pick = "Südkorea gewinnt"
    elif opp_win > team_win and opp_win > draw:
        pick = f'{DISPLAY_NAMES_KOREA.get(fx["opponent"], fx["opponent"])} gewinnt'
    else:
        pick = "Unentschieden"

    preds.append({
        "match": fx["display"],
        "date": fx["date"],
        "suedkorea_xg": round(pred["team_xg_pred"], 3),
        "gegner_xg": round(pred["opponent_xg_pred"], 3),
        "suedkorea_sieg_%": round(100 * team_win, 1),
        "remis_%": round(100 * draw, 1),
        "gegner_sieg_%": round(100 * opp_win, 1),
        "modell_pick": pick,
        "wahrscheinlichster_score": pred["most_likely_score"],
        "score_wahrscheinlichkeit_%": round(100 * pred["most_likely_score_prob"], 2),
    })

    for tg, og, p in pred["top_scorelines"]:
        score_rows.append({
            "match": fx["display"],
            "score": f"{tg}:{og}",
            "wahrscheinlichkeit_%": round(100 * p, 2),
        })

summary_korea = pl.DataFrame(preds)
top_scores_korea = pl.DataFrame(score_rows)

score_summary_rows = []

for fx in fixtures_korea:
    match_name = fx["display"]

    pred_row = summary_korea.filter(pl.col("match") == match_name).row(0, named=True)
    rows_for_match = top_scores_korea.filter(pl.col("match") == match_name)

    scores = []
    for r in rows_for_match.iter_rows(named=True):
        h, a = map(int, r["score"].split(":"))
        p = float(r["wahrscheinlichkeit_%"]) / 100.0
        scores.append((h, a, p))

    probs = np.array([p for _, _, p in scores], dtype=float)
    probs = probs / probs.sum()

    avg_h = sum(h * w for (h, a, _), w in zip(scores, probs))
    avg_a = sum(a * w for (h, a, _), w in zip(scores, probs))

    top1 = scores[0]
    top2 = scores[1]
    diff_pp = abs(top1[2] - top2[2]) * 100

    practical_pick = (
        f"{top1[0]}:{top1[1]} oder {top2[0]}:{top2[1]}"
        if diff_pp < 0.75
        else f"{top1[0]}:{top1[1]}"
    )

    score_summary_rows.append({
        "match": match_name,
        "win_pick": pred_row["modell_pick"],
        "expected_score": f'{pred_row["suedkorea_xg"]:.2f}:{pred_row["gegner_xg"]:.2f}',
        "expected_rounded": f'{round(pred_row["suedkorea_xg"])}:{round(pred_row["gegner_xg"])}',
        "top8_weighted": f"{avg_h:.2f}:{avg_a:.2f}",
        "top8_rounded": f"{round(avg_h)}:{round(avg_a)}",
        "practical_score_pick": practical_pick,
        "top1_top2_diff_pp": round(diff_pp, 3),
    })

score_summary_korea = pl.DataFrame(score_summary_rows)

print("Südkorea Gruppenprognose")
print(summary_korea)

print("\nSüdkorea Top-Scorelines")
print(top_scores_korea)

print("\nSüdkorea Score-Zusammenfassung")
print(score_summary_korea)


Südkorea Gruppenprognose
shape: (3, 10)
┌────────────────────────┬────────────┬──────────────┬───────────┬──────────────────┬─────────┬───────────────┬──────────────────┬──────────────────────────┬────────────────────────────┐
│ match                  ┆ date       ┆ suedkorea_xg ┆ gegner_xg ┆ suedkorea_sieg_% ┆ remis_% ┆ gegner_sieg_% ┆ modell_pick      ┆ wahrscheinlichster_score ┆ score_wahrscheinlichkeit_% │
│ ---                    ┆ ---        ┆ ---          ┆ ---       ┆ ---              ┆ ---     ┆ ---           ┆ ---              ┆ ---                      ┆ ---                        │
│ str                    ┆ date       ┆ f64          ┆ f64       ┆ f64              ┆ f64     ┆ f64           ┆ str              ┆ str                      ┆ f64                        │
╞════════════════════════╪════════════╪══════════════╪═══════════╪══════════════════╪═════════╪═══════════════╪══════════════════╪══════════════════════════╪════════════════════════════╡
│ Südkorea vs Tschechien 

## 46. Monte Carlo WM 2026

Simuliert die komplette WM viele Male mit dem aktiven Modell. Ergebnis: Titelchancen, 16telfinalisten, Exit-Runden und häufigste Exit-Gegner.


In [56]:
# Was diese Zelle macht:
# Simuliert die komplette WM viele Male mit dem aktiven Modell. Ergebnis: Titelchancen, 16telfinalisten, Exit-Runden und häufigste Exit-Gegner.

import re
import math
import time
import numpy as np
import polars as pl
from pathlib import Path
from collections import defaultdict, Counter
from datetime import date

# Monte Carlo WM 2026
# Nutzt den aktiven Prediction Helper aus predict_match/predict_neutral: v2 oder Hybrid, je nach Auswahl-Zelle.
# Voraussetzung:
# - Prediction Helper v2 wurde ausgeführt
# - predict_match(...)
# - predict_neutral(...)
# - score_grid im Output vorhanden

N_SIMS = 50_000
RNG_SEED = 42
MAX_GOALS = 10

rng = np.random.default_rng(RNG_SEED)

pl.Config.set_tbl_cols(35)
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_width_chars(340)
pl.Config.set_fmt_str_lengths(160)

required_prediction_helpers = ["predict_match", "predict_neutral"]
missing_prediction_helpers = [x for x in required_prediction_helpers if x not in globals()]

if missing_prediction_helpers:
    raise RuntimeError(
        "Bitte erst die Prediction-Helper-v2-Zelle ausführen. "
        f"Fehlt aktuell: {missing_prediction_helpers}"
    )

print("Monte Carlo nutzt Prediction Helper:")
print("predict_match:", predict_match)
print("predict_neutral:", predict_neutral)

if "calibration" in globals():
    print("Calibration:", calibration)

# 1. CSV-Ordner finden

REQUIRED_FILES = ["teams.csv", "matches.csv", "tournament_stages.csv", "host_cities.csv"]

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "archive",
    Path("D:/ml/projects/WeltmeisterKI"),
    Path("D:/ml/projects/WeltmeisterKI/data/raw/wc2026"),
    Path("C:/ml/projects/WeltmeisterKI"),
    Path("C:/Users/samue/Downloads/archive"),
]

CSV_DIR = None
for p in candidate_dirs:
    if all((p / f).exists() for f in REQUIRED_FILES):
        CSV_DIR = p
        break

if CSV_DIR is None:
    raise FileNotFoundError(
        "Ich finde teams.csv/matches.csv/tournament_stages.csv/host_cities.csv nicht. "
        "Lege sie in denselben Ordner wie das Notebook oder passe CSV_DIR manuell an."
    )

print("CSV_DIR:", CSV_DIR)

teams = pl.read_csv(CSV_DIR / "teams.csv")
matches = pl.read_csv(CSV_DIR / "matches.csv")
stages = pl.read_csv(CSV_DIR / "tournament_stages.csv")
cities = pl.read_csv(CSV_DIR / "host_cities.csv")

# 2. Placeholder ersetzen

PLACEHOLDER_REPLACEMENTS = {
    "Winner UEFA Playoff D": "Czech Republic",
    "Winner UEFA Playoff A": "Bosnia and Herzegovina",
    "Winner UEFA Playoff C": "Turkey",
    "Winner UEFA Playoff B": "Sweden",
    "Winner FIFA Playoff 2": "Iraq",
    "Winner FIFA Playoff 1": "DR Congo",
}

teams = teams.with_columns(
    pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.keys())))
    .then(pl.col("team_name").replace(PLACEHOLDER_REPLACEMENTS))
    .otherwise(pl.col("team_name"))
    .alias("team_name")
)

if "is_placeholder" in teams.columns:
    teams = teams.with_columns(
        pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.values())))
        .then(False)
        .otherwise(pl.col("is_placeholder"))
        .alias("is_placeholder")
    )

# Patch für fehlerhafte self-reference in alter CSV.
matches = matches.with_columns(
    pl.when((pl.col("match_number") == 100) & (pl.col("match_label").str.contains("W100")))
    .then(pl.lit("W95 vs W96"))
    .otherwise(pl.col("match_label"))
    .alias("match_label")
)

# 3. Lookups

team_id_to_name = dict(teams.select(["id", "team_name"]).iter_rows())
team_to_group = dict(teams.select(["team_name", "group_letter"]).iter_rows())

stage_id_to_name = dict(stages.select(["id", "stage_name"]).iter_rows())
city_id_to_country = dict(cities.select(["id", "country"]).iter_rows())
city_id_to_city = dict(cities.select(["id", "city_name"]).iter_rows())

group_stage_id = None
for sid, sname in stage_id_to_name.items():
    if "group" in str(sname).lower():
        group_stage_id = sid
        break

if group_stage_id is None:
    raise RuntimeError("Group-Stage-ID nicht gefunden.")

group_matches = (
    matches
    .filter(pl.col("stage_id") == group_stage_id)
    .sort("match_number")
    .to_dicts()
)

ko_matches = (
    matches
    .filter(pl.col("stage_id") != group_stage_id)
    .sort("match_number")
    .to_dicts()
)

all_teams = sorted(team_to_group.keys())

print("Teams:", len(all_teams))
print("Group matches:", len(group_matches))
print("KO matches:", len(ko_matches))

# 4. Aktive Match-Wahrscheinlichkeiten

HOST_COUNTRY_TO_TEAM = {
    "United States": "USA",
    "USA": "USA",
    "Mexico": "Mexico",
    "Canada": "Canada",
}

def parse_match_date(x):
    return date.fromisoformat(str(x)[:10])

def outcome_probs_from_grid(grid):
    p_home = float(np.tril(grid, -1).sum())
    p_draw = float(np.trace(grid))
    p_away = float(np.triu(grid, 1).sum())

    probs = np.array([p_home, p_draw, p_away], dtype=float)
    probs = np.clip(probs, 1e-9, 1.0)
    probs = probs / probs.sum()

    return float(probs[0]), float(probs[1]), float(probs[2])

dist_cache = {}

def _copy_grid_as_float(grid):
    g = np.asarray(grid, dtype=float).copy()
    g = np.clip(g, 0.0, 1.0)
    g = g / g.sum()
    return g

def predict_match_distribution(home, away, match_date, country):
    """
    Aktive Modell-Version:
    Gibt eine komplette kalibrierte Score-Verteilung in Home/Away-Reihenfolge zurück.

    Wichtig:
    - Neutral: predict_neutral(home, away) -> score_grid ist home/away aus Sicht der Argumente.
    - Host home: predict_match(home, away, neutral=False)
    - Host away: predict_match(away, home, neutral=False), dann Grid transponieren.
    """
    key = (home, away, str(match_date), country, str(globals().get("ACTIVE_MODEL_NAME", "active_model")), str(globals().get("PLAYED_MATCHES_VERSION", "no_played")))
    if key in dist_cache:
        return dist_cache[key]

    host_team = HOST_COUNTRY_TO_TEAM.get(country)

    if host_team == home:
        pred = predict_match(
            home,
            away,
            match_date,
            country=country,
            neutral=False,
            max_goals=MAX_GOALS,
        )

        grid = _copy_grid_as_float(pred["score_grid"])
        home_xg = float(pred["home_xg"])
        away_xg = float(pred["away_xg"])

    elif host_team == away:
        pred_reversed = predict_match(
            away,
            home,
            match_date,
            country=country,
            neutral=False,
            max_goals=MAX_GOALS,
        )

        # pred_reversed Grid ist away/home. Wir brauchen home/away.
        grid = _copy_grid_as_float(pred_reversed["score_grid"]).T
        home_xg = float(pred_reversed["away_xg"])
        away_xg = float(pred_reversed["home_xg"])

    else:
        pred = predict_neutral(
            home,
            away,
            match_date,
            country=country,
            max_goals=MAX_GOALS,
        )

        # predict_neutral gibt Grid in team/opponent-Reihenfolge zurück,
        # also hier home/away.
        grid = _copy_grid_as_float(pred["score_grid"])
        home_xg = float(pred["team_xg_pred"])
        away_xg = float(pred["opponent_xg_pred"])

    p_home_win, p_draw, p_away_win = outcome_probs_from_grid(grid)

    top_i, top_j = np.unravel_index(np.argmax(grid), grid.shape)

    dist = {
        "home": home,
        "away": away,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "grid": grid,
        "flat": grid.reshape(-1),
        "p_home_win": p_home_win,
        "p_draw": p_draw,
        "p_away_win": p_away_win,
        "most_likely_score": f"{top_i}:{top_j}",
        "most_likely_score_prob": float(grid[top_i, top_j]),
    }

    dist_cache[key] = dist
    return dist

def sample_score_from_dist(dist):
    grid = dist["grid"]
    flat = dist["flat"]

    idx = rng.choice(flat.size, p=flat)
    h = idx // grid.shape[1]
    a = idx % grid.shape[1]

    return int(h), int(a)

def simulate_match(home, away, match_date, country, knockout=False):
    dist = predict_match_distribution(home, away, match_date, country)
    hs, aw = sample_score_from_dist(dist)

    tiebreak = ""

    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        if knockout:
            # Bei KO-Remis: Elfmeterschießen.
            # Stärkeindikator: kalibrierte non-draw Win-Wahrscheinlichkeiten.
            p_home = dist["p_home_win"]
            p_away = dist["p_away_win"]

            if p_home + p_away <= 0:
                pen_home_prob = 0.5
            else:
                pen_home_prob = p_home / (p_home + p_away)

            if rng.random() < pen_home_prob:
                winner, loser = home, away
            else:
                winner, loser = away, home

            tiebreak = " n.E."
        else:
            winner, loser = None, None

    return {
        "home": home,
        "away": away,
        "home_score": hs,
        "away_score": aw,
        "score": f"{hs}:{aw}{tiebreak}",
        "winner": winner,
        "loser": loser,
        "home_xg": dist["home_xg"],
        "away_xg": dist["away_xg"],
        "p_home_win": dist["p_home_win"],
        "p_draw": dist["p_draw"],
        "p_away_win": dist["p_away_win"],
        "most_likely_score": dist["most_likely_score"],
        "most_likely_score_prob": dist["most_likely_score_prob"],
    }

# 5. Gruppenphase

def empty_table_row(team):
    return {
        "team": team,
        "group": team_to_group[team],
        "mp": 0,
        "w": 0,
        "d": 0,
        "l": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "pts": 0,
    }

def add_group_result(table, home, away, hs, aw):
    table[home]["mp"] += 1
    table[away]["mp"] += 1

    table[home]["gf"] += hs
    table[home]["ga"] += aw
    table[away]["gf"] += aw
    table[away]["ga"] += hs

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if hs > aw:
        table[home]["w"] += 1
        table[away]["l"] += 1
        table[home]["pts"] += 3
    elif aw > hs:
        table[away]["w"] += 1
        table[home]["l"] += 1
        table[away]["pts"] += 3
    else:
        table[home]["d"] += 1
        table[away]["d"] += 1
        table[home]["pts"] += 1
        table[away]["pts"] += 1

def rank_group_rows(rows):
    # Vereinfachter Tie-break:
    # Punkte, Tordifferenz, Tore, Siege, dann Random-Tiebreak.
    return sorted(
        rows,
        key=lambda r: (
            r["pts"],
            r["gd"],
            r["gf"],
            r["w"],
            rng.random(),
        ),
        reverse=True
    )

# 6. Third-place Slots

def split_match_label(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

third_slot_pattern = re.compile(r"^3([A-L]+)$")

def collect_third_slots():
    slots = []
    for m in ko_matches:
        parts = split_match_label(m["match_label"])
        for p in parts:
            if third_slot_pattern.fullmatch(p):
                slots.append(p)

    seen = set()
    out = []
    for s in slots:
        if s not in seen:
            seen.add(s)
            out.append(s)

    return out

THIRD_SLOTS = collect_third_slots()
print("Third-place slots:", THIRD_SLOTS)

def allocate_third_place_slots(best_thirds):
    third_by_group = {r["group"]: r for r in best_thirds}
    available_groups = set(third_by_group.keys())
    third_rank_order = [r["group"] for r in best_thirds]

    allowed = {
        slot: set(slot[1:])
        for slot in THIRD_SLOTS
    }

    ordered_slots = sorted(
        THIRD_SLOTS,
        key=lambda s: len(allowed[s] & available_groups)
    )

    def rec(i, remaining, assignment):
        if i == len(ordered_slots):
            return assignment

        slot = ordered_slots[i]
        candidates = [
            g for g in third_rank_order
            if g in remaining and g in allowed[slot]
        ]

        for g in candidates:
            new_assignment = dict(assignment)
            new_assignment[slot] = third_by_group[g]["team"]
            result = rec(i + 1, remaining - {g}, new_assignment)
            if result is not None:
                return result

        return None

    assignment = rec(0, available_groups, {})

    if assignment is None:
        raise RuntimeError(
            f"Kann Third-place Slots nicht zuordnen. "
            f"Best third groups: {sorted(available_groups)}, Slots: {THIRD_SLOTS}"
        )

    return assignment

# 7. KO Resolver

def resolve_slot(slot, qualifiers, match_results):
    slot = str(slot).strip()

    if re.fullmatch(r"[12][A-L]", slot):
        return qualifiers[slot]

    if third_slot_pattern.fullmatch(slot):
        return qualifiers[slot]

    if re.fullmatch(r"W\d+", slot):
        num = int(slot[1:])
        return match_results[num]["winner"]

    if re.fullmatch(r"RU\d+", slot):
        num = int(slot[2:])
        return match_results[num]["loser"]

    raise RuntimeError(f"Unbekannter KO-Slot: {slot}")

# 8. Ein Turnier simulieren

def simulate_tournament_once():
    table = {team: empty_table_row(team) for team in all_teams}
    group_match_records = []

    for m in group_matches:
        home = team_id_to_name[int(m["home_team_id"])]
        away = team_id_to_name[int(m["away_team_id"])]
        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]

        fixed = get_played_match_result(int(m["match_number"]), home, away) if "get_played_match_result" in globals() else None
        if fixed is not None:
            res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_match_distribution)
        else:
            res = simulate_match(home, away, d, country, knockout=False)

        add_group_result(table, home, away, res["home_score"], res["away_score"])

        group_match_records.append({
            "match_number": int(m["match_number"]),
            "group": team_to_group[home],
            "home": home,
            "away": away,
            "score": res["score"],
            "status": res.get("status", "Prognose"),
        })

    group_rankings = {}
    qualifiers = {}

    for g in sorted(set(team_to_group.values())):
        rows = [dict(v) for v in table.values() if v["group"] == g]
        ranked = rank_group_rows(rows)
        group_rankings[g] = ranked

        qualifiers[f"1{g}"] = ranked[0]["team"]
        qualifiers[f"2{g}"] = ranked[1]["team"]

    thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
    best_thirds = rank_group_rows(thirds)[:8]
    third_assignment = allocate_third_place_slots(best_thirds)
    qualifiers.update(third_assignment)

    round32_teams = set()
    for key, team in qualifiers.items():
        if re.fullmatch(r"[123][A-L]+", key):
            round32_teams.add(team)

    match_results = {}
    knockout_records = []

    for m in ko_matches:
        mn = int(m["match_number"])
        stage = stage_id_to_name[int(m["stage_id"])]
        parts = split_match_label(m["match_label"])

        if len(parts) != 2:
            raise RuntimeError(f"Kann match_label nicht splitten: {m['match_label']}")

        home = resolve_slot(parts[0], qualifiers, match_results)
        away = resolve_slot(parts[1], qualifiers, match_results)

        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]

        res = simulate_match(home, away, d, country, knockout=True)
        match_results[mn] = res

        knockout_records.append({
            "match_number": mn,
            "stage": stage,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": res["winner"],
            "loser": res["loser"],
        })

    final_match = None
    third_place_match = None

    for rec in knockout_records:
        stage_lower = rec["stage"].lower()
        if stage_lower == "final":
            final_match = rec
        elif "third" in stage_lower:
            third_place_match = rec

    if final_match is None:
        raise RuntimeError("Finale nicht gefunden.")

    gold = final_match["winner"]
    silver = final_match["loser"]

    if third_place_match is not None:
        bronze = third_place_match["winner"]
        fourth = third_place_match["loser"]
    else:
        bronze = None
        fourth = None

    finish_status = {team: "Group Stage" for team in all_teams}
    exit_opponent = {team: None for team in all_teams}

    for team in round32_teams:
        finish_status[team] = "Round of 32"

    for rec in knockout_records:
        loser = rec["loser"]
        winner = rec["winner"]
        stage = rec["stage"]

        if stage == "Final":
            finish_status[loser] = "Final / Silver"
            exit_opponent[loser] = winner
        elif "Third" in stage:
            finish_status[loser] = "Fourth Place"
            exit_opponent[loser] = winner
            finish_status[winner] = "Third Place"
        else:
            finish_status[loser] = stage
            exit_opponent[loser] = winner

    finish_status[gold] = "Champion"
    exit_opponent[gold] = None

    return {
        "gold": gold,
        "silver": silver,
        "bronze": bronze,
        "fourth": fourth,
        "round32_teams": round32_teams,
        "group_winners": {g: rows[0]["team"] for g, rows in group_rankings.items()},
        "group_runners_up": {g: rows[1]["team"] for g, rows in group_rankings.items()},
        "best_thirds": [r["team"] for r in best_thirds],
        "finish_status": finish_status,
        "exit_opponent": exit_opponent,
        "final_pair": tuple(sorted([gold, silver])),
    }

# 9. Monte Carlo

title_counts = Counter()
silver_counts = Counter()
bronze_counts = Counter()
fourth_counts = Counter()
round32_counts = Counter()
group_winner_counts = Counter()
best_third_counts = Counter()
finish_counts = defaultdict(Counter)
exit_vs_counts = defaultdict(Counter)
final_pair_counts = Counter()

start = time.time()

for i in range(1, N_SIMS + 1):
    sim = simulate_tournament_once()

    title_counts[sim["gold"]] += 1
    silver_counts[sim["silver"]] += 1

    if sim["bronze"] is not None:
        bronze_counts[sim["bronze"]] += 1

    if sim["fourth"] is not None:
        fourth_counts[sim["fourth"]] += 1

    final_pair_counts[sim["final_pair"]] += 1

    for t in sim["round32_teams"]:
        round32_counts[t] += 1

    for g, t in sim["group_winners"].items():
        group_winner_counts[t] += 1

    for t in sim["best_thirds"]:
        best_third_counts[t] += 1

    for t, status in sim["finish_status"].items():
        finish_counts[t][status] += 1

    for t, opp in sim["exit_opponent"].items():
        if opp is not None:
            exit_vs_counts[t][opp] += 1

    if i % max(1, N_SIMS // 10) == 0:
        elapsed = time.time() - start
        print(f"{i:,}/{N_SIMS:,} Simulationen fertig | {elapsed:.1f}s | Cache: {len(dist_cache)} Matchups")

elapsed = time.time() - start
print(f"\nFertig: {N_SIMS:,} Simulationen in {elapsed:.1f}s")
print("Unique predicted matchups in cache:", len(dist_cache))

# 10. Ergebnis-Tabellen

def pct(x):
    return round(100 * x / N_SIMS, 3)

title_rows = []
for team in all_teams:
    title_rows.append({
        "team": team,
        "gruppe": team_to_group[team],
        "weltmeister_%": pct(title_counts[team]),
        "finale_%": pct(title_counts[team] + silver_counts[team]),
        "halbfinale_medal_zone_%": pct(
            title_counts[team]
            + silver_counts[team]
            + bronze_counts[team]
            + fourth_counts[team]
        ),
        "16telfinale_%": pct(round32_counts[team]),
        "gruppensieger_%": pct(group_winner_counts[team]),
        "bester_dritter_%": pct(best_third_counts[team]),
    })

title_board = (
    pl.DataFrame(title_rows)
    .sort(["weltmeister_%", "finale_%", "16telfinale_%"], descending=True)
)

likely_round32_board = (
    title_board
    .sort(["16telfinale_%", "weltmeister_%"], descending=True)
    .head(32)
)

medal_rows = []
for team in all_teams:
    medal_rows.append({
        "team": team,
        "gold_%": pct(title_counts[team]),
        "silber_%": pct(silver_counts[team]),
        "bronze_%": pct(bronze_counts[team]),
        "vierter_%": pct(fourth_counts[team]),
    })

medal_board = (
    pl.DataFrame(medal_rows)
    .filter(
        (pl.col("gold_%") > 0)
        | (pl.col("silber_%") > 0)
        | (pl.col("bronze_%") > 0)
        | (pl.col("vierter_%") > 0)
    )
    .sort(["gold_%", "silber_%", "bronze_%"], descending=True)
)

final_pair_rows = []
for pair, c in final_pair_counts.most_common(20):
    final_pair_rows.append({
        "finale": f"{pair[0]} vs {pair[1]}",
        "wahrscheinlichkeit_%": pct(c),
    })

final_pair_board = pl.DataFrame(final_pair_rows)

def team_finish_board(team):
    rows = []
    for status, c in finish_counts[team].most_common():
        rows.append({
            "team": team,
            "finish": status,
            "wahrscheinlichkeit_%": pct(c),
        })

    return pl.DataFrame(
        rows,
        schema={
            "team": pl.String,
            "finish": pl.String,
            "wahrscheinlichkeit_%": pl.Float64,
        }
    )

def team_exit_vs_board(team, top_n=12):
    rows = []
    for opp, c in exit_vs_counts[team].most_common(top_n):
        rows.append({
            "team": team,
            "verliert_gegen": opp,
            "wahrscheinlichkeit_%": pct(c),
        })

    return pl.DataFrame(
        rows,
        schema={
            "team": pl.String,
            "verliert_gegen": pl.String,
            "wahrscheinlichkeit_%": pl.Float64,
        }
    )

def most_likely_exit_sentence(team):
    fb = team_finish_board(team)

    if fb.height == 0:
        return f"{team}: keine Finish-Daten gefunden."

    top = fb.row(0, named=True)

    if top["finish"] == "Champion":
        return f"{team}: häufigstes Ergebnis ist Weltmeister ({top['wahrscheinlichkeit_%']}%)."

    ev = team_exit_vs_board(team, top_n=1)

    if ev.height > 0:
        opp = ev.row(0, named=True)
        return (
            f"{team}: häufigstes Finish = {top['finish']} ({top['wahrscheinlichkeit_%']}%). "
            f"Häufigster Exit-Gegner: {opp['verliert_gegen']} ({opp['wahrscheinlichkeit_%']}%)."
        )

    return f"{team}: häufigstes Finish = {top['finish']} ({top['wahrscheinlichkeit_%']}%)."

# 11. Ausgaben

print("\n=== Titelwahrscheinlichkeiten Top 20 ===")
print(title_board.head(20))

print("\n=== Wahrscheinlichste 32 16telfinalisten + Titelchance ===")
print(likely_round32_board)

print("\n=== Medaillenwahrscheinlichkeiten ===")
print(medal_board.head(25))

print("\n=== Häufigste Finals ===")
print(final_pair_board)

print("\n=== USA: Wann fliegen sie raus? ===")
print(team_finish_board("USA"))
print("\nUSA häufigste Exit-Gegner:")
print(team_exit_vs_board("USA"))

print("\n=== Deutschland: Rundenverteilung ===")
print(team_finish_board("Germany"))
print("\nDeutschland häufigste Exit-Gegner:")
print(team_exit_vs_board("Germany"))

print("\n=== Südkorea: Rundenverteilung ===")
print(team_finish_board("South Korea"))
print("\nSüdkorea häufigste Exit-Gegner:")
print(team_exit_vs_board("South Korea"))

print("\n=== Kurzinterpretation ===")
print(most_likely_exit_sentence("USA"))
print(most_likely_exit_sentence("Germany"))
print(most_likely_exit_sentence("South Korea"))


Monte Carlo nutzt Prediction Helper:
predict_match: <function predict_match_fixed_hybrid_scoreline at 0x000001588F70F9C0>
predict_neutral: <function predict_neutral_fixed_hybrid_scoreline at 0x000001588F70F920>
Calibration: {'lambda_scale': 1.1, 'draw_boost': 1.1, 'poisson_weight': 0.75, 'log_loss': 0.8577838343889644}
CSV_DIR: C:\ml\projects\WeltmeisterKI
Teams: 48
Group matches: 72
KO matches: 32
Third-place slots: ['3ABCDF', '3CDFGH', '3CEFHI', '3EHIJK', '3AEHIJ', '3BEFIJ', '3EFGIJ', '3DEIJL']
5,000/50,000 Simulationen fertig | 339.6s | Cache: 3247 Matchups
10,000/50,000 Simulationen fertig | 421.0s | Cache: 3794 Matchups
15,000/50,000 Simulationen fertig | 475.8s | Cache: 4092 Matchups
20,000/50,000 Simulationen fertig | 523.0s | Cache: 4319 Matchups
25,000/50,000 Simulationen fertig | 567.2s | Cache: 4517 Matchups
30,000/50,000 Simulationen fertig | 604.4s | Cache: 4656 Matchups
35,000/50,000 Simulationen fertig | 641.6s | Cache: 4796 Matchups
40,000/50,000 Simulationen fertig | 6

## 48. Aggregierter Turnierbaum

Aggregiert aus vielen Simulationen einen Turnierbaum mit häufigsten Matchups, Scores und Gewinnern pro Spiel.


In [57]:
# Was diese Zelle macht:
# Aggregiert aus vielen Simulationen einen Turnierbaum mit häufigsten Matchups, Scores und Gewinnern pro Spiel.

import re
import time
import numpy as np
import polars as pl
from collections import Counter, defaultdict

# Aggregierter Monte-Carlo-Turnierbaum mit exakten Scores

N_TREE_SIMS = 50_000
TREE_RNG_SEED = 123

# Wichtig: simulate_match/rank_group_rows nutzen global rng.
rng = np.random.default_rng(TREE_RNG_SEED)

pl.Config.set_tbl_cols(40)
pl.Config.set_tbl_rows(140)
pl.Config.set_tbl_width_chars(420)
pl.Config.set_fmt_str_lengths(220)

required_names = [
    "all_teams",
    "group_matches",
    "ko_matches",
    "team_id_to_name",
    "team_to_group",
    "stage_id_to_name",
    "city_id_to_country",
    "city_id_to_city",
    "parse_match_date",
    "simulate_match",
    "empty_table_row",
    "add_group_result",
    "rank_group_rows",
    "allocate_third_place_slots",
    "resolve_slot",
]

missing = [x for x in required_names if x not in globals()]
if missing:
    raise RuntimeError(
        "Diese Zelle braucht erst die große Monte-Carlo-Zelle davor. "
        f"Fehlt aktuell: {missing}"
    )

def pct_tree(x):
    return round(100 * x / N_TREE_SIMS, 3)

def split_match_label_tree(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

def result_winner_label(res):
    if res["winner"] is None:
        return "Draw"
    return res["winner"]

def pretty_result(score, winner):
    if winner == "Draw":
        return f"{score} Remis"
    return f"{winner} gewinnt {score}"

def simulate_tournament_once_with_records():
    table = {team: empty_table_row(team) for team in all_teams}
    group_records = []

    # Gruppenphase
    for m in group_matches:
        mn = int(m["match_number"])
        home = team_id_to_name[int(m["home_team_id"])]
        away = team_id_to_name[int(m["away_team_id"])]
        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]
        city = city_id_to_city[int(m["city_id"])]
        stage = stage_id_to_name[int(m["stage_id"])]

        fixed = get_played_match_result(mn, home, away) if "get_played_match_result" in globals() else None
        if fixed is not None:
            res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_match_distribution)
        else:
            res = simulate_match(home, away, d, country, knockout=False)

        add_group_result(table, home, away, res["home_score"], res["away_score"])

        winner = result_winner_label(res)

        group_records.append({
            "match_number": mn,
            "stage": stage,
            "label": str(m["match_label"]),
            "date": d,
            "city": city,
            "country": country,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": winner,
            "home_xg": res["home_xg"],
            "away_xg": res["away_xg"],
            "status": res.get("status", "Prognose"),
        })

    # Gruppentabellen / Qualifikation
    group_rankings = {}
    qualifiers = {}

    for g in sorted(set(team_to_group.values())):
        rows = [dict(v) for v in table.values() if v["group"] == g]
        ranked = rank_group_rows(rows)
        group_rankings[g] = ranked

        qualifiers[f"1{g}"] = ranked[0]["team"]
        qualifiers[f"2{g}"] = ranked[1]["team"]

    thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
    best_thirds = rank_group_rows(thirds)[:8]

    third_assignment = allocate_third_place_slots(best_thirds)
    qualifiers.update(third_assignment)

    round32_teams = set()
    for key, team in qualifiers.items():
        if re.fullmatch(r"[123][A-L]+", key):
            round32_teams.add(team)

    # KO-Phase
    match_results = {}
    ko_records = []

    for m in ko_matches:
        mn = int(m["match_number"])
        stage = stage_id_to_name[int(m["stage_id"])]
        label = str(m["match_label"])
        parts = split_match_label_tree(label)

        if len(parts) != 2:
            raise RuntimeError(f"Kann match_label nicht splitten: {label}")

        home = resolve_slot(parts[0], qualifiers, match_results)
        away = resolve_slot(parts[1], qualifiers, match_results)

        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]
        city = city_id_to_city[int(m["city_id"])]

        res = simulate_match(home, away, d, country, knockout=True)
        match_results[mn] = res

        winner = result_winner_label(res)

        ko_records.append({
            "match_number": mn,
            "stage": stage,
            "label": label,
            "date": d,
            "city": city,
            "country": country,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": winner,
            "loser": res["loser"],
            "home_xg": res["home_xg"],
            "away_xg": res["away_xg"],
        })

    final_rec = next((r for r in ko_records if str(r["stage"]).lower() == "final"), None)
    third_rec = next((r for r in ko_records if "third" in str(r["stage"]).lower()), None)

    gold = final_rec["winner"]
    silver = final_rec["loser"]
    bronze = third_rec["winner"] if third_rec else None
    fourth = third_rec["loser"] if third_rec else None

    return {
        "group_records": group_records,
        "ko_records": ko_records,
        "round32_teams": round32_teams,
        "gold": gold,
        "silver": silver,
        "bronze": bronze,
        "fourth": fourth,
    }

# Simulationen aggregieren

match_meta = {}
matchup_counts = defaultdict(Counter)
result_counts = defaultdict(Counter)
winner_counts = defaultdict(Counter)
score_counts = defaultdict(Counter)

title_counts_tree = Counter()
silver_counts_tree = Counter()
bronze_counts_tree = Counter()
fourth_counts_tree = Counter()
round32_counts_tree = Counter()

start = time.time()

for i in range(1, N_TREE_SIMS + 1):
    sim = simulate_tournament_once_with_records()

    title_counts_tree[sim["gold"]] += 1
    silver_counts_tree[sim["silver"]] += 1
    if sim["bronze"] is not None:
        bronze_counts_tree[sim["bronze"]] += 1
    if sim["fourth"] is not None:
        fourth_counts_tree[sim["fourth"]] += 1

    for t in sim["round32_teams"]:
        round32_counts_tree[t] += 1

    for rec in sim["group_records"] + sim["ko_records"]:
        mn = rec["match_number"]
        home = rec["home"]
        away = rec["away"]
        score = rec["score"]
        winner = rec["winner"]

        match_meta[mn] = {
            "match_number": mn,
            "stage": rec["stage"],
            "label": rec["label"],
            "date": rec["date"],
            "city": rec["city"],
            "country": rec["country"],
        }

        matchup_counts[mn][(home, away)] += 1
        result_counts[mn][(home, away, score, winner)] += 1
        winner_counts[mn][winner] += 1
        score_counts[mn][score] += 1

    if i % max(1, N_TREE_SIMS // 10) == 0:
        elapsed = time.time() - start
        print(f"{i:,}/{N_TREE_SIMS:,} Simulationen aggregiert | {elapsed:.1f}s")

print(f"\nFertig in {time.time() - start:.1f}s")

# Aggregierten Turnierbaum bauen

tree_rows = []

for mn in sorted(match_meta.keys()):
    meta = match_meta[mn]

    top_matchup, top_matchup_n = matchup_counts[mn].most_common(1)[0]
    top_home, top_away = top_matchup

    # Häufigstes kohärentes Ergebnis innerhalb des häufigsten Matchups.
    result_counter_for_top_matchup = Counter()
    for (h, a, score, winner), c in result_counts[mn].items():
        if h == top_home and a == top_away:
            result_counter_for_top_matchup[(score, winner)] += c

    (top_score_for_matchup, top_result_winner), top_result_n = result_counter_for_top_matchup.most_common(1)[0]

    # Häufigster Gewinner insgesamt, unabhängig vom Matchup.
    top_winner, top_winner_n = winner_counts[mn].most_common(1)[0]

    # Häufigster Score insgesamt, unabhängig vom Matchup.
    top_score_overall, top_score_overall_n = score_counts[mn].most_common(1)[0]

    tree_rows.append({
        "match": mn,
        "stage": meta["stage"],
        "label": meta["label"],
        "date": meta["date"],
        "city": meta["city"],
        "country": meta["country"],

        "häufigster_matchup": f"{top_home} vs {top_away}",
        "matchup_%": round(100 * top_matchup_n / N_TREE_SIMS, 3),

        "häufigstes_resultat": pretty_result(top_score_for_matchup, top_result_winner),
        "resultat_%_wenn_matchup": round(100 * top_result_n / top_matchup_n, 3),
        "resultat_%_gesamt": round(100 * top_result_n / N_TREE_SIMS, 3),

        "häufigster_score_gesamt": top_score_overall,
        "score_gesamt_%": round(100 * top_score_overall_n / N_TREE_SIMS, 3),

        "häufigster_winner_gesamt": top_winner,
        "winner_gesamt_%": round(100 * top_winner_n / N_TREE_SIMS, 3),
    })

full_tree_agg = pl.DataFrame(tree_rows).sort("match")

group_tree_agg = full_tree_agg.filter(pl.col("stage").str.contains("Group"))
ko_tree_agg = full_tree_agg.filter(~pl.col("stage").str.contains("Group"))

# 16telfinalisten + Titelchance aus derselben Aggregation

round32_title_rows = []

for team in all_teams:
    round32_title_rows.append({
        "team": team,
        "gruppe": team_to_group[team],
        "16telfinale_%": pct_tree(round32_counts_tree[team]),
        "weltmeister_%": pct_tree(title_counts_tree[team]),
        "finale_%": pct_tree(title_counts_tree[team] + silver_counts_tree[team]),
        "halbfinale_medal_zone_%": pct_tree(
            title_counts_tree[team]
            + silver_counts_tree[team]
            + bronze_counts_tree[team]
            + fourth_counts_tree[team]
        ),
    })

round32_title_board = (
    pl.DataFrame(round32_title_rows)
    .sort(["16telfinale_%", "weltmeister_%"], descending=True)
)

likely_16telfinalisten_title = round32_title_board.head(32)

# Match-Detail-Helfer

def show_match_details(match_number, top_n=12):
    mn = int(match_number)

    if mn not in match_meta:
        print(f"Match {mn} nicht gefunden.")
        return

    meta = match_meta[mn]
    print(f"Match {mn} | {meta['stage']} | {meta['label']} | {meta['date']} | {meta['city']}, {meta['country']}")

    matchup_rows = []
    for (home, away), c in matchup_counts[mn].most_common(top_n):
        matchup_rows.append({
            "matchup": f"{home} vs {away}",
            "wahrscheinlichkeit_%": round(100 * c / N_TREE_SIMS, 3),
        })

    result_rows = []
    for (home, away, score, winner), c in result_counts[mn].most_common(top_n):
        result_rows.append({
            "matchup": f"{home} vs {away}",
            "resultat": pretty_result(score, winner),
            "wahrscheinlichkeit_%": round(100 * c / N_TREE_SIMS, 3),
        })

    print("\nHäufigste Matchups:")
    print(pl.DataFrame(matchup_rows))

    print("\nHäufigste exakte Resultate:")
    print(pl.DataFrame(result_rows))

# Ausgaben

print("\n=== Alle Gruppenspiele aggregiert ===")
print(
    group_tree_agg.select([
        "match",
        "label",
        "date",
        "city",
        "häufigster_matchup",
        "häufigstes_resultat",
        "resultat_%_wenn_matchup",
        "häufigster_winner_gesamt",
        "winner_gesamt_%",
    ])
)

print("\n=== Aggregierter KO-Turnierbaum ===")

stage_order = [
    "Round of 32",
    "Round of 16",
    "Quarterfinals",
    "Semifinals",
    "Third Place Playoff",
    "Final",
]

for stage in stage_order:
    part = ko_tree_agg.filter(pl.col("stage") == stage)
    if part.height == 0:
        continue

    print(f"\n--- {stage} ---")
    print(
        part.select([
            "match",
            "label",
            "date",
            "city",
            "häufigster_matchup",
            "matchup_%",
            "häufigstes_resultat",
            "resultat_%_wenn_matchup",
            "resultat_%_gesamt",
            "häufigster_winner_gesamt",
            "winner_gesamt_%",
        ])
    )

print("\n=== Wahrscheinlichste 32 16telfinalisten + WM-Titelchance ===")
print(likely_16telfinalisten_title)

print("\n=== Titelwahrscheinlichkeiten Top 20 ===")
print(
    round32_title_board
    .sort(["weltmeister_%", "finale_%"], descending=True)
    .head(20)
)

print("\nTipp: Details zu einem einzelnen Match anzeigen, z.B. Finale:")
print("show_match_details(104)")


5,000/50,000 Simulationen aggregiert | 29.4s
10,000/50,000 Simulationen aggregiert | 59.4s
15,000/50,000 Simulationen aggregiert | 88.8s
20,000/50,000 Simulationen aggregiert | 116.2s
25,000/50,000 Simulationen aggregiert | 143.1s
30,000/50,000 Simulationen aggregiert | 170.3s
35,000/50,000 Simulationen aggregiert | 199.3s
40,000/50,000 Simulationen aggregiert | 226.7s
45,000/50,000 Simulationen aggregiert | 253.9s
50,000/50,000 Simulationen aggregiert | 280.7s

Fertig in 280.7s

=== Alle Gruppenspiele aggregiert ===
shape: (72, 9)
┌───────┬─────────┬────────────┬────────────────────────┬───────────────────────────────────────┬────────────────────────────────────┬─────────────────────────┬──────────────────────────┬─────────────────┐
│ match ┆ label   ┆ date       ┆ city                   ┆ häufigster_matchup                    ┆ häufigstes_resultat                ┆ resultat_%_wenn_matchup ┆ häufigster_winner_gesamt ┆ winner_gesamt_% │
│ ---   ┆ ---     ┆ ---        ┆ ---              

In [58]:
# Was diese Zelle macht:
# Gibt für ein Team aus, wie oft es jede KO-Runde erreicht, dort ausscheidet,
# das Finale erreicht und Weltmeister wird.

import polars as pl
from collections import Counter, defaultdict

TEAM_TO_ANALYZE = "Germany"

required = ["result_counts", "match_meta", "N_TREE_SIMS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Bitte zuerst die Monte-Carlo-Aggregation laufen lassen. Fehlt: {missing}")

STAGE_ORDER = [
    "Round of 32",
    "Round of 16",
    "Quarterfinals",
    "Semifinals",
    "Final",
]

def pct(x):
    return round(100 * x / N_TREE_SIMS, 3)

def stage_key(stage):
    s = str(stage).lower()
    if "round of 32" in s:
        return "Round of 32"
    if "round of 16" in s:
        return "Round of 16"
    if "quarter" in s:
        return "Quarterfinals"
    if "semi" in s:
        return "Semifinals"
    if s == "final" or " final" in s:
        return "Final"
    return None

reached_counts = Counter()
winner_counts_by_stage = Counter()
lost_counts_by_stage = Counter()

for match_no, counter in result_counts.items():
    stage = stage_key(match_meta[match_no]["stage"])
    if stage is None:
        continue

    for (home, away, score, winner), c in counter.items():
        if TEAM_TO_ANALYZE in (home, away):
            reached_counts[stage] += c

            if winner == TEAM_TO_ANALYZE:
                winner_counts_by_stage[stage] += c
            else:
                lost_counts_by_stage[stage] += c

champion_count = winner_counts_by_stage["Final"]
final_lost_count = lost_counts_by_stage["Final"]

# Gruppenphase raus = nicht Round of 32 erreicht
group_exit_count = N_TREE_SIMS - reached_counts["Round of 32"]

rows = []

rows.append({
    "phase": "Gruppenphase überstehen / 16-telfinale erreichen",
    "erreicht_%": pct(reached_counts["Round of 32"]),
    "scheidet_hier_aus_%": pct(group_exit_count),
})

for stage in STAGE_ORDER:
    rows.append({
        "phase": stage,
        "erreicht_%": pct(reached_counts[stage]),
        "scheidet_hier_aus_%": pct(lost_counts_by_stage[stage]),
    })

rows.append({
    "phase": "Weltmeister",
    "erreicht_%": pct(champion_count),
    "scheidet_hier_aus_%": 0.0,
})

team_path_summary = pl.DataFrame(rows)

print(f"=== Turnierchancen: {TEAM_TO_ANALYZE} ===")
print(team_path_summary)

print("\nKurzfassung:")
print(f"16-telfinale erreichen: {pct(reached_counts['Round of 32'])}%")
print(f"Achtelfinale erreichen: {pct(reached_counts['Round of 16'])}%")
print(f"Viertelfinale erreichen: {pct(reached_counts['Quarterfinals'])}%")
print(f"Halbfinale erreichen: {pct(reached_counts['Semifinals'])}%")
print(f"Finale erreichen: {pct(reached_counts['Final'])}%")
print(f"Weltmeister werden: {pct(champion_count)}%")


=== Turnierchancen: Germany ===
shape: (7, 3)
┌──────────────────────────────────────────────────┬────────────┬─────────────────────┐
│ phase                                            ┆ erreicht_% ┆ scheidet_hier_aus_% │
│ ---                                              ┆ ---        ┆ ---                 │
│ str                                              ┆ f64        ┆ f64                 │
╞══════════════════════════════════════════════════╪════════════╪═════════════════════╡
│ Gruppenphase überstehen / 16-telfinale erreichen ┆ 100.0      ┆ 0.0                 │
│ Round of 32                                      ┆ 100.0      ┆ 18.452              │
│ Round of 16                                      ┆ 81.548     ┆ 20.818              │
│ Quarterfinals                                    ┆ 60.73      ┆ 24.798              │
│ Semifinals                                       ┆ 35.932     ┆ 15.024              │
│ Final                                            ┆ 20.908     ┆ 10.296  

## 51. Exakte KO-Ausscheidungen

Analysiert für Deutschland und Südkorea, in welcher KO-Runde und mit welchem konkreten Endstand sie am häufigsten ausscheiden.


In [59]:
# Was diese Zelle macht:
# Analysiert für Deutschland und Südkorea, in welcher KO-Runde und mit welchem konkreten Endstand sie am häufigsten ausscheiden.

import re
import polars as pl
from collections import Counter, defaultdict

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_width_chars(340)
pl.Config.set_fmt_str_lengths(180)

TARGET_TEAMS = ["Germany", "South Korea"]

required = ["result_counts", "match_meta", "N_TREE_SIMS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Diese Zelle muss nach der aggregierten Turnierbaum-Zelle laufen. "
        f"Fehlt aktuell: {missing}"
    )

def parse_score_team_view(score, home, away, team):
    """
    Wandelt Home/Away-Score in Team-Sicht um.
    Beispiel:
    home=France, away=Germany, score=2:1, team=Germany -> 1:2
    """
    score_str = str(score)
    pen = "n.E." in score_str

    base = score_str.replace(" n.E.", "").strip()
    hg, ag = map(int, base.split(":"))

    if team == home:
        tg, og = hg, ag
    elif team == away:
        tg, og = ag, hg
    else:
        raise ValueError(f"{team} ist nicht in Match: {home} vs {away}")

    out = f"{tg}:{og}"
    if pen:
        out += " n.E."
    return out

def is_title_exit_stage(stage):
    s = str(stage).lower()
    if "group" in s:
        return False
    if "third" in s:
        return False
    return True

exit_exact_counts = {team: Counter() for team in TARGET_TEAMS}
exit_score_counts = {team: Counter() for team in TARGET_TEAMS}
exit_stage_counts = {team: Counter() for team in TARGET_TEAMS}
exit_stage_score_counts = {team: Counter() for team in TARGET_TEAMS}

for mn, counter in result_counts.items():
    meta = match_meta[mn]
    stage = meta["stage"]

    if not is_title_exit_stage(stage):
        continue

    for (home, away, score, winner), c in counter.items():
        for team in TARGET_TEAMS:
            if team not in (home, away):
                continue

            # In KO-Spielen heißt: Team war dabei und hat nicht gewonnen -> Titel-Aus.
            if winner == team:
                continue

            opponent = away if team == home else home
            team_score = parse_score_team_view(score, home, away, team)

            key = (stage, opponent, team_score)
            exit_exact_counts[team][key] += c
            exit_score_counts[team][team_score] += c
            exit_stage_counts[team][stage] += c
            exit_stage_score_counts[team][(stage, team_score)] += c

def pct(x, denom=N_TREE_SIMS):
    return round(100 * x / denom, 3) if denom else 0.0

overview_rows = []
exact_rows = []
score_rows = []
stage_score_rows = []

for team in TARGET_TEAMS:
    ko_exit_n = sum(exit_exact_counts[team].values())

    round32_n = round32_counts_tree[team] if "round32_counts_tree" in globals() else None
    champion_n = title_counts_tree[team] if "title_counts_tree" in globals() else None
    group_stage_n = N_TREE_SIMS - round32_n if round32_n is not None else None

    overview_rows.append({
        "team": team,
        "gruppenphase_raus_%": pct(group_stage_n) if group_stage_n is not None else None,
        "ko_ausscheiden_mit_score_%": pct(ko_exit_n),
        "weltmeister_%": pct(champion_n) if champion_n is not None else None,
        "ko_exit_count": ko_exit_n,
    })

    for (stage, opponent, team_score), c in exit_exact_counts[team].most_common(20):
        exact_rows.append({
            "team": team,
            "stage": stage,
            "gegner": opponent,
            "team_score": team_score,
            "lesart": f"{team} verliert {team_score} gegen {opponent}",
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

    for team_score, c in exit_score_counts[team].most_common(12):
        score_rows.append({
            "team": team,
            "team_score": team_score,
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

    for (stage, team_score), c in exit_stage_score_counts[team].most_common(20):
        stage_score_rows.append({
            "team": team,
            "stage": stage,
            "team_score": team_score,
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

exit_overview = pl.DataFrame(overview_rows)
exit_exact_top = pl.DataFrame(exact_rows)
exit_score_top = pl.DataFrame(score_rows)
exit_stage_score_top = pl.DataFrame(stage_score_rows)

print("=== Übersicht ===")
print(exit_overview)

print("\n=== Häufigste konkrete KO-Ausscheidungen mit Gegner + Endstand ===")
print(exit_exact_top)

print("\n=== Häufigste reine Ausscheidungs-Endstände, egal gegen wen ===")
print(exit_score_top)

print("\n=== Häufigste Ausscheidungs-Endstände je Runde ===")
print(exit_stage_score_top)


=== Übersicht ===
shape: (2, 5)
┌─────────────┬─────────────────────┬────────────────────────────┬───────────────┬───────────────┐
│ team        ┆ gruppenphase_raus_% ┆ ko_ausscheiden_mit_score_% ┆ weltmeister_% ┆ ko_exit_count │
│ ---         ┆ ---                 ┆ ---                        ┆ ---           ┆ ---           │
│ str         ┆ f64                 ┆ f64                        ┆ f64           ┆ i64           │
╞═════════════╪═════════════════════╪════════════════════════════╪═══════════════╪═══════════════╡
│ Germany     ┆ 0.0                 ┆ 89.388                     ┆ 10.612        ┆ 44694         │
│ South Korea ┆ 3.578               ┆ 96.29                      ┆ 0.132         ┆ 48145         │
└─────────────┴─────────────────────┴────────────────────────────┴───────────────┴───────────────┘

=== Häufigste konkrete KO-Ausscheidungen mit Gegner + Endstand ===
shape: (40, 7)
┌─────────────┬───────────────┬─────────────┬────────────┬───────────────────────────────────

In [61]:
# Deutschland Turnierfortschritt: Erreichen vs Ausscheiden je Runde
# Diese Zelle zeigt, wie wahrscheinlich Deutschland bestimmte Runden erreicht
# und in welcher Runde Deutschland ausscheidet.
# Muss nach Monte Carlo + aggregiertem Turnierbaum laufen.

import polars as pl
from collections import Counter

TEAM = "Germany"

required = [
    "N_TREE_SIMS",
    "round32_counts_tree",
    "title_counts_tree",
    "silver_counts_tree",
    "bronze_counts_tree",
    "fourth_counts_tree",
    "exit_stage_counts",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Diese Zelle muss nach der KO-Ausscheidungs-Zelle laufen. "
        f"Fehlt aktuell: {missing}"
    )

def pct_de(x):
    return round(100 * x / N_TREE_SIMS, 3) if N_TREE_SIMS else 0.0

# exit_stage_counts[TEAM] enthält nur KO-Aus, nicht Gruppenphase.
team_exit_stages = exit_stage_counts[TEAM]

# Ausschiede nach Runde
out_r32 = team_exit_stages.get("Round of 32", 0)
out_r16 = team_exit_stages.get("Round of 16", 0)
out_qf = team_exit_stages.get("Quarterfinals", 0)
out_sf = team_exit_stages.get("Semifinals", 0)

# Finale / Platzierung
champion = title_counts_tree[TEAM]
runner_up = silver_counts_tree[TEAM]
third_place = bronze_counts_tree[TEAM]
fourth_place = fourth_counts_tree[TEAM]

# Achtung:
# "erreicht Round of 32" = kommt aus der Gruppe raus.
# "erreicht Round of 16" = gewinnt Round of 32.
# "erreicht Quarterfinals" = gewinnt Round of 16.
# "erreicht Semifinals" = gewinnt Quarterfinals.
# "erreicht Final/Third Place zone" = gewinnt Semifinal? Nein:
# Nach Halbfinale landet man im Finale oder Spiel um Platz 3.

reaches_r32 = round32_counts_tree[TEAM]
group_out = N_TREE_SIMS - reaches_r32

reaches_r16 = reaches_r32 - out_r32
reaches_qf = reaches_r16 - out_r16
reaches_sf = reaches_qf - out_qf

# Wer ins Halbfinale kommt, beendet das Turnier als Champion/Silver/Bronze/Fourth.
# Halbfinale verlieren bedeutet danach Spiel um Platz 3; "scheidet im Halbfinale aus"
# ist sportlich korrekt: Titelchance endet im Halbfinale.
title_ends_in_sf = out_sf

reaches_final = champion + runner_up
reaches_third_place_game = third_place + fourth_place

progress_rows = [
    {
        "runde": "Gruppenphase",
        "erreicht_%": 100.0,
        "scheidet_dort_aus_%": pct_de(group_out),
        "lesart": "Deutschland ist sicher in der Gruppenphase; diese Spalte zeigt Gruppen-Aus.",
    },
    {
        "runde": "16-tel-Finale / Round of 32",
        "erreicht_%": pct_de(reaches_r32),
        "scheidet_dort_aus_%": pct_de(out_r32),
        "lesart": "Kommt aus der Gruppe raus; verliert dort ggf. direkt im 16-tel-Finale.",
    },
    {
        "runde": "Achtelfinale / Round of 16",
        "erreicht_%": pct_de(reaches_r16),
        "scheidet_dort_aus_%": pct_de(out_r16),
        "lesart": "Gewinnt das 16-tel-Finale; verliert dort ggf. im Achtelfinale.",
    },
    {
        "runde": "Viertelfinale",
        "erreicht_%": pct_de(reaches_qf),
        "scheidet_dort_aus_%": pct_de(out_qf),
        "lesart": "Gewinnt das Achtelfinale; verliert dort ggf. im Viertelfinale.",
    },
    {
        "runde": "Halbfinale",
        "erreicht_%": pct_de(reaches_sf),
        "scheidet_dort_aus_%": pct_de(title_ends_in_sf),
        "lesart": "Gewinnt das Viertelfinale; Titelchance endet ggf. im Halbfinale.",
    },
    {
        "runde": "Finale",
        "erreicht_%": pct_de(reaches_final),
        "scheidet_dort_aus_%": pct_de(runner_up),
        "lesart": "Gewinnt das Halbfinale; verliert dort ggf. das Finale.",
    },
    {
        "runde": "Weltmeister",
        "erreicht_%": pct_de(champion),
        "scheidet_dort_aus_%": 0.0,
        "lesart": "Deutschland gewinnt das Turnier.",
    },
]

progress_deutschland = pl.DataFrame(progress_rows)

print("=== Deutschland: Runden erreichen und dort ausscheiden ===")
print(progress_deutschland)

print("\n=== Kurzfassung ===")
for r in progress_deutschland.iter_rows(named=True):
    print(
        f"{r['runde']}: erreicht {r['erreicht_%']}% | "
        f"scheidet dort aus {r['scheidet_dort_aus_%']}%"
    )


=== Deutschland: Runden erreichen und dort ausscheiden ===
shape: (7, 4)
┌─────────────────────────────┬────────────┬─────────────────────┬─────────────────────────────────────────────────────────────────────────────┐
│ runde                       ┆ erreicht_% ┆ scheidet_dort_aus_% ┆ lesart                                                                      │
│ ---                         ┆ ---        ┆ ---                 ┆ ---                                                                         │
│ str                         ┆ f64        ┆ f64                 ┆ str                                                                         │
╞═════════════════════════════╪════════════╪═════════════════════╪═════════════════════════════════════════════════════════════════════════════╡
│ Gruppenphase                ┆ 100.0      ┆ 0.0                 ┆ Deutschland ist sicher in der Gruppenphase; diese Spalte zeigt Gruppen-Aus. │
│ 16-tel-Finale / Round of 32 ┆ 100.0      ┆ 18.452      

In [62]:
# Zelle XX: Word-Export für Einzelbaum und aggregierten Monte-Carlo-Baum
# Erstellt ein Word-Dokument mit Einzel-Turnierbaum, aggregiertem Monte-Carlo-Turnierbaum,
# Titelchancen, 16telfinalisten und wichtigen Zusatzstatistiken.

from pathlib import Path
from datetime import datetime
import subprocess
import sys
import math
import numpy as np

try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.section import WD_ORIENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "python-docx"], check=True)
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.section import WD_ORIENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT

try:
    import polars as pl
except Exception:
    pl = None

EXPORT_DIR = Path.cwd()
DOCX_PATH = EXPORT_DIR / f"WeltmeisterKI4_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"

def to_polars_df(obj):
    if obj is None:
        return None
    if pl is not None and isinstance(obj, pl.DataFrame):
        return obj
    try:
        return pl.DataFrame(obj)
    except Exception:
        return None

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def clean_value(x):
    if x is None:
        return ""
    try:
        if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
            return ""
    except Exception:
        pass
    if isinstance(x, (list, tuple, set)):
        return ", ".join(str(v) for v in x)
    return str(x)

def compact_df(df, preferred_cols=None, max_rows=None):
    df = to_polars_df(df)
    if df is None or df.height == 0:
        return None

    if preferred_cols:
        cols = existing_cols(df, preferred_cols)
        if cols:
            df = df.select(cols)

    if max_rows is not None and df.height > max_rows:
        df = df.head(max_rows)

    return df

def add_table_from_df(doc, title, df, preferred_cols=None, max_rows=None, note=None):
    df = compact_df(df, preferred_cols=preferred_cols, max_rows=max_rows)

    doc.add_heading(title, level=2)

    if note:
        p = doc.add_paragraph(note)
        p.style = "Intense Quote"

    if df is None or df.height == 0:
        doc.add_paragraph("Keine Daten vorhanden. Bitte die passende Vorzelle ausführen.")
        return

    table = doc.add_table(rows=1, cols=len(df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"

    hdr = table.rows[0].cells
    for j, col in enumerate(df.columns):
        hdr[j].text = str(col)
        hdr[j].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
        for run in hdr[j].paragraphs[0].runs:
            run.bold = True

    for row in df.iter_rows(named=True):
        cells = table.add_row().cells
        for j, col in enumerate(df.columns):
            cells[j].text = clean_value(row.get(col))
            cells[j].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

    doc.add_paragraph("")

def counter_to_prob_df(counter, total, name_col, pct_col, top_n=25):
    if counter is None or total in (None, 0):
        return None
    rows = []
    for name, count in counter.most_common(top_n):
        rows.append({
            name_col: name,
            "count": int(count),
            pct_col: round(100 * count / total, 3),
        })
    return pl.DataFrame(rows) if rows else None

def build_medal_prob_df():
    if not all(x in globals() for x in ["title_counts_tree", "silver_counts_tree", "bronze_counts_tree", "fourth_counts_tree", "N_TREE_SIMS"]):
        return None

    teams = set()
    for c in [title_counts_tree, silver_counts_tree, bronze_counts_tree, fourth_counts_tree]:
        teams.update(c.keys())

    rows = []
    for team in sorted(teams):
        rows.append({
            "team": team,
            "gold_%": round(100 * title_counts_tree[team] / N_TREE_SIMS, 3),
            "silver_%": round(100 * silver_counts_tree[team] / N_TREE_SIMS, 3),
            "bronze_%": round(100 * bronze_counts_tree[team] / N_TREE_SIMS, 3),
            "fourth_%": round(100 * fourth_counts_tree[team] / N_TREE_SIMS, 3),
        })

    if not rows:
        return None

    return (
        pl.DataFrame(rows)
        .sort(["gold_%", "silver_%", "bronze_%"], descending=True)
    )

def make_single_run_tables():
    if "simulate_tournament_once_with_records" not in globals():
        return None, None, None

    old_rng = globals().get("rng", None)

    try:
        if "np" in globals():
            globals()["rng"] = np.random.default_rng(20260621)

        sim = simulate_tournament_once_with_records()

        group_df = pl.DataFrame(sim.get("group_records", []))
        ko_df = pl.DataFrame(sim.get("ko_records", []))
        medals_df = pl.DataFrame([{
            "gold": sim.get("gold"),
            "silver": sim.get("silver"),
            "bronze": sim.get("bronze"),
            "fourth": sim.get("fourth"),
        }])

        return group_df, ko_df, medals_df

    finally:
        if old_rng is not None:
            globals()["rng"] = old_rng

single_group_df, single_ko_df, single_medals_df = make_single_run_tables()

doc = Document()

section = doc.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width = Inches(11.69)
section.page_height = Inches(8.27)
section.left_margin = Inches(0.45)
section.right_margin = Inches(0.45)
section.top_margin = Inches(0.45)
section.bottom_margin = Inches(0.45)

styles = doc.styles
styles["Normal"].font.name = "Aptos"
styles["Normal"].font.size = Pt(8)

title = doc.add_heading("WeltmeisterKI4 WM 2026 Report", level=0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitle = doc.add_paragraph()
subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
subtitle.add_run(f"Export erstellt am {datetime.now().strftime('%d.%m.%Y %H:%M')}").bold = True

meta_rows = [{
    "active_model": globals().get("ACTIVE_MODEL_NAME", "unbekannt"),
    "live_factor": globals().get("REAL_MATCH_STATE_REPEATS", "unbekannt"),
    "tree_sims": globals().get("N_TREE_SIMS", globals().get("N_SIMS", "unbekannt")),
    "played_matches": len(globals().get("PLAYED_MATCHES", [])),
}]
add_table_from_df(doc, "Modell-Setup", pl.DataFrame(meta_rows))

doc.add_heading("Einzel-Durchlauf Turnierbaum", level=1)

add_table_from_df(
    doc,
    "Einzel-Durchlauf: Medaillen",
    single_medals_df,
    preferred_cols=["gold", "silver", "bronze", "fourth"],
)

add_table_from_df(
    doc,
    "Einzel-Durchlauf: Gruppenspiele",
    single_group_df,
    preferred_cols=[
        "match_number", "group", "home", "away", "score", "winner",
        "home_score", "away_score", "p_home_win", "p_draw", "p_away_win",
        "most_likely_score", "score_prob_%", "status",
    ],
    note="Ein einzelner simulierter Turnierdurchlauf. Gespielte Matches bleiben fix, offene Matches werden aus dem aktiven Modell gezogen.",
)

add_table_from_df(
    doc,
    "Einzel-Durchlauf: KO-Spiele",
    single_ko_df,
    preferred_cols=[
        "match_number", "stage", "home", "away", "score", "winner", "loser",
        "p_home_advance", "p_away_advance", "p_home_win", "p_draw", "p_away_win",
        "most_likely_score", "score_prob_%",
    ],
)

doc.add_page_break()
doc.add_heading("Aggregierter Monte-Carlo-Turnierbaum", level=1)

agg_df = globals().get("full_tree_agg")
add_table_from_df(
    doc,
    "Aggregierter Baum: Alle Spiele",
    agg_df,
    preferred_cols=[
        "match", "stage", "häufigstes_matchup", "haeufigstes_matchup",
        "top_matchup", "matchup", "matchup_%",
        "häufigstes_resultat", "haeufigstes_resultat", "top_result", "score",
        "result_%",
        "häufigster_gewinner", "haeufigster_gewinner", "top_winner", "winner",
        "winner_%",
    ],
    note="Aggregiert aus Monte-Carlo-Simulationen. Die Prozentwerte zeigen, wie oft Matchup/Resultat/Gewinner in den Simulationen vorkamen.",
)

doc.add_heading("Monte-Carlo-Zusatzboards", level=1)

if "likely_16telfinalisten_title" in globals():
    add_table_from_df(
        doc,
        "Wahrscheinlichste 16telfinalisten und Titelchance",
        likely_16telfinalisten_title,
        preferred_cols=["team", "16telfinale_%", "title_%", "weltmeister_%", "finale_%", "halbfinale_medal_zone_%"],
        max_rows=40,
    )
elif "round32_title_board" in globals():
    add_table_from_df(
        doc,
        "Wahrscheinlichste 16telfinalisten und Titelchance",
        round32_title_board,
        preferred_cols=["team", "16telfinale_%", "title_%", "weltmeister_%", "finale_%", "halbfinale_medal_zone_%"],
        max_rows=40,
    )

if "title_board" in globals():
    add_table_from_df(
        doc,
        "Titelwahrscheinlichkeiten Top 25",
        title_board,
        preferred_cols=["team", "title_%", "weltmeister_%", "count"],
        max_rows=25,
    )
elif "title_counts_tree" in globals():
    title_df = counter_to_prob_df(title_counts_tree, N_TREE_SIMS, "team", "weltmeister_%", top_n=25)
    add_table_from_df(doc, "Titelwahrscheinlichkeiten Top 25", title_df)

medal_df = globals().get("medal_board")
if medal_df is None:
    medal_df = build_medal_prob_df()

add_table_from_df(
    doc,
    "Medaillenwahrscheinlichkeiten",
    medal_df,
    preferred_cols=["team", "gold_%", "silber_%", "silver_%", "bronze_%", "vierter_%", "fourth_%"],
    max_rows=30,
)

if "final_pair_board" in globals():
    add_table_from_df(
        doc,
        "Häufigste Finals",
        final_pair_board,
        preferred_cols=["finale", "final_pair", "teams", "count", "wahrscheinlichkeit_%", "prob_%"],
        max_rows=25,
    )

for team in ["Germany", "South Korea", "USA"]:
    if "team_finish_board" in globals():
        try:
            add_table_from_df(
                doc,
                f"{team}: Rundenverteilung",
                team_finish_board(team),
                preferred_cols=["finish", "n", "wahrscheinlichkeit_%", "prob_%"],
                max_rows=20,
            )
        except Exception as e:
            doc.add_paragraph(f"{team}: Rundenverteilung konnte nicht exportiert werden: {e}")

    if "team_exit_vs_board" in globals():
        try:
            add_table_from_df(
                doc,
                f"{team}: Häufigste Exit-Gegner",
                team_exit_vs_board(team),
                preferred_cols=["gegner", "opponent", "n", "wahrscheinlichkeit_%", "prob_%"],
                max_rows=20,
            )
        except Exception as e:
            doc.add_paragraph(f"{team}: Exit-Gegner konnten nicht exportiert werden: {e}")

if "exit_overview" in globals():
    doc.add_heading("Exakte KO-Ausscheidungen", level=1)

    add_table_from_df(
        doc,
        "Exit-Übersicht",
        exit_overview,
        preferred_cols=["team", "gruppenphase_raus_%", "ko_ausscheiden_mit_score_%", "weltmeister_%", "ko_exit_count"],
    )

    if "exit_exact_top" in globals():
        add_table_from_df(
            doc,
            "Häufigste konkrete KO-Ausscheidungen",
            exit_exact_top,
            preferred_cols=["team", "stage", "gegner", "team_score", "lesart", "gesamt_%", "wenn_ko_exit_%"],
            max_rows=30,
        )

    if "exit_score_top" in globals():
        add_table_from_df(
            doc,
            "Häufigste Ausscheidungs-Endstände",
            exit_score_top,
            preferred_cols=["team", "team_score", "gesamt_%", "wenn_ko_exit_%"],
            max_rows=25,
        )

if "summary" in globals() or "score_summary" in globals():
    doc.add_heading("Deutschland Einzelprognosen", level=1)

    if "summary" in globals():
        add_table_from_df(
            doc,
            "Deutschland Gruppenspiele",
            summary,
            preferred_cols=[
                "match", "status", "score", "deutschland_xg", "gegner_xg",
                "deutschland_sieg_%", "remis_%", "deutschland_niederlage_%",
                "modell_pick", "wahrscheinlichster_modell_score", "score_wahrscheinlichkeit_%",
            ],
        )

    if "score_summary" in globals():
        add_table_from_df(
            doc,
            "Deutschland Score-Zusammenfassung",
            score_summary,
            preferred_cols=[
                "match", "win_pick", "deutschland_sieg_%", "remis_%", "deutschland_niederlage_%",
                "expected_score", "expected_rounded", "top8_weighted", "top8_rounded",
                "practical_score_pick", "top1_top2_diff_pp",
            ],
        )

if "summary_korea" in globals() or "score_summary_korea" in globals():
    doc.add_heading("Südkorea Einzelprognosen", level=1)

    if "summary_korea" in globals():
        add_table_from_df(
            doc,
            "Südkorea Gruppenspiele",
            summary_korea,
            preferred_cols=[
                "match", "status", "score", "suedkorea_xg", "gegner_xg",
                "suedkorea_sieg_%", "remis_%", "suedkorea_niederlage_%",
                "modell_pick", "wahrscheinlichster_modell_score", "score_wahrscheinlichkeit_%",
            ],
        )

    if "score_summary_korea" in globals():
        add_table_from_df(
            doc,
            "Südkorea Score-Zusammenfassung",
            score_summary_korea,
            max_rows=20,
        )

doc.add_paragraph("")
footer = doc.add_paragraph()
footer.alignment = WD_ALIGN_PARAGRAPH.CENTER
footer.add_run("WeltmeisterKI4 - probabilistische Prognosen, keine Garantien.").italic = True

doc.save(DOCX_PATH)

print("Word-Report gespeichert:")
print(DOCX_PATH)

Word-Report gespeichert:
C:\ml\projects\WeltmeisterKI\WeltmeisterKI4_Report_20260621_1434.docx
